# DeFi Liquidation Cascade Predictor
## Temporal Graph Networks on Cross-Protocol Composability Graphs

**Single-cell execution** — installs dependencies, writes source files, embeds real TVL data,
runs the full 9-phase pipeline (TGN + 5 baselines + ablation + statistical tests + visualization),
and exports results.

**Instructions:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run the single code cell below
3. Results zip will auto-download when complete

In [ ]:
# @title Run Full DeFi Cascade Predictor Pipeline {display-mode: "form"}
# ============================================================================
# DeFi Liquidation Cascade Predictor - Single Cell Execution
# ============================================================================

import subprocess, sys, os, time as _time

# ---- Install Dependencies ----
print("=" * 70)
print("INSTALLING DEPENDENCIES...")
print("=" * 70)

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkg.split(),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pip("torch-geometric")

try:
    import torch
    tv = torch.__version__.split("+")[0]
    cuda_tag = torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
    wheel_url = f"https://data.pyg.org/whl/torch-{tv}+cu{cuda_tag}.html"
    subprocess.check_call(
        f"{sys.executable} -m pip install -q torch-scatter torch-sparse -f {wheel_url}",
        shell=True, timeout=120, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except Exception:
    pass

pip("xgboost loguru pyyaml")

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Training will be slower.")

# ---- Write Source Files ----
print("\n" + "=" * 70)
print("WRITING SOURCE FILES...")
print("=" * 70)

import json, base64, gzip
from pathlib import Path

PROJECT_ROOT = Path("/content/defi_cascade_predictor")

for d in ["data/collectors", "data/processing", "data/real",
          "models/layers", "models/baselines",
          "training", "evaluation", "experiments",
          "config", "outputs/figures", "outputs/results", "outputs/checkpoints"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

SOURCE_FILES = json.loads('{"data/__init__.py": "IiIiRGF0YSBtb2R1bGVzLiIiIgo=", "data/collectors/__init__.py": "IiIiRGF0YSBjb2xsZWN0b3IgbW9kdWxlcy4iIiIKZnJvbSAuY29pbmdlY2tvX2NvbGxlY3RvciBpbXBvcnQgQ29pbkdlY2tvQ29sbGVjdG9yCmZyb20gLmNhc2NhZGVfbGFiZWxlciBpbXBvcnQgQ2FzY2FkZUxhYmVsZXIKX19hbGxfXyA9IFsiQ29pbkdlY2tvQ29sbGVjdG9yIiwgIkNhc2NhZGVMYWJlbGVyIl0K", "data/processing/__init__.py": "IiIiRGF0YSBwcm9jZXNzaW5nIG1vZHVsZXMuIiIiCmZyb20gLmdyYXBoX2NvbnN0cnVjdG9yIGltcG9ydCBDb21wb3NhYmlsaXR5R3JhcGhDb25zdHJ1Y3Rvcgpmcm9tIC5mZWF0dXJlX2VuZ2luZWVyIGltcG9ydCBGZWF0dXJlRW5naW5lZXIKX19hbGxfXyA9IFsiQ29tcG9zYWJpbGl0eUdyYXBoQ29uc3RydWN0b3IiLCAiRmVhdHVyZUVuZ2luZWVyIl0K", "models/__init__.py": "IiIiTW9kZWwgaW1wbGVtZW50YXRpb25zLiIiIgpmcm9tIC50Z24gaW1wb3J0IFRlbXBvcmFsR3JhcGhOZXR3b3JrCl9fYWxsX18gPSBbIlRlbXBvcmFsR3JhcGhOZXR3b3JrIl0K", "models/layers/__init__.py": "IiIiTmV1cmFsIG5ldHdvcmsgbGF5ZXJzLiIiIgo=", "models/baselines/__init__.py": "IiIiQmFzZWxpbmUgbW9kZWxzLiIiIgo=", "training/__init__.py": "IiIiVHJhaW5pbmcgcGlwZWxpbmUuIiIiCmZyb20gLmxvc3NlcyBpbXBvcnQgRm9jYWxMb3NzLCBDYXNjYWRlTG9zcwpfX2FsbF9fID0gWyJGb2NhbExvc3MiLCAiQ2FzY2FkZUxvc3MiXQo=", "evaluation/__init__.py": "IiIiRXZhbHVhdGlvbiBtb2R1bGVzLiIiIgo=", "experiments/__init__.py": "IiIiRXhwZXJpbWVudCBvcmNoZXN0cmF0aW9uLiIiIgo=", "models/layers/temporal_attention.py": "IiIiClRlbXBvcmFsIGF0dGVudGlvbiBsYXllciBmb3IgdGhlIFRlbXBvcmFsIEdyYXBoIE5ldHdvcmsuCkltcGxlbWVudHMgbXVsdGktaGVhZCBhdHRlbnRpb24gb3ZlciB0ZW1wb3JhbCBub2RlIG5laWdoYm9yaG9vZHMKd2l0aCB0aW1lLWF3YXJlIHBvc2l0aW9uYWwgZW5jb2RpbmcuCiIiIgoKaW1wb3J0IG1hdGgKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKY2xhc3MgVGltZUVuY29kaW5nKG5uLk1vZHVsZSk6CiAgICAiIiJMZWFybmFibGUgdGltZSBlbmNvZGluZyB1c2luZyBCb2NobmVyJ3MgdGhlb3JlbS1pbnNwaXJlZCBhcHByb2FjaC4KCiAgICBNYXBzIHRpbWUgZGVsdGFzIHRvIGEgZml4ZWQtZGltZW5zaW9uYWwgcmVwcmVzZW50YXRpb24gdXNpbmcKICAgIGxlYXJuYWJsZSBmcmVxdWVuY3kgcGFyYW1ldGVycy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW06IGludCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5kaW0gPSBkaW0KICAgICAgICBzZWxmLncgPSBubi5MaW5lYXIoMSwgZGltKQogICAgICAgIG5uLmluaXQueGF2aWVyX3VuaWZvcm1fKHNlbGYudy53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi53LmJpYXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgdDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgdDogVGltZSBkZWx0YSB0ZW5zb3Igb2Ygc2hhcGUgW2JhdGNoX3NpemVdIG9yIFtiYXRjaF9zaXplLCAxXS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVGltZSBlbmNvZGluZyBvZiBzaGFwZSBbYmF0Y2hfc2l6ZSwgZGltXS4KICAgICAgICAiIiIKICAgICAgICBpZiB0LmRpbSgpID09IDE6CiAgICAgICAgICAgIHQgPSB0LnVuc3F1ZWV6ZSgxKQogICAgICAgIHQgPSB0LmZsb2F0KCkKICAgICAgICBvdXRwdXQgPSB0b3JjaC5jb3Moc2VsZi53KHQpKQogICAgICAgIHJldHVybiBvdXRwdXQKCgpjbGFzcyBUZW1wb3JhbEF0dGVudGlvbkxheWVyKG5uLk1vZHVsZSk6CiAgICAiIiJNdWx0aS1oZWFkIHRlbXBvcmFsIGF0dGVudGlvbiBsYXllci4KCiAgICBDb21wdXRlcyBhdHRlbnRpb24gb3ZlciB0ZW1wb3JhbCBub2RlIG5laWdoYm9yaG9vZHMsIGluY29ycG9yYXRpbmcKICAgIHRpbWUgZW5jb2RpbmdzLCBzb3VyY2Ugbm9kZSBmZWF0dXJlcywgZWRnZSBmZWF0dXJlcywgYW5kIG5laWdoYm9yCiAgICBmZWF0dXJlcyB0byBwcm9kdWNlIHVwZGF0ZWQgbm9kZSBlbWJlZGRpbmdzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbm9kZV9kaW06IGludCwKICAgICAgICBlZGdlX2RpbTogaW50LAogICAgICAgIHRpbWVfZGltOiBpbnQsCiAgICAgICAgb3V0cHV0X2RpbTogaW50LAogICAgICAgIG51bV9oZWFkczogaW50ID0gNCwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMSwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ub2RlX2RpbSA9IG5vZGVfZGltCiAgICAgICAgc2VsZi5lZGdlX2RpbSA9IGVkZ2VfZGltCiAgICAgICAgc2VsZi50aW1lX2RpbSA9IHRpbWVfZGltCiAgICAgICAgc2VsZi5vdXRwdXRfZGltID0gb3V0cHV0X2RpbQogICAgICAgIHNlbGYubnVtX2hlYWRzID0gbnVtX2hlYWRzCiAgICAgICAgc2VsZi5oZWFkX2RpbSA9IG91dHB1dF9kaW0gLy8gbnVtX2hlYWRzCgogICAgICAgIGFzc2VydCBvdXRwdXRfZGltICUgbnVtX2hlYWRzID09IDAsICJvdXRwdXRfZGltIG11c3QgYmUgZGl2aXNpYmxlIGJ5IG51bV9oZWFkcyIKCiAgICAgICAgIyBJbnB1dCBwcm9qZWN0aW9uIGRpbWVuc2lvbgogICAgICAgIHRvdGFsX2lucHV0X2RpbSA9IG5vZGVfZGltICsgZWRnZV9kaW0gKyB0aW1lX2RpbQoKICAgICAgICAjIE11bHRpLWhlYWQgYXR0ZW50aW9uIHByb2plY3Rpb25zCiAgICAgICAgc2VsZi5xdWVyeV9wcm9qID0gbm4uTGluZWFyKG5vZGVfZGltICsgdGltZV9kaW0sIG91dHB1dF9kaW0pCiAgICAgICAgc2VsZi5rZXlfcHJvaiA9IG5uLkxpbmVhcih0b3RhbF9pbnB1dF9kaW0sIG91dHB1dF9kaW0pCiAgICAgICAgc2VsZi52YWx1ZV9wcm9qID0gbm4uTGluZWFyKHRvdGFsX2lucHV0X2RpbSwgb3V0cHV0X2RpbSkKICAgICAgICBzZWxmLm91dHB1dF9wcm9qID0gbm4uTGluZWFyKG91dHB1dF9kaW0sIG91dHB1dF9kaW0pCgogICAgICAgICMgTGF5ZXIgbm9ybWFsaXphdGlvbgogICAgICAgIHNlbGYubGF5ZXJfbm9ybSA9IG5uLkxheWVyTm9ybShvdXRwdXRfZGltKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICAgICAgIyBGZWVkLWZvcndhcmQgbmV0d29yawogICAgICAgIHNlbGYuZmZuID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKG91dHB1dF9kaW0sIG91dHB1dF9kaW0gKiAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICBubi5MaW5lYXIob3V0cHV0X2RpbSAqIDIsIG91dHB1dF9kaW0pLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICkKICAgICAgICBzZWxmLmZmbl9ub3JtID0gbm4uTGF5ZXJOb3JtKG91dHB1dF9kaW0pCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBxdWVyeV9ub2RlX2ZlYXR1cmVzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgbmVpZ2hib3Jfbm9kZV9mZWF0dXJlczogdG9yY2guVGVuc29yLAogICAgICAgIGVkZ2VfZmVhdHVyZXM6IHRvcmNoLlRlbnNvciwKICAgICAgICB0aW1lX2VuY29kaW5nczogdG9yY2guVGVuc29yLAogICAgICAgIG5laWdoYm9yX3RpbWVfZW5jb2RpbmdzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgbWFzazogdG9yY2guVGVuc29yID0gTm9uZSwKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHF1ZXJ5X25vZGVfZmVhdHVyZXM6IFtiYXRjaCwgbm9kZV9kaW1dIC0gdGFyZ2V0IG5vZGUgZmVhdHVyZXMuCiAgICAgICAgICAgIG5laWdoYm9yX25vZGVfZmVhdHVyZXM6IFtiYXRjaCwgbl9uZWlnaGJvcnMsIG5vZGVfZGltXS4KICAgICAgICAgICAgZWRnZV9mZWF0dXJlczogW2JhdGNoLCBuX25laWdoYm9ycywgZWRnZV9kaW1dLgogICAgICAgICAgICB0aW1lX2VuY29kaW5nczogW2JhdGNoLCB0aW1lX2RpbV0gLSBxdWVyeSBub2RlIHRpbWUgZW5jb2RpbmcuCiAgICAgICAgICAgIG5laWdoYm9yX3RpbWVfZW5jb2RpbmdzOiBbYmF0Y2gsIG5fbmVpZ2hib3JzLCB0aW1lX2RpbV0uCiAgICAgICAgICAgIG1hc2s6IFtiYXRjaCwgbl9uZWlnaGJvcnNdIC0gYm9vbGVhbiBtYXNrIChUcnVlID0gaWdub3JlKS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVXBkYXRlZCBub2RlIGVtYmVkZGluZ3MgW2JhdGNoLCBvdXRwdXRfZGltXS4KICAgICAgICAiIiIKICAgICAgICBiYXRjaF9zaXplID0gcXVlcnlfbm9kZV9mZWF0dXJlcy5zaXplKDApCiAgICAgICAgbl9uZWlnaGJvcnMgPSBuZWlnaGJvcl9ub2RlX2ZlYXR1cmVzLnNpemUoMSkKCiAgICAgICAgIyBDb25zdHJ1Y3QgcXVlcnkgZnJvbSB0YXJnZXQgbm9kZSArIGl0cyB0aW1lIGVuY29kaW5nCiAgICAgICAgcXVlcnlfaW5wdXQgPSB0b3JjaC5jYXQoW3F1ZXJ5X25vZGVfZmVhdHVyZXMsIHRpbWVfZW5jb2RpbmdzXSwgZGltPS0xKQogICAgICAgIFEgPSBzZWxmLnF1ZXJ5X3Byb2oocXVlcnlfaW5wdXQpICAjIFtiYXRjaCwgb3V0cHV0X2RpbV0KCiAgICAgICAgIyBDb25zdHJ1Y3Qga2V5L3ZhbHVlIGZyb20gbmVpZ2hib3IgZmVhdHVyZXMgKyBlZGdlIGZlYXR1cmVzICsgdGltZQogICAgICAgIG5laWdoYm9yX2lucHV0ID0gdG9yY2guY2F0KAogICAgICAgICAgICBbbmVpZ2hib3Jfbm9kZV9mZWF0dXJlcywgZWRnZV9mZWF0dXJlcywgbmVpZ2hib3JfdGltZV9lbmNvZGluZ3NdLAogICAgICAgICAgICBkaW09LTEsCiAgICAgICAgKSAgIyBbYmF0Y2gsIG5fbmVpZ2hib3JzLCB0b3RhbF9pbnB1dF9kaW1dCiAgICAgICAgSyA9IHNlbGYua2V5X3Byb2oobmVpZ2hib3JfaW5wdXQpICAjIFtiYXRjaCwgbl9uZWlnaGJvcnMsIG91dHB1dF9kaW1dCiAgICAgICAgViA9IHNlbGYudmFsdWVfcHJvaihuZWlnaGJvcl9pbnB1dCkgICMgW2JhdGNoLCBuX25laWdoYm9ycywgb3V0cHV0X2RpbV0KCiAgICAgICAgIyBSZXNoYXBlIGZvciBtdWx0aS1oZWFkIGF0dGVudGlvbgogICAgICAgIFEgPSBRLnZpZXcoYmF0Y2hfc2l6ZSwgMSwgc2VsZi5udW1faGVhZHMsIHNlbGYuaGVhZF9kaW0pLnRyYW5zcG9zZSgxLCAyKQogICAgICAgIEsgPSBLLnZpZXcoYmF0Y2hfc2l6ZSwgbl9uZWlnaGJvcnMsIHNlbGYubnVtX2hlYWRzLCBzZWxmLmhlYWRfZGltKS50cmFuc3Bvc2UoCiAgICAgICAgICAgIDEsIDIKICAgICAgICApCiAgICAgICAgViA9IFYudmlldyhiYXRjaF9zaXplLCBuX25laWdoYm9ycywgc2VsZi5udW1faGVhZHMsIHNlbGYuaGVhZF9kaW0pLnRyYW5zcG9zZSgKICAgICAgICAgICAgMSwgMgogICAgICAgICkKICAgICAgICAjIFE6IFtiYXRjaCwgaGVhZHMsIDEsIGhlYWRfZGltXQogICAgICAgICMgSywgVjogW2JhdGNoLCBoZWFkcywgbl9uZWlnaGJvcnMsIGhlYWRfZGltXQoKICAgICAgICAjIFNjYWxlZCBkb3QtcHJvZHVjdCBhdHRlbnRpb24KICAgICAgICBzY29yZXMgPSB0b3JjaC5tYXRtdWwoUSwgSy50cmFuc3Bvc2UoLTIsIC0xKSkgLyBtYXRoLnNxcnQoc2VsZi5oZWFkX2RpbSkKICAgICAgICAjIHNjb3JlczogW2JhdGNoLCBoZWFkcywgMSwgbl9uZWlnaGJvcnNdCgogICAgICAgIGlmIG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG1hc2sgPSBtYXNrLnVuc3F1ZWV6ZSgxKS51bnNxdWVlemUoMikgICMgW2JhdGNoLCAxLCAxLCBuX25laWdoYm9yc10KICAgICAgICAgICAgc2NvcmVzID0gc2NvcmVzLm1hc2tlZF9maWxsKG1hc2ssIGZsb2F0KCItaW5mIikpCgogICAgICAgIGF0dG5fd2VpZ2h0cyA9IEYuc29mdG1heChzY29yZXMsIGRpbT0tMSkKICAgICAgICBhdHRuX3dlaWdodHMgPSBzZWxmLmRyb3BvdXQoYXR0bl93ZWlnaHRzKQoKICAgICAgICAjIEFwcGx5IGF0dGVudGlvbiB0byB2YWx1ZXMKICAgICAgICBjb250ZXh0ID0gdG9yY2gubWF0bXVsKGF0dG5fd2VpZ2h0cywgVikgICMgW2JhdGNoLCBoZWFkcywgMSwgaGVhZF9kaW1dCiAgICAgICAgY29udGV4dCA9IGNvbnRleHQudHJhbnNwb3NlKDEsIDIpLmNvbnRpZ3VvdXMoKS52aWV3KGJhdGNoX3NpemUsIHNlbGYub3V0cHV0X2RpbSkKCiAgICAgICAgIyBPdXRwdXQgcHJvamVjdGlvbiArIHJlc2lkdWFsICsgbm9ybQogICAgICAgIG91dHB1dCA9IHNlbGYub3V0cHV0X3Byb2ooY29udGV4dCkKICAgICAgICBvdXRwdXQgPSBzZWxmLmRyb3BvdXQob3V0cHV0KQoKICAgICAgICAjIFJlc2lkdWFsIGNvbm5lY3Rpb24gKHByb2plY3QgcXVlcnkgdG8gbWF0Y2ggb3V0cHV0IGRpbSBpZiBuZWVkZWQpCiAgICAgICAgaWYgcXVlcnlfbm9kZV9mZWF0dXJlcy5zaXplKC0xKSAhPSBzZWxmLm91dHB1dF9kaW06CiAgICAgICAgICAgIHJlc2lkdWFsID0gbm4uZnVuY3Rpb25hbC5saW5lYXIoCiAgICAgICAgICAgICAgICBxdWVyeV9ub2RlX2ZlYXR1cmVzLAogICAgICAgICAgICAgICAgdG9yY2guZXllKHNlbGYub3V0cHV0X2RpbSwgc2VsZi5ub2RlX2RpbSwgZGV2aWNlPW91dHB1dC5kZXZpY2UpLAogICAgICAgICAgICApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVzaWR1YWwgPSBxdWVyeV9ub2RlX2ZlYXR1cmVzCiAgICAgICAgb3V0cHV0ID0gc2VsZi5sYXllcl9ub3JtKG91dHB1dCArIHJlc2lkdWFsKQoKICAgICAgICAjIEZlZWQtZm9yd2FyZCArIHJlc2lkdWFsCiAgICAgICAgZmZuX291dHB1dCA9IHNlbGYuZmZuKG91dHB1dCkKICAgICAgICBvdXRwdXQgPSBzZWxmLmZmbl9ub3JtKG91dHB1dCArIGZmbl9vdXRwdXQpCgogICAgICAgIHJldHVybiBvdXRwdXQK", "models/layers/message_passing.py": "IiIiCk1lc3NhZ2UgcGFzc2luZyBsYXllciBmb3IgdGhlIFRlbXBvcmFsIEdyYXBoIE5ldHdvcmsuCkltcGxlbWVudHMgZ3JhcGggYXR0ZW50aW9uLWJhc2VkIG1lc3NhZ2UgcGFzc2luZyBvdmVyIHRoZQpEZUZpIGNvbXBvc2FiaWxpdHkgZ3JhcGguCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gdG9yY2hfZ2VvbWV0cmljLm5uIGltcG9ydCBNZXNzYWdlUGFzc2luZyBhcyBQeUdNZXNzYWdlUGFzc2luZwpmcm9tIHRvcmNoX2dlb21ldHJpYy51dGlscyBpbXBvcnQgc29mdG1heAoKCmNsYXNzIE1lc3NhZ2VQYXNzaW5nTGF5ZXIoUHlHTWVzc2FnZVBhc3NpbmcpOgogICAgIiIiR3JhcGggYXR0ZW50aW9uLWJhc2VkIG1lc3NhZ2UgcGFzc2luZyBmb3IgdGhlIGNvbXBvc2FiaWxpdHkgZ3JhcGguCgogICAgRXh0ZW5kcyBQeUcncyBNZXNzYWdlUGFzc2luZyB3aXRoIGVkZ2UtdHlwZS1hd2FyZSBhdHRlbnRpb24KICAgIGFuZCB0ZW1wb3JhbCBub2RlIG1lbW9yeSBpbnRlZ3JhdGlvbi4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGluX2RpbTogaW50LAogICAgICAgIG91dF9kaW06IGludCwKICAgICAgICBlZGdlX2RpbTogaW50ID0gMTYsCiAgICAgICAgaGVhZHM6IGludCA9IDQsCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSAwLjEsCiAgICAgICAgY29uY2F0OiBib29sID0gVHJ1ZSwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhhZ2dyPSJhZGQiLCBub2RlX2RpbT0wKQogICAgICAgIHNlbGYuaW5fZGltID0gaW5fZGltCiAgICAgICAgc2VsZi5vdXRfZGltID0gb3V0X2RpbQogICAgICAgIHNlbGYuaGVhZHMgPSBoZWFkcwogICAgICAgIHNlbGYuaGVhZF9kaW0gPSBvdXRfZGltIC8vIGhlYWRzCiAgICAgICAgc2VsZi5jb25jYXQgPSBjb25jYXQKICAgICAgICBzZWxmLmRyb3BvdXQgPSBkcm9wb3V0CgogICAgICAgICMgTGluZWFyIHRyYW5zZm9ybWF0aW9ucwogICAgICAgIHNlbGYubGluX3NyYyA9IG5uLkxpbmVhcihpbl9kaW0sIG91dF9kaW0sIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5saW5fZHN0ID0gbm4uTGluZWFyKGluX2RpbSwgb3V0X2RpbSwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLmxpbl9lZGdlID0gbm4uTGluZWFyKGVkZ2VfZGltLCBvdXRfZGltLCBiaWFzPUZhbHNlKQoKICAgICAgICAjIEF0dGVudGlvbiBwYXJhbWV0ZXJzCiAgICAgICAgc2VsZi5hdHRfc3JjID0gbm4uUGFyYW1ldGVyKHRvcmNoLlRlbnNvcigxLCBoZWFkcywgc2VsZi5oZWFkX2RpbSkpCiAgICAgICAgc2VsZi5hdHRfZHN0ID0gbm4uUGFyYW1ldGVyKHRvcmNoLlRlbnNvcigxLCBoZWFkcywgc2VsZi5oZWFkX2RpbSkpCiAgICAgICAgc2VsZi5hdHRfZWRnZSA9IG5uLlBhcmFtZXRlcih0b3JjaC5UZW5zb3IoMSwgaGVhZHMsIHNlbGYuaGVhZF9kaW0pKQoKICAgICAgICAjIE91dHB1dAogICAgICAgIGlmIGNvbmNhdDoKICAgICAgICAgICAgc2VsZi5saW5fb3V0ID0gbm4uTGluZWFyKG91dF9kaW0sIG91dF9kaW0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5saW5fb3V0ID0gbm4uTGluZWFyKHNlbGYuaGVhZF9kaW0sIG91dF9kaW0pCgogICAgICAgIHNlbGYubGF5ZXJfbm9ybSA9IG5uLkxheWVyTm9ybShvdXRfZGltKQogICAgICAgIHNlbGYuZHJvcG91dF9sYXllciA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICAgICAgc2VsZi5fcmVzZXRfcGFyYW1ldGVycygpCgogICAgZGVmIF9yZXNldF9wYXJhbWV0ZXJzKHNlbGYpOgogICAgICAgIG5uLmluaXQueGF2aWVyX3VuaWZvcm1fKHNlbGYubGluX3NyYy53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi5saW5fZHN0LndlaWdodCkKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLmxpbl9lZGdlLndlaWdodCkKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLmF0dF9zcmMpCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi5hdHRfZHN0KQogICAgICAgIG5uLmluaXQueGF2aWVyX3VuaWZvcm1fKHNlbGYuYXR0X2VkZ2UpCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICB4OiB0b3JjaC5UZW5zb3IsCiAgICAgICAgZWRnZV9pbmRleDogdG9yY2guVGVuc29yLAogICAgICAgIGVkZ2VfYXR0cjogdG9yY2guVGVuc29yID0gTm9uZSwKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHg6IE5vZGUgZmVhdHVyZXMgW251bV9ub2RlcywgaW5fZGltXS4KICAgICAgICAgICAgZWRnZV9pbmRleDogWzIsIG51bV9lZGdlc10uCiAgICAgICAgICAgIGVkZ2VfYXR0cjogT3B0aW9uYWwgZWRnZSBmZWF0dXJlcyBbbnVtX2VkZ2VzLCBlZGdlX2RpbV0uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIFVwZGF0ZWQgbm9kZSBmZWF0dXJlcyBbbnVtX25vZGVzLCBvdXRfZGltXS4KICAgICAgICAiIiIKICAgICAgICAjIExpbmVhciBwcm9qZWN0aW9ucwogICAgICAgIHhfc3JjID0gc2VsZi5saW5fc3JjKHgpCiAgICAgICAgeF9kc3QgPSBzZWxmLmxpbl9kc3QoeCkKCiAgICAgICAgIyBSZXNoYXBlIGZvciBtdWx0aS1oZWFkIGF0dGVudGlvbgogICAgICAgIHhfc3JjID0geF9zcmMudmlldygtMSwgc2VsZi5oZWFkcywgc2VsZi5oZWFkX2RpbSkKICAgICAgICB4X2RzdCA9IHhfZHN0LnZpZXcoLTEsIHNlbGYuaGVhZHMsIHNlbGYuaGVhZF9kaW0pCgogICAgICAgICMgUHJvY2VzcyBlZGdlIGZlYXR1cmVzCiAgICAgICAgaWYgZWRnZV9hdHRyIGlzIG5vdCBOb25lOgogICAgICAgICAgICBpZiBlZGdlX2F0dHIuc2l6ZSgtMSkgIT0gc2VsZi5saW5fZWRnZS5pbl9mZWF0dXJlczoKICAgICAgICAgICAgICAgICMgUGFkIG9yIHByb2plY3QgZWRnZSBmZWF0dXJlcwogICAgICAgICAgICAgICAgZWRnZV9hdHRyX3Byb2MgPSB0b3JjaC56ZXJvcygKICAgICAgICAgICAgICAgICAgICBlZGdlX2F0dHIuc2l6ZSgwKSwKICAgICAgICAgICAgICAgICAgICBzZWxmLmxpbl9lZGdlLmluX2ZlYXR1cmVzLAogICAgICAgICAgICAgICAgICAgIGRldmljZT1lZGdlX2F0dHIuZGV2aWNlLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgbWluX2RpbSA9IG1pbihlZGdlX2F0dHIuc2l6ZSgtMSksIHNlbGYubGluX2VkZ2UuaW5fZmVhdHVyZXMpCiAgICAgICAgICAgICAgICBlZGdlX2F0dHJfcHJvY1s6LCA6bWluX2RpbV0gPSBlZGdlX2F0dHJbOiwgOm1pbl9kaW1dCiAgICAgICAgICAgICAgICBlZGdlX2F0dHIgPSBlZGdlX2F0dHJfcHJvYwogICAgICAgICAgICBlZGdlX2F0dHIgPSBzZWxmLmxpbl9lZGdlKGVkZ2VfYXR0cikudmlldygtMSwgc2VsZi5oZWFkcywgc2VsZi5oZWFkX2RpbSkKCiAgICAgICAgIyBNZXNzYWdlIHBhc3NpbmcKICAgICAgICBvdXQgPSBzZWxmLnByb3BhZ2F0ZSgKICAgICAgICAgICAgZWRnZV9pbmRleCwKICAgICAgICAgICAgeD0oeF9zcmMsIHhfZHN0KSwKICAgICAgICAgICAgZWRnZV9hdHRyPWVkZ2VfYXR0ciwKICAgICAgICAgICAgc2l6ZT1Ob25lLAogICAgICAgICkKCiAgICAgICAgaWYgc2VsZi5jb25jYXQ6CiAgICAgICAgICAgIG91dCA9IG91dC52aWV3KC0xLCBzZWxmLm91dF9kaW0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0ID0gb3V0Lm1lYW4oZGltPTEpCgogICAgICAgIG91dCA9IHNlbGYubGluX291dChvdXQpCiAgICAgICAgb3V0ID0gc2VsZi5kcm9wb3V0X2xheWVyKG91dCkKCiAgICAgICAgIyBSZXNpZHVhbCBjb25uZWN0aW9uCiAgICAgICAgaWYgeC5zaXplKC0xKSA9PSBzZWxmLm91dF9kaW06CiAgICAgICAgICAgIG91dCA9IHNlbGYubGF5ZXJfbm9ybShvdXQgKyB4KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG91dCA9IHNlbGYubGF5ZXJfbm9ybShvdXQpCgogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgbWVzc2FnZSgKICAgICAgICBzZWxmLAogICAgICAgIHhfajogdG9yY2guVGVuc29yLAogICAgICAgIHhfaTogdG9yY2guVGVuc29yLAogICAgICAgIGVkZ2VfYXR0cjogdG9yY2guVGVuc29yLAogICAgICAgIGluZGV4OiB0b3JjaC5UZW5zb3IsCiAgICAgICAgcHRyPU5vbmUsCiAgICAgICAgc2l6ZV9pPU5vbmUsCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJDb21wdXRlIG1lc3NhZ2VzIHdpdGggYXR0ZW50aW9uIHdlaWdodHMuIiIiCiAgICAgICAgIyBBdHRlbnRpb24gc2NvcmVzCiAgICAgICAgYWxwaGFfc3JjID0gKHhfaiAqIHNlbGYuYXR0X3NyYykuc3VtKGRpbT0tMSkKICAgICAgICBhbHBoYV9kc3QgPSAoeF9pICogc2VsZi5hdHRfZHN0KS5zdW0oZGltPS0xKQoKICAgICAgICBhbHBoYSA9IGFscGhhX3NyYyArIGFscGhhX2RzdAoKICAgICAgICBpZiBlZGdlX2F0dHIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGFscGhhX2VkZ2UgPSAoZWRnZV9hdHRyICogc2VsZi5hdHRfZWRnZSkuc3VtKGRpbT0tMSkKICAgICAgICAgICAgYWxwaGEgPSBhbHBoYSArIGFscGhhX2VkZ2UKCiAgICAgICAgYWxwaGEgPSBGLmxlYWt5X3JlbHUoYWxwaGEsIDAuMikKICAgICAgICBhbHBoYSA9IHNvZnRtYXgoYWxwaGEsIGluZGV4LCBwdHIsIHNpemVfaSkKICAgICAgICBhbHBoYSA9IEYuZHJvcG91dChhbHBoYSwgcD1zZWxmLmRyb3BvdXQsIHRyYWluaW5nPXNlbGYudHJhaW5pbmcpCgogICAgICAgICMgV2VpZ2h0ZWQgbWVzc2FnZXMKICAgICAgICBtc2cgPSB4X2ogKiBhbHBoYS51bnNxdWVlemUoLTEpCiAgICAgICAgcmV0dXJuIG1zZwoKCmNsYXNzIEhldGVyb01lc3NhZ2VQYXNzaW5nTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIkhhbmRsZXMgbWVzc2FnZSBwYXNzaW5nIG92ZXIgaGV0ZXJvZ2VuZW91cyBlZGdlIHR5cGVzLgoKICAgIEFwcGxpZXMgc2VwYXJhdGUgTWVzc2FnZVBhc3NpbmdMYXllcnMgZm9yIGVhY2ggZWRnZSB0eXBlIGFuZAogICAgYWdncmVnYXRlcyB0aGUgcmVzdWx0cyB3aXRoIGEgcmVzaWR1YWwgY29ubmVjdGlvbiB0byB0aGUgaW5wdXQuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbl9kaW06IGludCwKICAgICAgICBvdXRfZGltOiBpbnQsCiAgICAgICAgZWRnZV90eXBlczogbGlzdFtzdHJdLAogICAgICAgIGVkZ2VfZGltOiBpbnQgPSAxNiwKICAgICAgICBoZWFkczogaW50ID0gNCwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMSwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5lZGdlX3R5cGVzID0gZWRnZV90eXBlcwogICAgICAgIHNlbGYub3V0X2RpbSA9IG91dF9kaW0KCiAgICAgICAgIyBPbmUgbWVzc2FnZSBwYXNzaW5nIGxheWVyIHBlciBlZGdlIHR5cGUKICAgICAgICBzZWxmLm1wX2xheWVycyA9IG5uLk1vZHVsZURpY3QoewogICAgICAgICAgICBldHlwZTogTWVzc2FnZVBhc3NpbmdMYXllcigKICAgICAgICAgICAgICAgIGluX2RpbSwgb3V0X2RpbSwgZWRnZV9kaW0sIGhlYWRzLCBkcm9wb3V0CiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIGV0eXBlIGluIGVkZ2VfdHlwZXMKICAgICAgICB9KQoKICAgICAgICAjIEdhdGVkIGFnZ3JlZ2F0aW9uOiBsZWFybiBwZXItZWRnZS10eXBlIGltcG9ydGFuY2UKICAgICAgICBzZWxmLmVkZ2VfdHlwZV9nYXRlcyA9IG5uLlBhcmFtZXRlckRpY3QoewogICAgICAgICAgICBldHlwZTogbm4uUGFyYW1ldGVyKHRvcmNoLm9uZXMoMSkpCiAgICAgICAgICAgIGZvciBldHlwZSBpbiBlZGdlX3R5cGVzCiAgICAgICAgfSkKCiAgICAgICAgIyBBZ2dyZWdhdGlvbjogbWVhbiBvZiBhY3RpdmUgZWRnZSB0eXBlcyArIHByb2plY3Rpb24KICAgICAgICBzZWxmLmFnZ3JlZ2F0ZSA9IG5uLkxpbmVhcihvdXRfZGltLCBvdXRfZGltKQogICAgICAgIHNlbGYubGF5ZXJfbm9ybSA9IG5uLkxheWVyTm9ybShvdXRfZGltKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICAgICAgIyBSZXNpZHVhbCBwcm9qZWN0aW9uIGlmIGRpbXMgbWlzbWF0Y2gKICAgICAgICBzZWxmLnJlc2lkdWFsX3Byb2ogPSAoCiAgICAgICAgICAgIG5uLkxpbmVhcihpbl9kaW0sIG91dF9kaW0pIGlmIGluX2RpbSAhPSBvdXRfZGltIGVsc2Ugbm4uSWRlbnRpdHkoKQogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIHg6IHRvcmNoLlRlbnNvciwKICAgICAgICBlZGdlX2luZGV4X2RpY3Q6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdLAogICAgICAgIGVkZ2VfYXR0cl9kaWN0OiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSA9IE5vbmUsCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiIKICAgICAgICBBcmdzOgogICAgICAgICAgICB4OiBOb2RlIGZlYXR1cmVzIFtudW1fbm9kZXMsIGluX2RpbV0uCiAgICAgICAgICAgIGVkZ2VfaW5kZXhfZGljdDogRGljdCBtYXBwaW5nIGVkZ2VfdHlwZSAtPiBbMiwgbnVtX2VkZ2VzXS4KICAgICAgICAgICAgZWRnZV9hdHRyX2RpY3Q6IE9wdGlvbmFsIGRpY3QgbWFwcGluZyBlZGdlX3R5cGUgLT4gZWRnZSBmZWF0dXJlcy4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVXBkYXRlZCBub2RlIGZlYXR1cmVzIFtudW1fbm9kZXMsIG91dF9kaW1dLgogICAgICAgICIiIgogICAgICAgIGlmIGVkZ2VfYXR0cl9kaWN0IGlzIE5vbmU6CiAgICAgICAgICAgIGVkZ2VfYXR0cl9kaWN0ID0ge30KCiAgICAgICAgIyBXZWlnaHRlZCBzdW0gb2YgYWN0aXZlIGVkZ2UtdHlwZSBvdXRwdXRzIChza2lwIGVtcHR5IHR5cGVzKQogICAgICAgIHdlaWdodGVkX3N1bSA9IHRvcmNoLnplcm9zKHguc2l6ZSgwKSwgc2VsZi5vdXRfZGltLCBkZXZpY2U9eC5kZXZpY2UpCiAgICAgICAgdG90YWxfd2VpZ2h0ID0gdG9yY2guemVyb3MoMSwgZGV2aWNlPXguZGV2aWNlKQoKICAgICAgICBmb3IgZXR5cGUgaW4gc2VsZi5lZGdlX3R5cGVzOgogICAgICAgICAgICBpZiBldHlwZSBpbiBlZGdlX2luZGV4X2RpY3QgYW5kIGVkZ2VfaW5kZXhfZGljdFtldHlwZV0uc2l6ZSgxKSA+IDA6CiAgICAgICAgICAgICAgICBlZGdlX2F0dHIgPSBlZGdlX2F0dHJfZGljdC5nZXQoZXR5cGUpCiAgICAgICAgICAgICAgICBvdXQgPSBzZWxmLm1wX2xheWVyc1tldHlwZV0oCiAgICAgICAgICAgICAgICAgICAgeCwgZWRnZV9pbmRleF9kaWN0W2V0eXBlXSwgZWRnZV9hdHRyCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBnYXRlID0gdG9yY2guc2lnbW9pZChzZWxmLmVkZ2VfdHlwZV9nYXRlc1tldHlwZV0pCiAgICAgICAgICAgICAgICB3ZWlnaHRlZF9zdW0gPSB3ZWlnaHRlZF9zdW0gKyBnYXRlICogb3V0CiAgICAgICAgICAgICAgICB0b3RhbF93ZWlnaHQgPSB0b3RhbF93ZWlnaHQgKyBnYXRlCgogICAgICAgICMgTm9ybWFsaXplIGJ5IG51bWJlciBvZiBhY3RpdmUgZWRnZSB0eXBlcwogICAgICAgIGlmIHRvdGFsX3dlaWdodC5pdGVtKCkgPiAwOgogICAgICAgICAgICB3ZWlnaHRlZF9zdW0gPSB3ZWlnaHRlZF9zdW0gLyB0b3RhbF93ZWlnaHQKCiAgICAgICAgb3V0cHV0ID0gc2VsZi5hZ2dyZWdhdGUod2VpZ2h0ZWRfc3VtKQogICAgICAgIG91dHB1dCA9IHNlbGYuZHJvcG91dChvdXRwdXQpCgogICAgICAgICMgUmVzaWR1YWwgY29ubmVjdGlvbgogICAgICAgIHJlc2lkdWFsID0gc2VsZi5yZXNpZHVhbF9wcm9qKHgpCiAgICAgICAgb3V0cHV0ID0gc2VsZi5sYXllcl9ub3JtKG91dHB1dCArIHJlc2lkdWFsKQoKICAgICAgICByZXR1cm4gb3V0cHV0Cg==", "models/layers/memory_module.py": "IiIiCk1lbW9yeSBtb2R1bGUgZm9yIHRoZSBUZW1wb3JhbCBHcmFwaCBOZXR3b3JrLgoKTWFpbnRhaW5zIHBlci1ub2RlIG1lbW9yeSB2ZWN0b3JzIHRoYXQgYXJlIHVwZGF0ZWQgYXQgZWFjaCBpbnRlcmFjdGlvbiwKY2FwdHVyaW5nIGxvbmctdGVybSB0ZW1wb3JhbCBwYXR0ZXJucyBpbiB0aGUgRGVGaSBjb21wb3NhYmlsaXR5IGdyYXBoLgoKS2V5IGRlc2lnbjogbWVtb3J5IHVwZGF0ZXMgYXJlIE5PVCBkZXRhY2hlZCwgc28gZ3JhZGllbnRzIGZsb3cgdGhyb3VnaAp0aGUgR1JVIGFjcm9zcyB0aW1lc3RlcHMgd2l0aGluIGEgVEJQVFQgd2luZG93LiBkZXRhY2hfbWVtb3J5KCkgaXMKY2FsbGVkIGF0IHdpbmRvdyBib3VuZGFyaWVzIHRvIHRydW5jYXRlIGJhY2twcm9wYWdhdGlvbi4KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKCmNsYXNzIE1lbW9yeU1vZHVsZShubi5Nb2R1bGUpOgogICAgIiIiR1JVL1JOTi1iYXNlZCBtZW1vcnkgbW9kdWxlIHRoYXQgbWFpbnRhaW5zIHBlci1ub2RlIHN0YXRlIHZlY3RvcnMuCgogICAgRWFjaCBwcm90b2NvbCBub2RlIGhhcyBhIHBlcnNpc3RlbnQgbWVtb3J5IHZlY3RvciB0aGF0IGdldHMgdXBkYXRlZAogICAgd2hlbiBuZXcgZXZlbnRzIChUVkwgY2hhbmdlcywgbGlxdWlkYXRpb25zLCBldGMuKSBvY2N1ciwgYWxsb3dpbmcKICAgIHRoZSBtb2RlbCB0byBjYXB0dXJlIHRlbXBvcmFsIGR5bmFtaWNzIGxpa2UgYnVpbGRpbmcgcmlzay4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG51bV9ub2RlczogaW50LAogICAgICAgIG1lbW9yeV9kaW06IGludCwKICAgICAgICBtZXNzYWdlX2RpbTogaW50LAogICAgICAgIHVwZGF0ZXJfdHlwZTogc3RyID0gImdydSIsCiAgICApOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIG51bV9ub2RlczogTnVtYmVyIG9mIG5vZGVzIGluIHRoZSBncmFwaCAocHJvdG9jb2xzKS4KICAgICAgICAgICAgbWVtb3J5X2RpbTogRGltZW5zaW9uIG9mIHBlci1ub2RlIG1lbW9yeSB2ZWN0b3JzLgogICAgICAgICAgICBtZXNzYWdlX2RpbTogRGltZW5zaW9uIG9mIGluY29taW5nIG1lc3NhZ2VzLgogICAgICAgICAgICB1cGRhdGVyX3R5cGU6ICJncnUiIG9yICJybm4iLgogICAgICAgICIiIgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubnVtX25vZGVzID0gbnVtX25vZGVzCiAgICAgICAgc2VsZi5tZW1vcnlfZGltID0gbWVtb3J5X2RpbQogICAgICAgIHNlbGYubWVzc2FnZV9kaW0gPSBtZXNzYWdlX2RpbQoKICAgICAgICAjIE1lbW9yeSBzdG9yYWdlIOKAlCBrZXB0IGFzIGEgcGxhaW4gdGVuc29yIChub3QgYnVmZmVyKSBzbyB0aGF0CiAgICAgICAgIyBncmFkaWVudC1jYXJyeWluZyB0ZW5zb3JzIGNhbiBiZSBhc3NpZ25lZCB3aXRob3V0IGF1dG9ncmFkIGlzc3Vlcy4KICAgICAgICAjIERldmljZSBtYW5hZ2VtZW50IGlzIGhhbmRsZWQgZXhwbGljaXRseSBpbiByZXNldF9tZW1vcnkgLyB0bygpLgogICAgICAgIHNlbGYubWVtb3J5OiB0b3JjaC5UZW5zb3IgPSB0b3JjaC56ZXJvcyhudW1fbm9kZXMsIG1lbW9yeV9kaW0pCiAgICAgICAgc2VsZi5sYXN0X3VwZGF0ZTogdG9yY2guVGVuc29yID0gdG9yY2guemVyb3MobnVtX25vZGVzKQoKICAgICAgICAjIE1lbW9yeSB1cGRhdGVyCiAgICAgICAgaWYgdXBkYXRlcl90eXBlID09ICJncnUiOgogICAgICAgICAgICBzZWxmLnVwZGF0ZXIgPSBubi5HUlVDZWxsKG1lc3NhZ2VfZGltLCBtZW1vcnlfZGltKQogICAgICAgIGVsaWYgdXBkYXRlcl90eXBlID09ICJybm4iOgogICAgICAgICAgICBzZWxmLnVwZGF0ZXIgPSBubi5STk5DZWxsKG1lc3NhZ2VfZGltLCBtZW1vcnlfZGltKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmtub3duIHVwZGF0ZXIgdHlwZToge3VwZGF0ZXJfdHlwZX0iKQoKICAgICAgICAjIE1lc3NhZ2UgZnVuY3Rpb246IHRyYW5zZm9ybXMgcmF3IGV2ZW50cyBpbnRvIG1lc3NhZ2VzCiAgICAgICAgc2VsZi5tZXNzYWdlX2ZuID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKG1lbW9yeV9kaW0gKiAyICsgbWVzc2FnZV9kaW0sIG1lc3NhZ2VfZGltKSwKICAgICAgICAgICAgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIobWVzc2FnZV9kaW0sIG1lc3NhZ2VfZGltKSwKICAgICAgICApCgogICAgZGVmIF90b19kZXZpY2Uoc2VsZiwgZGV2aWNlOiB0b3JjaC5kZXZpY2UpOgogICAgICAgICIiIk1vdmUgbWVtb3J5IHRlbnNvcnMgdG8gc3BlY2lmaWVkIGRldmljZS4iIiIKICAgICAgICBzZWxmLm1lbW9yeSA9IHNlbGYubWVtb3J5LnRvKGRldmljZSkKICAgICAgICBzZWxmLmxhc3RfdXBkYXRlID0gc2VsZi5sYXN0X3VwZGF0ZS50byhkZXZpY2UpCgogICAgZGVmIGdldF9tZW1vcnkoc2VsZiwgbm9kZV9pZHM6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiUmV0cmlldmUgY3VycmVudCBtZW1vcnkgdmVjdG9ycy4KCiAgICAgICAgUmV0dXJucyBtZW1vcnkgV0lUSCBncmFkaWVudCBoaXN0b3J5IHNvIHRoYXQgdXBzdHJlYW0gY29tcHV0YXRpb25zCiAgICAgICAgY2FuIGJhY2twcm9wYWdhdGUgdGhyb3VnaCBtZW1vcnkgcmVhZHMgdG8gcHJldmlvdXMgdXBkYXRlcy4KICAgICAgICAiIiIKICAgICAgICBpZiBub2RlX2lkcyBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gc2VsZi5tZW1vcnkKICAgICAgICByZXR1cm4gc2VsZi5tZW1vcnlbbm9kZV9pZHNdCgogICAgZGVmIGNvbXB1dGVfbWVzc2FnZXMoCiAgICAgICAgc2VsZiwKICAgICAgICBzb3VyY2VfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgdGFyZ2V0X2lkczogdG9yY2guVGVuc29yLAogICAgICAgIGVkZ2VfZmVhdHVyZXM6IHRvcmNoLlRlbnNvciwKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIkNvbXB1dGUgbWVzc2FnZXMgZnJvbSBzb3VyY2UgdG8gdGFyZ2V0IG5vZGVzLiIiIgogICAgICAgIHNvdXJjZV9tZW1vcnkgPSBzZWxmLm1lbW9yeVtzb3VyY2VfaWRzXQogICAgICAgIHRhcmdldF9tZW1vcnkgPSBzZWxmLm1lbW9yeVt0YXJnZXRfaWRzXQoKICAgICAgICBtc2dfaW5wdXQgPSB0b3JjaC5jYXQoCiAgICAgICAgICAgIFtzb3VyY2VfbWVtb3J5LCB0YXJnZXRfbWVtb3J5LCBlZGdlX2ZlYXR1cmVzXSwgZGltPS0xCiAgICAgICAgKQogICAgICAgIG1lc3NhZ2VzID0gc2VsZi5tZXNzYWdlX2ZuKG1zZ19pbnB1dCkKICAgICAgICByZXR1cm4gbWVzc2FnZXMKCiAgICBkZWYgYWdncmVnYXRlX21lc3NhZ2VzKAogICAgICAgIHNlbGYsCiAgICAgICAgbm9kZV9pZHM6IHRvcmNoLlRlbnNvciwKICAgICAgICBtZXNzYWdlczogdG9yY2guVGVuc29yLAogICAgICAgIGFnZ3JlZ2F0b3I6IHN0ciA9ICJsYXN0IiwKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIkFnZ3JlZ2F0ZSBtZXNzYWdlcyBmb3IgZWFjaCBub2RlLiIiIgogICAgICAgIHVuaXF1ZV9ub2RlcyA9IG5vZGVfaWRzLnVuaXF1ZSgpCiAgICAgICAgYWdncmVnYXRlZCA9IHRvcmNoLnplcm9zKAogICAgICAgICAgICBsZW4odW5pcXVlX25vZGVzKSwgc2VsZi5tZXNzYWdlX2RpbSwgZGV2aWNlPW1lc3NhZ2VzLmRldmljZQogICAgICAgICkKCiAgICAgICAgZm9yIGksIG5vZGVfaWQgaW4gZW51bWVyYXRlKHVuaXF1ZV9ub2Rlcyk6CiAgICAgICAgICAgIG1hc2sgPSBub2RlX2lkcyA9PSBub2RlX2lkCiAgICAgICAgICAgIG5vZGVfbWVzc2FnZXMgPSBtZXNzYWdlc1ttYXNrXQogICAgICAgICAgICBpZiBhZ2dyZWdhdG9yID09ICJtZWFuIjoKICAgICAgICAgICAgICAgIGFnZ3JlZ2F0ZWRbaV0gPSBub2RlX21lc3NhZ2VzLm1lYW4oZGltPTApCiAgICAgICAgICAgIGVsaWYgYWdncmVnYXRvciA9PSAibGFzdCI6CiAgICAgICAgICAgICAgICBhZ2dyZWdhdGVkW2ldID0gbm9kZV9tZXNzYWdlc1stMV0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGFnZ3JlZ2F0ZWRbaV0gPSBub2RlX21lc3NhZ2VzLm1lYW4oZGltPTApCgogICAgICAgIHJldHVybiB1bmlxdWVfbm9kZXMsIGFnZ3JlZ2F0ZWQKCiAgICBkZWYgdXBkYXRlX21lbW9yeSgKICAgICAgICBzZWxmLAogICAgICAgIG5vZGVfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgbWVzc2FnZXM6IHRvcmNoLlRlbnNvciwKICAgICAgICB0aW1lc3RhbXBzOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZSwKICAgICk6CiAgICAgICAgIiIiVXBkYXRlIG1lbW9yeSB2ZWN0b3JzIOKAlCBncmFkaWVudHMgZmxvdyB0aHJvdWdoIGZvciBUQlBUVC4KCiAgICAgICAgQ3JlYXRlcyBhIG5ldyB0ZW5zb3IgKG5vIGluLXBsYWNlIG1vZGlmaWNhdGlvbikgc28gdGhhdCBhdXRvZ3JhZAogICAgICAgIGNhbiB0cmFjayB0aGUgZGVwZW5kZW5jeSBjaGFpbiBhY3Jvc3MgdGltZXN0ZXBzLgogICAgICAgICIiIgogICAgICAgIGlmIGxlbihub2RlX2lkcykgPT0gMDoKICAgICAgICAgICAgcmV0dXJuCgogICAgICAgIGN1cnJlbnRfbWVtb3J5ID0gc2VsZi5tZW1vcnlbbm9kZV9pZHNdCiAgICAgICAgbmV3X21lbW9yeSA9IHNlbGYudXBkYXRlcihtZXNzYWdlcywgY3VycmVudF9tZW1vcnkpCgogICAgICAgICMgQnVpbGQgdXBkYXRlZCBtZW1vcnkgV0lUSE9VVCBpbi1wbGFjZSBtb2RpZmljYXRpb24uCiAgICAgICAgIyBzY2F0dGVyIG5ld19tZW1vcnkgaW50byBhIGZyZXNoIGNvcHkgc28gYXV0b2dyYWQgY2FuIHRyYWNrIGl0LgogICAgICAgIHVwZGF0ZWQgPSBzZWxmLm1lbW9yeS5kZXRhY2goKS5jbG9uZSgpCiAgICAgICAgdXBkYXRlZFtub2RlX2lkc10gPSBuZXdfbWVtb3J5CiAgICAgICAgc2VsZi5tZW1vcnkgPSB1cGRhdGVkCgogICAgICAgIGlmIHRpbWVzdGFtcHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYubGFzdF91cGRhdGUgPSBzZWxmLmxhc3RfdXBkYXRlLmRldGFjaCgpLmNsb25lKCkKICAgICAgICAgICAgc2VsZi5sYXN0X3VwZGF0ZVtub2RlX2lkc10gPSB0aW1lc3RhbXBzLmRldGFjaCgpCgogICAgZGVmIHJlc2V0X21lbW9yeShzZWxmLCBub2RlX2lkczogT3B0aW9uYWxbdG9yY2guVGVuc29yXSA9IE5vbmUpOgogICAgICAgICIiIlJlc2V0IG1lbW9yeSB0byB6ZXJvcyAoZGV0YWNoZWQpLiIiIgogICAgICAgIGRldmljZSA9IHNlbGYubWVtb3J5LmRldmljZQogICAgICAgIGlmIG5vZGVfaWRzIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYubWVtb3J5ID0gdG9yY2guemVyb3MoCiAgICAgICAgICAgICAgICBzZWxmLm51bV9ub2Rlcywgc2VsZi5tZW1vcnlfZGltLCBkZXZpY2U9ZGV2aWNlCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2VsZi5sYXN0X3VwZGF0ZSA9IHRvcmNoLnplcm9zKHNlbGYubnVtX25vZGVzLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYubWVtb3J5ID0gc2VsZi5tZW1vcnkuZGV0YWNoKCkuY2xvbmUoKQogICAgICAgICAgICBzZWxmLm1lbW9yeVtub2RlX2lkc10gPSAwLjAKICAgICAgICAgICAgc2VsZi5sYXN0X3VwZGF0ZSA9IHNlbGYubGFzdF91cGRhdGUuZGV0YWNoKCkuY2xvbmUoKQogICAgICAgICAgICBzZWxmLmxhc3RfdXBkYXRlW25vZGVfaWRzXSA9IDAuMAoKICAgIGRlZiBkZXRhY2hfbWVtb3J5KHNlbGYpOgogICAgICAgICIiIkRldGFjaCBtZW1vcnkgZnJvbSBjb21wdXRhdGlvbiBncmFwaCAoVEJQVFQgYm91bmRhcnkpLiIiIgogICAgICAgIHNlbGYubWVtb3J5ID0gc2VsZi5tZW1vcnkuZGV0YWNoKCkKICAgICAgICBzZWxmLmxhc3RfdXBkYXRlID0gc2VsZi5sYXN0X3VwZGF0ZS5kZXRhY2goKQo=", "models/tgn.py": "...", "models/baselines/static_gnn.py": "IiIiClN0YXRpYyBHTk4gYmFzZWxpbmU6IEdBVCB3aXRob3V0IHRlbXBvcmFsIGNvbXBvbmVudHMuClVzZXMgdGhlIHNhbWUgZ3JhcGggc3RydWN0dXJlIGJ1dCB3aXRob3V0IG1lbW9yeSBvciB0aW1lIGVuY29kaW5nLgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRvcmNoX2dlb21ldHJpYy5ubiBpbXBvcnQgR0FUQ29udiwgZ2xvYmFsX21lYW5fcG9vbCwgZ2xvYmFsX21heF9wb29sCgoKY2xhc3MgU3RhdGljR05OQ2FzY2FkZVByZWRpY3Rvcihubi5Nb2R1bGUpOgogICAgIiIiR3JhcGggQXR0ZW50aW9uIE5ldHdvcmsgYmFzZWxpbmUgd2l0aG91dCB0ZW1wb3JhbCBjb21wb25lbnRzLgoKICAgIEFibGF0aW9uOiBtZWFzdXJlcyB0aGUgY29udHJpYnV0aW9uIG9mIHRlbXBvcmFsIG1vZGVsaW5nIGJ5CiAgICBjb21wYXJpbmcgYWdhaW5zdCBhIHN0YXRpYyBncmFwaCBzbmFwc2hvdCBhcHByb2FjaC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG5vZGVfZmVhdHVyZV9kaW06IGludCwKICAgICAgICBoaWRkZW5fZGltOiBpbnQgPSAxMjgsCiAgICAgICAgbnVtX2xheWVyczogaW50ID0gMiwKICAgICAgICBoZWFkczogaW50ID0gNCwKICAgICAgICBwcmVkaWN0aW9uX2hvcml6b25zOiBsaXN0W2ludF0gPSBbMjQsIDcyLCAxNjgsIDcyMF0sCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSAwLjEsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucHJlZGljdGlvbl9ob3Jpem9ucyA9IHByZWRpY3Rpb25faG9yaXpvbnMKCiAgICAgICAgIyBJbnB1dCBwcm9qZWN0aW9uCiAgICAgICAgc2VsZi5pbnB1dF9wcm9qID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKG5vZGVfZmVhdHVyZV9kaW0sIGhpZGRlbl9kaW0pLAogICAgICAgICAgICBubi5MYXllck5vcm0oaGlkZGVuX2RpbSksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICApCgogICAgICAgICMgR0FUIGxheWVycwogICAgICAgIHNlbGYuZ2F0X2xheWVycyA9IG5uLk1vZHVsZUxpc3QoKQogICAgICAgIHNlbGYubm9ybXMgPSBubi5Nb2R1bGVMaXN0KCkKICAgICAgICBmb3IgaSBpbiByYW5nZShudW1fbGF5ZXJzKToKICAgICAgICAgICAgaW5fZGltID0gaGlkZGVuX2RpbSBpZiBpID09IDAgZWxzZSBoaWRkZW5fZGltICogaGVhZHMKICAgICAgICAgICAgc2VsZi5nYXRfbGF5ZXJzLmFwcGVuZCgKICAgICAgICAgICAgICAgIEdBVENvbnYoaW5fZGltLCBoaWRkZW5fZGltLCBoZWFkcz1oZWFkcywgZHJvcG91dD1kcm9wb3V0LCBjb25jYXQ9VHJ1ZSkKICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLm5vcm1zLmFwcGVuZChubi5MYXllck5vcm0oaGlkZGVuX2RpbSAqIGhlYWRzKSkKCiAgICAgICAgIyBGaW5hbCBwcm9qZWN0aW9uCiAgICAgICAgc2VsZi5maW5hbF9wcm9qID0gbm4uTGluZWFyKGhpZGRlbl9kaW0gKiBoZWFkcywgaGlkZGVuX2RpbSkKCiAgICAgICAgIyBHcmFwaCBwb29saW5nIGF0dGVudGlvbgogICAgICAgIHNlbGYucG9vbF9hdHRuID0gbm4uTGluZWFyKGhpZGRlbl9kaW0sIDEpCgogICAgICAgICMgUHJlZGljdGlvbiBoZWFkcwogICAgICAgIHNlbGYucHJlZGljdGlvbl9oZWFkcyA9IG5uLk1vZHVsZURpY3QoKQogICAgICAgIGZvciBoIGluIHByZWRpY3Rpb25faG9yaXpvbnM6CiAgICAgICAgICAgIHNlbGYucHJlZGljdGlvbl9oZWFkc1tmImhlYWRfe2h9aCJdID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW5fZGltICogMiwgaGlkZGVuX2RpbSksCiAgICAgICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICAgICAgbm4uTGluZWFyKGhpZGRlbl9kaW0sIDEpLAogICAgICAgICAgICApCgogICAgICAgIHNlbGYuc2V2ZXJpdHlfaGVhZCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW5fZGltICogMiwgNjQpLAogICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcig2NCwgMSksCiAgICAgICAgICAgIG5uLlNpZ21vaWQoKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBub2RlX2ZlYXR1cmVzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgZWRnZV9pbmRleDogdG9yY2guVGVuc29yLAogICAgICAgIGVkZ2VfYXR0cjogdG9yY2guVGVuc29yID0gTm9uZSwKICAgICkgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgbm9kZV9mZWF0dXJlczogW251bV9ub2RlcywgZmVhdHVyZV9kaW1dLgogICAgICAgICAgICBlZGdlX2luZGV4OiBbMiwgbnVtX2VkZ2VzXSAoaG9tb2dlbmVvdXMpLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IG9mIHByZWRpY3Rpb25zLgogICAgICAgICIiIgogICAgICAgIHggPSBzZWxmLmlucHV0X3Byb2oobm9kZV9mZWF0dXJlcykKCiAgICAgICAgZm9yIGdhdCwgbm9ybSBpbiB6aXAoc2VsZi5nYXRfbGF5ZXJzLCBzZWxmLm5vcm1zKToKICAgICAgICAgICAgeCA9IGdhdCh4LCBlZGdlX2luZGV4KQogICAgICAgICAgICB4ID0gbm9ybSh4KQogICAgICAgICAgICB4ID0gRi5lbHUoeCkKCiAgICAgICAgeCA9IHNlbGYuZmluYWxfcHJvaih4KSAgIyBbbnVtX25vZGVzLCBoaWRkZW5fZGltXQoKICAgICAgICAjIEF0dGVudGlvbiBwb29saW5nCiAgICAgICAgYXR0biA9IEYuc29mdG1heChzZWxmLnBvb2xfYXR0bih4KSwgZGltPTApCiAgICAgICAgZ3JhcGhfZW1iZWQgPSAoYXR0biAqIHgpLnN1bShkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIG1heF9lbWJlZCA9IHgubWF4KGRpbT0wLCBrZWVwZGltPVRydWUpLnZhbHVlcwogICAgICAgIGNvbWJpbmVkID0gdG9yY2guY2F0KFtncmFwaF9lbWJlZCwgbWF4X2VtYmVkXSwgZGltPS0xKQoKICAgICAgICBvdXRwdXRzID0ge30KICAgICAgICBmb3IgaCBpbiBzZWxmLnByZWRpY3Rpb25faG9yaXpvbnM6CiAgICAgICAgICAgIG91dHB1dHNbZiJjYXNjYWRlX3tofWgiXSA9IHNlbGYucHJlZGljdGlvbl9oZWFkc1tmImhlYWRfe2h9aCJdKAogICAgICAgICAgICAgICAgY29tYmluZWQKICAgICAgICAgICAgKS5zcXVlZXplKCkKCiAgICAgICAgb3V0cHV0c1sic2V2ZXJpdHkiXSA9IHNlbGYuc2V2ZXJpdHlfaGVhZChjb21iaW5lZCkuc3F1ZWV6ZSgpCiAgICAgICAgcmV0dXJuIG91dHB1dHMK", "models/baselines/lstm_model.py": "IiIiCkxTVE0gYmFzZWxpbmU6IHByb2Nlc3NlcyB0ZW1wb3JhbCBub2RlIGZlYXR1cmVzIHdpdGhvdXQgZ3JhcGggc3RydWN0dXJlLgpNZWFzdXJlcyB0aGUgY29udHJpYnV0aW9uIG9mIHRoZSBncmFwaCB0b3BvbG9neSB0byBwcmVkaWN0aW9uIHF1YWxpdHkuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCmNsYXNzIExTVE1DYXNjYWRlUHJlZGljdG9yKG5uLk1vZHVsZSk6CiAgICAiIiJMU1RNIGJhc2VsaW5lIHRoYXQgcHJvY2Vzc2VzIGNvbmNhdGVuYXRlZCBwcm90b2NvbCBmZWF0dXJlcyBhcwogICAgYSBtdWx0aXZhcmlhdGUgdGltZSBzZXJpZXMsIGlnbm9yaW5nIGdyYXBoIHN0cnVjdHVyZS4KCiAgICBBYmxhdGlvbjogcXVhbnRpZmllcyB0aGUgdmFsdWUgYWRkZWQgYnkgZ3JhcGgtYmFzZWQgbW9kZWxpbmcuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9kaW06IGludCwKICAgICAgICBoaWRkZW5fZGltOiBpbnQgPSAxMjgsCiAgICAgICAgbnVtX2xheWVyczogaW50ID0gMiwKICAgICAgICBudW1fbm9kZXM6IGludCA9IDE1LAogICAgICAgIHByZWRpY3Rpb25faG9yaXpvbnM6IGxpc3RbaW50XSA9IFsyNCwgNzIsIDE2OCwgNzIwXSwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMiwKICAgICAgICBiaWRpcmVjdGlvbmFsOiBib29sID0gRmFsc2UsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubnVtX25vZGVzID0gbnVtX25vZGVzCiAgICAgICAgc2VsZi5oaWRkZW5fZGltID0gaGlkZGVuX2RpbQogICAgICAgIHNlbGYucHJlZGljdGlvbl9ob3Jpem9ucyA9IHByZWRpY3Rpb25faG9yaXpvbnMKCiAgICAgICAgIyBGbGF0dGVuIGFsbCBub2RlIGZlYXR1cmVzIGludG8gb25lIHZlY3RvciBwZXIgdGltZXN0ZXAKICAgICAgICB0b3RhbF9pbnB1dCA9IGlucHV0X2RpbSAqIG51bV9ub2RlcwoKICAgICAgICBzZWxmLmlucHV0X3Byb2ogPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIodG90YWxfaW5wdXQsIGhpZGRlbl9kaW0gKiAyKSwKICAgICAgICAgICAgbm4uTGF5ZXJOb3JtKGhpZGRlbl9kaW0gKiAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICBubi5MaW5lYXIoaGlkZGVuX2RpbSAqIDIsIGhpZGRlbl9kaW0pLAogICAgICAgICkKCiAgICAgICAgc2VsZi5sc3RtID0gbm4uTFNUTSgKICAgICAgICAgICAgaW5wdXRfc2l6ZT1oaWRkZW5fZGltLAogICAgICAgICAgICBoaWRkZW5fc2l6ZT1oaWRkZW5fZGltLAogICAgICAgICAgICBudW1fbGF5ZXJzPW51bV9sYXllcnMsCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0PVRydWUsCiAgICAgICAgICAgIGRyb3BvdXQ9ZHJvcG91dCBpZiBudW1fbGF5ZXJzID4gMSBlbHNlIDAsCiAgICAgICAgICAgIGJpZGlyZWN0aW9uYWw9YmlkaXJlY3Rpb25hbCwKICAgICAgICApCgogICAgICAgIGxzdG1fb3V0X2RpbSA9IGhpZGRlbl9kaW0gKiAoMiBpZiBiaWRpcmVjdGlvbmFsIGVsc2UgMSkKCiAgICAgICAgIyBMYXllck5vcm0gb24gTFNUTSBvdXRwdXQgc3RhYmlsaXplcyB0cmFpbmluZwogICAgICAgIHNlbGYub3V0cHV0X25vcm0gPSBubi5MYXllck5vcm0obHN0bV9vdXRfZGltKQoKICAgICAgICAjIFByZWRpY3Rpb24gaGVhZHMKICAgICAgICBzZWxmLnByZWRpY3Rpb25faGVhZHMgPSBubi5Nb2R1bGVEaWN0KCkKICAgICAgICBmb3IgaCBpbiBwcmVkaWN0aW9uX2hvcml6b25zOgogICAgICAgICAgICBzZWxmLnByZWRpY3Rpb25faGVhZHNbZiJoZWFkX3tofWgiXSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBubi5MaW5lYXIobHN0bV9vdXRfZGltLCBoaWRkZW5fZGltKSwKICAgICAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgICAgIG5uLkRyb3BvdXQoZHJvcG91dCksCiAgICAgICAgICAgICAgICBubi5MaW5lYXIoaGlkZGVuX2RpbSwgMSksCiAgICAgICAgICAgICkKCiAgICAgICAgc2VsZi5zZXZlcml0eV9oZWFkID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGxzdG1fb3V0X2RpbSwgNjQpLAogICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcig2NCwgMSksCiAgICAgICAgICAgIG5uLlNpZ21vaWQoKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBmZWF0dXJlX3NlcXVlbmNlOiB0b3JjaC5UZW5zb3IsCiAgICApIC0+IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGZlYXR1cmVfc2VxdWVuY2U6IFtiYXRjaCwgc2VxX2xlbiwgbnVtX25vZGVzLCBmZWF0dXJlX2RpbV0KICAgICAgICAgICAgICAgIG9yIFtzZXFfbGVuLCBudW1fbm9kZXMsIGZlYXR1cmVfZGltXSAodW5iYXRjaGVkKS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCBvZiBwcmVkaWN0aW9ucy4KICAgICAgICAiIiIKICAgICAgICBpZiBmZWF0dXJlX3NlcXVlbmNlLmRpbSgpID09IDM6CiAgICAgICAgICAgIGZlYXR1cmVfc2VxdWVuY2UgPSBmZWF0dXJlX3NlcXVlbmNlLnVuc3F1ZWV6ZSgwKQoKICAgICAgICBiYXRjaCwgc2VxX2xlbiwgbnVtX25vZGVzLCBmZWF0X2RpbSA9IGZlYXR1cmVfc2VxdWVuY2Uuc2hhcGUKCiAgICAgICAgIyBGbGF0dGVuIG5vZGVzIGludG8gZmVhdHVyZSB2ZWN0b3IKICAgICAgICB4ID0gZmVhdHVyZV9zZXF1ZW5jZS52aWV3KGJhdGNoLCBzZXFfbGVuLCAtMSkKICAgICAgICB4ID0gc2VsZi5pbnB1dF9wcm9qKHgpCgogICAgICAgICMgTFNUTQogICAgICAgIGxzdG1fb3V0LCAoaF9uLCBjX24pID0gc2VsZi5sc3RtKHgpCiAgICAgICAgIyBVc2UgbGFzdCBoaWRkZW4gc3RhdGUgd2l0aCBMYXllck5vcm0KICAgICAgICBsYXN0X2hpZGRlbiA9IHNlbGYub3V0cHV0X25vcm0obHN0bV9vdXRbOiwgLTEsIDpdKSAgIyBbYmF0Y2gsIGxzdG1fb3V0X2RpbV0KCiAgICAgICAgb3V0cHV0cyA9IHt9CiAgICAgICAgZm9yIGggaW4gc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zOgogICAgICAgICAgICBvdXRwdXRzW2YiY2FzY2FkZV97aH1oIl0gPSBzZWxmLnByZWRpY3Rpb25faGVhZHNbZiJoZWFkX3tofWgiXSgKICAgICAgICAgICAgICAgIGxhc3RfaGlkZGVuCiAgICAgICAgICAgICkuc3F1ZWV6ZSgpCgogICAgICAgIG91dHB1dHNbInNldmVyaXR5Il0gPSBzZWxmLnNldmVyaXR5X2hlYWQobGFzdF9oaWRkZW4pLnNxdWVlemUoKQogICAgICAgIHJldHVybiBvdXRwdXRzCg==", "models/baselines/xgboost_model.py": "IiIiClhHQm9vc3QgYmFzZWxpbmU6IGdyYWRpZW50IGJvb3N0ZWQgdHJlZXMgb24gdGFidWxhciBmZWF0dXJlcy4KTWVhc3VyZXMgdmFsdWUgb2YgZGVlcCBsZWFybmluZyBhbmQgZ3JhcGggc3RydWN0dXJlIG92ZXIgdHJhZGl0aW9uYWwgTUwuCiIiIgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCmZyb20gbG9ndXJ1IGltcG9ydCBsb2dnZXIKCgpjbGFzcyBYR0Jvb3N0Q2FzY2FkZVByZWRpY3RvcjoKICAgICIiIlhHQm9vc3QgYmFzZWxpbmUgdXNpbmcgaGFuZC1jcmFmdGVkIHRhYnVsYXIgZmVhdHVyZXMuCgogICAgRmVhdHVyZXMgaW5jbHVkZTogYWdncmVnYXRlZCBwcm90b2NvbCBmZWF0dXJlcywgbmV0d29yayBjZW50cmFsaXR5CiAgICBtZXRyaWNzLCBhbmQgcm9sbGluZyBzdGF0aXN0aWNzIOKAlCB3aXRob3V0IGFueSBncmFwaCBuZXVyYWwgbmV0d29yawogICAgb3IgdGVtcG9yYWwgZGVlcCBsZWFybmluZyBjb21wb25lbnRzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJlZGljdGlvbl9ob3Jpem9uczogbGlzdFtpbnRdID0gWzI0LCA3MiwgMTY4LCA3MjBdLAogICAgICAgICoqeGdiX3BhcmFtcywKICAgICk6CiAgICAgICAgc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zID0gcHJlZGljdGlvbl9ob3Jpem9ucwogICAgICAgIHNlbGYueGdiX3BhcmFtcyA9IHsKICAgICAgICAgICAgIm5fZXN0aW1hdG9ycyI6IHhnYl9wYXJhbXMuZ2V0KCJuX2VzdGltYXRvcnMiLCA1MDApLAogICAgICAgICAgICAibWF4X2RlcHRoIjogeGdiX3BhcmFtcy5nZXQoIm1heF9kZXB0aCIsIDgpLAogICAgICAgICAgICAibGVhcm5pbmdfcmF0ZSI6IHhnYl9wYXJhbXMuZ2V0KCJsZWFybmluZ19yYXRlIiwgMC4wNSksCiAgICAgICAgICAgICJzdWJzYW1wbGUiOiB4Z2JfcGFyYW1zLmdldCgic3Vic2FtcGxlIiwgMC44KSwKICAgICAgICAgICAgImNvbHNhbXBsZV9ieXRyZWUiOiB4Z2JfcGFyYW1zLmdldCgiY29sc2FtcGxlX2J5dHJlZSIsIDAuOCksCiAgICAgICAgICAgICJtaW5fY2hpbGRfd2VpZ2h0IjogeGdiX3BhcmFtcy5nZXQoIm1pbl9jaGlsZF93ZWlnaHQiLCA1KSwKICAgICAgICAgICAgInJlZ19hbHBoYSI6IHhnYl9wYXJhbXMuZ2V0KCJyZWdfYWxwaGEiLCAwLjEpLAogICAgICAgICAgICAicmVnX2xhbWJkYSI6IHhnYl9wYXJhbXMuZ2V0KCJyZWdfbGFtYmRhIiwgMS4wKSwKICAgICAgICAgICAgIm9iamVjdGl2ZSI6ICJiaW5hcnk6bG9naXN0aWMiLAogICAgICAgICAgICAiZXZhbF9tZXRyaWMiOiAiYXVjIiwKICAgICAgICAgICAgInJhbmRvbV9zdGF0ZSI6IDQyLAogICAgICAgICAgICAibl9qb2JzIjogMSwKICAgICAgICB9CiAgICAgICAgc2VsZi5tb2RlbHMgPSB7fQogICAgICAgIHNlbGYuc2V2ZXJpdHlfbW9kZWwgPSBOb25lCiAgICAgICAgc2VsZi5mZWF0dXJlX25hbWVzID0gTm9uZQoKICAgIGRlZiBwcmVwYXJlX2ZlYXR1cmVzKAogICAgICAgIHNlbGYsCiAgICAgICAgbm9kZV9mZWF0dXJlc19zZXF1ZW5jZTogbGlzdFtucC5uZGFycmF5XSwKICAgICAgICB3aW5kb3c6IGludCA9IDMwLAogICAgKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIkNvbnZlcnQgdGVtcG9yYWwgZ3JhcGggZmVhdHVyZXMgaW50byB0YWJ1bGFyIGZlYXR1cmVzLgoKICAgICAgICBBZ2dyZWdhdGVzIG5vZGUgZmVhdHVyZXMgYWNyb3NzIGFsbCBwcm90b2NvbHMgYW5kIGNvbXB1dGVzCiAgICAgICAgcm9sbGluZyBzdGF0aXN0aWNzIG92ZXIgdGhlIHRlbXBvcmFsIHdpbmRvdy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgbm9kZV9mZWF0dXJlc19zZXF1ZW5jZTogTGlzdCBvZiBbbnVtX25vZGVzLCBmZWF0dXJlX2RpbV0gYXJyYXlzLgogICAgICAgICAgICB3aW5kb3c6IE51bWJlciBvZiBwYXN0IHRpbWVzdGVwcyB0byBpbmNsdWRlLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBUYWJ1bGFyIGZlYXR1cmUgbWF0cml4IFtudW1fc2FtcGxlcywgbnVtX2ZlYXR1cmVzXS4KICAgICAgICAiIiIKICAgICAgICBhbGxfZmVhdHVyZXMgPSBbXQogICAgICAgIHNlcV9sZW4gPSBsZW4obm9kZV9mZWF0dXJlc19zZXF1ZW5jZSkKCiAgICAgICAgZm9yIHQgaW4gcmFuZ2Uod2luZG93LCBzZXFfbGVuKToKICAgICAgICAgICAgZmVhdHVyZXMgPSBbXQoKICAgICAgICAgICAgIyBDdXJyZW50IHRpbWVzdGVwIGZlYXR1cmVzIChmbGF0dGVuZWQgYWNyb3NzIGFsbCBub2RlcykKICAgICAgICAgICAgY3VycmVudCA9IG5vZGVfZmVhdHVyZXNfc2VxdWVuY2VbdF0uZmxhdHRlbigpCiAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjdXJyZW50KQoKICAgICAgICAgICAgIyBBZ2dyZWdhdGVkIHN0YXRpc3RpY3MgYWNyb3NzIG5vZGVzCiAgICAgICAgICAgIGN1cnJlbnRfMmQgPSBub2RlX2ZlYXR1cmVzX3NlcXVlbmNlW3RdCiAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjdXJyZW50XzJkLm1lYW4oYXhpcz0wKSkgICMgbWVhbiBhY3Jvc3MgcHJvdG9jb2xzCiAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjdXJyZW50XzJkLnN0ZChheGlzPTApKSAgICMgc3RkIGFjcm9zcyBwcm90b2NvbHMKICAgICAgICAgICAgZmVhdHVyZXMuZXh0ZW5kKGN1cnJlbnRfMmQubWluKGF4aXM9MCkpICAgIyBtaW4gYWNyb3NzIHByb3RvY29scwogICAgICAgICAgICBmZWF0dXJlcy5leHRlbmQoY3VycmVudF8yZC5tYXgoYXhpcz0wKSkgICAjIG1heCBhY3Jvc3MgcHJvdG9jb2xzCgogICAgICAgICAgICAjIFRlbXBvcmFsIHN0YXRpc3RpY3Mgb3ZlciB3aW5kb3cKICAgICAgICAgICAgd2luZG93X2RhdGEgPSBucC5hcnJheSgKICAgICAgICAgICAgICAgIG5vZGVfZmVhdHVyZXNfc2VxdWVuY2VbbWF4KDAsIHQgLSB3aW5kb3cpOnRdCiAgICAgICAgICAgICkgICMgW3dpbmRvdywgbm9kZXMsIGZlYXR1cmVzXQoKICAgICAgICAgICAgIyBNZWFuIGNoYW5nZSBvdmVyIHdpbmRvdwogICAgICAgICAgICBpZiB3aW5kb3dfZGF0YS5zaGFwZVswXSA+IDE6CiAgICAgICAgICAgICAgICBjaGFuZ2VzID0gbnAuZGlmZih3aW5kb3dfZGF0YSwgYXhpcz0wKQogICAgICAgICAgICAgICAgZmVhdHVyZXMuZXh0ZW5kKGNoYW5nZXMubWVhbihheGlzPSgwLCAxKSkpICAjIGF2ZyBjaGFuZ2UKICAgICAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjaGFuZ2VzLnN0ZChheGlzPSgwLCAxKSkpICAgIyBjaGFuZ2Ugdm9sYXRpbGl0eQoKICAgICAgICAgICAgICAgICMgVHJlbmQ6IGZpcnN0IHZzIGxhc3QgaW4gd2luZG93CiAgICAgICAgICAgICAgICB0cmVuZCA9IHdpbmRvd19kYXRhWy0xXS5tZWFuKGF4aXM9MCkgLSB3aW5kb3dfZGF0YVswXS5tZWFuKGF4aXM9MCkKICAgICAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZCh0cmVuZCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG5fZmVhdCA9IGN1cnJlbnRfMmQuc2hhcGVbMV0KICAgICAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChucC56ZXJvcyhuX2ZlYXQgKiAzKSkKCiAgICAgICAgICAgIGFsbF9mZWF0dXJlcy5hcHBlbmQoZmVhdHVyZXMpCgogICAgICAgIHJldHVybiBucC5hcnJheShhbGxfZmVhdHVyZXMsIGR0eXBlPW5wLmZsb2F0MzIpCgogICAgZGVmIGZpdCgKICAgICAgICBzZWxmLAogICAgICAgIFg6IG5wLm5kYXJyYXksCiAgICAgICAgeV9kaWN0OiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICAgICAgZXZhbF9zZXQ6IE9wdGlvbmFsW3R1cGxlXSA9IE5vbmUsCiAgICApOgogICAgICAgICIiIlRyYWluIFhHQm9vc3QgbW9kZWxzIGZvciBlYWNoIHByZWRpY3Rpb24gaG9yaXpvbi4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgWDogRmVhdHVyZSBtYXRyaXggW251bV9zYW1wbGVzLCBudW1fZmVhdHVyZXNdLgogICAgICAgICAgICB5X2RpY3Q6IERpY3QgbWFwcGluZyAiY2FzY2FkZV97aH1oIiAtPiBiaW5hcnkgbGFiZWxzLgogICAgICAgICAgICBldmFsX3NldDogT3B0aW9uYWwgKFhfdmFsLCB5X3ZhbF9kaWN0KSBmb3IgZWFybHkgc3RvcHBpbmcuCiAgICAgICAgIiIiCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgeGdib29zdCBhcyB4Z2IKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIGxvZ2dlci5lcnJvcigiWEdCb29zdCBub3QgaW5zdGFsbGVkLiBwaXAgaW5zdGFsbCB4Z2Jvb3N0IikKICAgICAgICAgICAgcmV0dXJuCgogICAgICAgICMgUmVwbGFjZSBOYU4vSW5mCiAgICAgICAgWCA9IG5wLm5hbl90b19udW0oWCwgbmFuPTAuMCwgcG9zaW5mPTFlNiwgbmVnaW5mPS0xZTYpCgogICAgICAgIGZvciBob3Jpem9uX2tleSwgeSBpbiB5X2RpY3QuaXRlbXMoKToKICAgICAgICAgICAgbG9nZ2VyLmluZm8oCiAgICAgICAgICAgICAgICBmIlRyYWluaW5nIFhHQm9vc3QgZm9yIHtob3Jpem9uX2tleX0gIgogICAgICAgICAgICAgICAgZiIocG9zX3JhdGU6IHt5Lm1lYW4oKTouNGZ9KSIKICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyBIYW5kbGUgY2xhc3MgaW1iYWxhbmNlCiAgICAgICAgICAgIHBvc19jb3VudCA9IHkuc3VtKCkKICAgICAgICAgICAgbmVnX2NvdW50ID0gbGVuKHkpIC0gcG9zX2NvdW50CiAgICAgICAgICAgIHNjYWxlX3BvcyA9IG5lZ19jb3VudCAvIG1heChwb3NfY291bnQsIDEpCgogICAgICAgICAgICBwYXJhbXMgPSB7KipzZWxmLnhnYl9wYXJhbXMsICJzY2FsZV9wb3Nfd2VpZ2h0Ijogc2NhbGVfcG9zfQogICAgICAgICAgICBtb2RlbCA9IHhnYi5YR0JDbGFzc2lmaWVyKCoqcGFyYW1zKQoKICAgICAgICAgICAgZml0X2t3YXJncyA9IHt9CiAgICAgICAgICAgIGlmIGV2YWxfc2V0IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgWF92YWwsIHlfdmFsX2RpY3QgPSBldmFsX3NldAogICAgICAgICAgICAgICAgWF92YWwgPSBucC5uYW5fdG9fbnVtKFhfdmFsLCBuYW49MC4wLCBwb3NpbmY9MWU2LCBuZWdpbmY9LTFlNikKICAgICAgICAgICAgICAgIGZpdF9rd2FyZ3NbImV2YWxfc2V0Il0gPSBbKFhfdmFsLCB5X3ZhbF9kaWN0W2hvcml6b25fa2V5XSldCiAgICAgICAgICAgICAgICBmaXRfa3dhcmdzWyJ2ZXJib3NlIl0gPSBGYWxzZQoKICAgICAgICAgICAgaWYgInZlcmJvc2UiIG5vdCBpbiBmaXRfa3dhcmdzOgogICAgICAgICAgICAgICAgZml0X2t3YXJnc1sidmVyYm9zZSJdID0gRmFsc2UKICAgICAgICAgICAgbW9kZWwuZml0KFgsIHksICoqZml0X2t3YXJncykKICAgICAgICAgICAgc2VsZi5tb2RlbHNbaG9yaXpvbl9rZXldID0gbW9kZWwKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiIgIEJlc3QgaXRlcmF0aW9uOiB7bW9kZWwuYmVzdF9pdGVyYXRpb24gaWYgaGFzYXR0cihtb2RlbCwgJ2Jlc3RfaXRlcmF0aW9uJykgZWxzZSAnTi9BJ30iKQoKICAgIGRlZiBwcmVkaWN0KHNlbGYsIFg6IG5wLm5kYXJyYXkpIC0+IGRpY3Rbc3RyLCBucC5uZGFycmF5XToKICAgICAgICAiIiJHZW5lcmF0ZSBjYXNjYWRlIHByb2JhYmlsaXR5IHByZWRpY3Rpb25zLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBYOiBGZWF0dXJlIG1hdHJpeCBbbnVtX3NhbXBsZXMsIG51bV9mZWF0dXJlc10uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIERpY3QgbWFwcGluZyBob3Jpem9uIGtleSAtPiBwcm9iYWJpbGl0eSBhcnJheS4KICAgICAgICAiIiIKICAgICAgICBYID0gbnAubmFuX3RvX251bShYLCBuYW49MC4wLCBwb3NpbmY9MWU2LCBuZWdpbmY9LTFlNikKICAgICAgICBwcmVkaWN0aW9ucyA9IHt9CiAgICAgICAgZm9yIGhvcml6b25fa2V5LCBtb2RlbCBpbiBzZWxmLm1vZGVscy5pdGVtcygpOgogICAgICAgICAgICBwcmVkaWN0aW9uc1tob3Jpem9uX2tleV0gPSBtb2RlbC5wcmVkaWN0X3Byb2JhKFgpWzosIDFdCiAgICAgICAgcmV0dXJuIHByZWRpY3Rpb25zCgogICAgZGVmIGdldF9mZWF0dXJlX2ltcG9ydGFuY2Uoc2VsZikgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgICAgICIiIkdldCBmZWF0dXJlIGltcG9ydGFuY2UgZm9yIGVhY2ggaG9yaXpvbiBtb2RlbC4iIiIKICAgICAgICBpbXBvcnRhbmNlcyA9IHt9CiAgICAgICAgZm9yIGhvcml6b25fa2V5LCBtb2RlbCBpbiBzZWxmLm1vZGVscy5pdGVtcygpOgogICAgICAgICAgICBpbXBvcnRhbmNlc1tob3Jpem9uX2tleV0gPSBtb2RlbC5mZWF0dXJlX2ltcG9ydGFuY2VzXwogICAgICAgIHJldHVybiBpbXBvcnRhbmNlcwo=", "models/baselines/sir_contagion.py": "IiIiClNJUiBDb250YWdpb24gTW9kZWwgYmFzZWxpbmUuCkFkYXB0cyB0aGUgZXBpZGVtaW9sb2dpY2FsIFN1c2NlcHRpYmxlLUluZmVjdGVkLVJlY292ZXJlZCBtb2RlbAp0byBEZUZpIHByb3RvY29sIGNvbnRhZ2lvbiBkeW5hbWljcy4KIiIiCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5pbnRlZ3JhdGUgaW1wb3J0IG9kZWludApmcm9tIHNjaXB5Lm9wdGltaXplIGltcG9ydCBtaW5pbWl6ZQpmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCgoKY2xhc3MgU0lSQ29udGFnaW9uTW9kZWw6CiAgICAiIiJTSVIgZXBpZGVtaWMgbW9kZWwgYWRhcHRlZCBmb3IgRGVGaSBsaXF1aWRhdGlvbiBjYXNjYWRlcy4KCiAgICBNb2RlbHMgZWFjaCBwcm90b2NvbCBhcyBlaXRoZXI6CiAgICAgIC0gUyAoU3VzY2VwdGlibGUpOiBvcGVyYXRpbmcgbm9ybWFsbHkgYnV0IGV4cG9zZWQgdG8gY2FzY2FkZSByaXNrCiAgICAgIC0gSSAoSW5mZWN0ZWQpOiBleHBlcmllbmNpbmcgbGlxdWlkYXRpb24gY2FzY2FkZSAvIFRWTCBjcmlzaXMKICAgICAgLSBSIChSZWNvdmVyZWQpOiBzdGFiaWxpemVkIGFmdGVyIGNhc2NhZGUKCiAgICBUaGUgaW5mZWN0aW9uIHJhdGUgYmV0YSBkZXBlbmRzIG9uIHRoZSBuZXR3b3JrIGFkamFjZW5jeSAoc2hhcmVkCiAgICBjb2xsYXRlcmFsLCBvcmFjbGUgZGVwZW5kZW5jaWVzLCBldGMuKSwgYW5kIHRoZSByZWNvdmVyeSByYXRlIGdhbW1hCiAgICBkZXBlbmRzIG9uIHByb3RvY29sIHJlc2lsaWVuY2UgZmFjdG9ycy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG51bV9wcm90b2NvbHM6IGludCwKICAgICAgICBhZGphY2VuY3lfbWF0cml4OiBucC5uZGFycmF5LAogICAgICAgIGJldGFfcmFuZ2U6IHR1cGxlID0gKDAuMDEsIDAuNSksCiAgICAgICAgZ2FtbWFfcmFuZ2U6IHR1cGxlID0gKDAuMDEsIDAuMyksCiAgICAgICAgbl9zaW11bGF0aW9uczogaW50ID0gMTAwMCwKICAgICAgICBwcmVkaWN0aW9uX2hvcml6b25zOiBsaXN0W2ludF0gPSBbMjQsIDcyLCAxNjgsIDcyMF0sCiAgICApOgogICAgICAgIHNlbGYubnVtX3Byb3RvY29scyA9IG51bV9wcm90b2NvbHMKICAgICAgICBzZWxmLmFkaiA9IGFkamFjZW5jeV9tYXRyaXgKICAgICAgICBzZWxmLmJldGFfcmFuZ2UgPSBiZXRhX3JhbmdlCiAgICAgICAgc2VsZi5nYW1tYV9yYW5nZSA9IGdhbW1hX3JhbmdlCiAgICAgICAgc2VsZi5uX3NpbXVsYXRpb25zID0gbl9zaW11bGF0aW9ucwogICAgICAgIHNlbGYucHJlZGljdGlvbl9ob3Jpem9ucyA9IHByZWRpY3Rpb25faG9yaXpvbnMKCiAgICAgICAgIyBGaXR0ZWQgcGFyYW1ldGVycwogICAgICAgIHNlbGYuYmV0YSA9IE5vbmUKICAgICAgICBzZWxmLmdhbW1hID0gTm9uZQogICAgICAgIHNlbGYucHJvdG9jb2xfdnVsbmVyYWJpbGl0eSA9IG5wLm9uZXMobnVtX3Byb3RvY29scykKCiAgICBkZWYgX3Npcl9vZGUoc2VsZiwgeSwgdCwgYmV0YSwgZ2FtbWEsIGFkaik6CiAgICAgICAgIiIiU0lSIE9ERSBzeXN0ZW0gYWRhcHRlZCBmb3IgbmV0d29yayBjb250YWdpb24uIiIiCiAgICAgICAgbiA9IHNlbGYubnVtX3Byb3RvY29scwogICAgICAgIFMgPSB5WzpuXQogICAgICAgIEkgPSB5W246MipuXQogICAgICAgIFIgPSB5WzIqbjpdCgogICAgICAgICMgTmV0d29yay1tZWRpYXRlZCBpbmZlY3Rpb246IGVhY2ggbm9kZSdzIGluZmVjdGlvbiByYXRlIGRlcGVuZHMKICAgICAgICAjIG9uIHRoZSBpbmZlY3Rpb24gbGV2ZWwgb2YgaXRzIG5laWdoYm9ycyB3ZWlnaHRlZCBieSBhZGphY2VuY3kKICAgICAgICBuZWlnaGJvcl9pbmZlY3Rpb24gPSBhZGogQCBJICAjIHdlaWdodGVkIHN1bSBvZiBpbmZlY3RlZCBuZWlnaGJvcnMKICAgICAgICBpbmZlY3Rpb25fcmF0ZSA9IGJldGEgKiBTICogbmVpZ2hib3JfaW5mZWN0aW9uICogc2VsZi5wcm90b2NvbF92dWxuZXJhYmlsaXR5CgogICAgICAgIHJlY292ZXJ5X3JhdGUgPSBnYW1tYSAqIEkKCiAgICAgICAgZFNkdCA9IC1pbmZlY3Rpb25fcmF0ZQogICAgICAgIGRJZHQgPSBpbmZlY3Rpb25fcmF0ZSAtIHJlY292ZXJ5X3JhdGUKICAgICAgICBkUmR0ID0gcmVjb3ZlcnlfcmF0ZQoKICAgICAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoW2RTZHQsIGRJZHQsIGRSZHRdKQoKICAgIGRlZiBmaXQoCiAgICAgICAgc2VsZiwKICAgICAgICBjYXNjYWRlX2V2ZW50czogbGlzdFtkaWN0XSwKICAgICAgICBwcm90b2NvbF90dmxfY2hhbmdlczogbnAubmRhcnJheSwKICAgICk6CiAgICAgICAgIiIiRml0IFNJUiBwYXJhbWV0ZXJzIHRvIGhpc3RvcmljYWwgY2FzY2FkZSBldmVudHMuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGNhc2NhZGVfZXZlbnRzOiBMaXN0IG9mIGNhc2NhZGUgZXZlbnQgZGljdHMgd2l0aCBzZXZlcml0eSBpbmZvLgogICAgICAgICAgICBwcm90b2NvbF90dmxfY2hhbmdlczogW251bV9ldmVudHMsIG51bV9wcm90b2NvbHNdIFRWTCBjaGFuZ2VzCiAgICAgICAgICAgICAgICBkdXJpbmcgZWFjaCBldmVudCAobmVnYXRpdmUgPSBhZmZlY3RlZCkuCiAgICAgICAgIiIiCiAgICAgICAgbG9nZ2VyLmluZm8oIkZpdHRpbmcgU0lSIGNvbnRhZ2lvbiBtb2RlbCBwYXJhbWV0ZXJzIikKCiAgICAgICAgZGVmIG9iamVjdGl2ZShwYXJhbXMpOgogICAgICAgICAgICBiZXRhLCBnYW1tYSA9IHBhcmFtcwogICAgICAgICAgICB0b3RhbF9lcnJvciA9IDAuMAoKICAgICAgICAgICAgZm9yIGksIGV2ZW50IGluIGVudW1lcmF0ZShjYXNjYWRlX2V2ZW50cyk6CiAgICAgICAgICAgICAgICBpZiBpID49IGxlbihwcm90b2NvbF90dmxfY2hhbmdlcyk6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgICAgICBvYnNlcnZlZCA9IHByb3RvY29sX3R2bF9jaGFuZ2VzW2ldCiAgICAgICAgICAgICAgICAjIERldGVybWluZSBpbml0aWFsbHkgaW5mZWN0ZWQgcHJvdG9jb2xzCiAgICAgICAgICAgICAgICBpbml0aWFsX2luZmVjdGVkID0gKG9ic2VydmVkIDwgLTAuMDUpLmFzdHlwZShmbG9hdCkKICAgICAgICAgICAgICAgIGlmIGluaXRpYWxfaW5mZWN0ZWQuc3VtKCkgPT0gMDoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgICAgICMgUnVuIFNJUiBzaW11bGF0aW9uCiAgICAgICAgICAgICAgICBwcmVkaWN0ZWQgPSBzZWxmLl9zaW11bGF0ZV9zaW5nbGUoCiAgICAgICAgICAgICAgICAgICAgYmV0YSwgZ2FtbWEsIGluaXRpYWxfaW5mZWN0ZWQsIGR1cmF0aW9uPTcKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICMgQ29tcGFyZSBwcmVkaWN0ZWQgaW5mZWN0aW9uIHNwcmVhZCB3aXRoIG9ic2VydmVkCiAgICAgICAgICAgICAgICB0b3RhbF9lcnJvciArPSBucC5tZWFuKAogICAgICAgICAgICAgICAgICAgIChwcmVkaWN0ZWQgLSAob2JzZXJ2ZWQgPCAtMC4wNSkuYXN0eXBlKGZsb2F0KSkgKiogMgogICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgcmV0dXJuIHRvdGFsX2Vycm9yCgogICAgICAgICMgT3B0aW1pemUKICAgICAgICByZXN1bHQgPSBtaW5pbWl6ZSgKICAgICAgICAgICAgb2JqZWN0aXZlLAogICAgICAgICAgICB4MD1bMC4xLCAwLjFdLAogICAgICAgICAgICBib3VuZHM9W3NlbGYuYmV0YV9yYW5nZSwgc2VsZi5nYW1tYV9yYW5nZV0sCiAgICAgICAgICAgIG1ldGhvZD0iTC1CRkdTLUIiLAogICAgICAgICkKCiAgICAgICAgc2VsZi5iZXRhLCBzZWxmLmdhbW1hID0gcmVzdWx0LngKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgZiJGaXR0ZWQgU0lSIHBhcmFtczogYmV0YT17c2VsZi5iZXRhOi40Zn0sIGdhbW1hPXtzZWxmLmdhbW1hOi40Zn0iCiAgICAgICAgKQoKICAgICAgICAjIEVzdGltYXRlIHBlci1wcm90b2NvbCB2dWxuZXJhYmlsaXR5IGZyb20gaGlzdG9yaWNhbCBkYXRhCiAgICAgICAgaWYgbGVuKHByb3RvY29sX3R2bF9jaGFuZ2VzKSA+IDA6CiAgICAgICAgICAgIGF2Z19pbXBhY3QgPSBucC5tZWFuKAogICAgICAgICAgICAgICAgbnAuYWJzKHByb3RvY29sX3R2bF9jaGFuZ2VzKSwgYXhpcz0wCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2VsZi5wcm90b2NvbF92dWxuZXJhYmlsaXR5ID0gYXZnX2ltcGFjdCAvIChhdmdfaW1wYWN0Lm1lYW4oKSArIDFlLTgpCgogICAgZGVmIF9zaW11bGF0ZV9zaW5nbGUoCiAgICAgICAgc2VsZiwKICAgICAgICBiZXRhOiBmbG9hdCwKICAgICAgICBnYW1tYTogZmxvYXQsCiAgICAgICAgaW5pdGlhbF9pbmZlY3RlZDogbnAubmRhcnJheSwKICAgICAgICBkdXJhdGlvbjogaW50ID0gNywKICAgICkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJSdW4gYSBzaW5nbGUgU0lSIHNpbXVsYXRpb24uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIFBlYWsgaW5mZWN0aW9uIGxldmVsIHBlciBwcm90b2NvbC4KICAgICAgICAiIiIKICAgICAgICBuID0gc2VsZi5udW1fcHJvdG9jb2xzCiAgICAgICAgUzAgPSAxLjAgLSBpbml0aWFsX2luZmVjdGVkCiAgICAgICAgSTAgPSBpbml0aWFsX2luZmVjdGVkLmNvcHkoKQogICAgICAgIFIwID0gbnAuemVyb3MobikKCiAgICAgICAgeTAgPSBucC5jb25jYXRlbmF0ZShbUzAsIEkwLCBSMF0pCiAgICAgICAgdCA9IG5wLmxpbnNwYWNlKDAsIGR1cmF0aW9uLCBkdXJhdGlvbiAqIDI0KSAgIyBob3VybHkKCiAgICAgICAgc29sdXRpb24gPSBvZGVpbnQoCiAgICAgICAgICAgIHNlbGYuX3Npcl9vZGUsIHkwLCB0LCBhcmdzPShiZXRhLCBnYW1tYSwgc2VsZi5hZGopCiAgICAgICAgKQoKICAgICAgICAjIEV4dHJhY3QgcGVhayBpbmZlY3Rpb24gbGV2ZWwgcGVyIHByb3RvY29sCiAgICAgICAgSV90aW1lc2VyaWVzID0gc29sdXRpb25bOiwgbjoyKm5dCiAgICAgICAgcGVha19pbmZlY3Rpb24gPSBJX3RpbWVzZXJpZXMubWF4KGF4aXM9MCkKCiAgICAgICAgcmV0dXJuIHBlYWtfaW5mZWN0aW9uCgogICAgZGVmIHByZWRpY3QoCiAgICAgICAgc2VsZiwKICAgICAgICBjdXJyZW50X3N0YXRlOiBucC5uZGFycmF5LAogICAgKSAtPiBkaWN0W3N0ciwgbnAubmRhcnJheV06CiAgICAgICAgIiIiUHJlZGljdCBjYXNjYWRlIHByb2JhYmlsaXR5IHVzaW5nIE1vbnRlIENhcmxvIFNJUiBzaW11bGF0aW9ucy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgY3VycmVudF9zdGF0ZTogW251bV9wcm90b2NvbHNdIGN1cnJlbnQgcmlzayBzdGF0ZSAoMC0xKS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCB3aXRoIGNhc2NhZGUgcHJvYmFiaWxpdHkgcGVyIGhvcml6b24uCiAgICAgICAgIiIiCiAgICAgICAgaWYgc2VsZi5iZXRhIGlzIE5vbmU6CiAgICAgICAgICAgICMgVXNlIGRlZmF1bHQgcGFyYW1ldGVycwogICAgICAgICAgICBzZWxmLmJldGEgPSAwLjEKICAgICAgICAgICAgc2VsZi5nYW1tYSA9IDAuMQoKICAgICAgICBwcmVkaWN0aW9ucyA9IHtmImNhc2NhZGVfe2h9aCI6IFtdIGZvciBoIGluIHNlbGYucHJlZGljdGlvbl9ob3Jpem9uc30KCiAgICAgICAgZm9yIF8gaW4gcmFuZ2Uoc2VsZi5uX3NpbXVsYXRpb25zKToKICAgICAgICAgICAgIyBSYW5kb20gcGVydHVyYmF0aW9uIHRvIGluaXRpYWwgc3RhdGUKICAgICAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKCkKICAgICAgICAgICAgbm9pc2UgPSBybmcubm9ybWFsKDAsIDAuMDUsIHNlbGYubnVtX3Byb3RvY29scykKICAgICAgICAgICAgcGVydHVyYmVkID0gbnAuY2xpcChjdXJyZW50X3N0YXRlICsgbm9pc2UsIDAsIDEpCgogICAgICAgICAgICAjIEluaXRpYWwgaW5mZWN0ZWQgPSBwcm90b2NvbHMgd2l0aCBoaWdoIHJpc2sKICAgICAgICAgICAgaW5pdGlhbF9pbmZlY3RlZCA9IChwZXJ0dXJiZWQgPiAwLjUpLmFzdHlwZShmbG9hdCkKCiAgICAgICAgICAgICMgU2ltdWxhdGUKICAgICAgICAgICAgcGVhayA9IHNlbGYuX3NpbXVsYXRlX3NpbmdsZSgKICAgICAgICAgICAgICAgIHNlbGYuYmV0YSwgc2VsZi5nYW1tYSwgaW5pdGlhbF9pbmZlY3RlZCwgZHVyYXRpb249NwogICAgICAgICAgICApCgogICAgICAgICAgICAjIENoZWNrIGlmIGNhc2NhZGUgb2NjdXJzIGF0IGVhY2ggaG9yaXpvbgogICAgICAgICAgICBmb3IgaCBpbiBzZWxmLnByZWRpY3Rpb25faG9yaXpvbnM6CiAgICAgICAgICAgICAgICAjIENhc2NhZGUgPSA+MzAlIG9mIHByb3RvY29scyBpbmZlY3RlZCBhYm92ZSB0aHJlc2hvbGQKICAgICAgICAgICAgICAgIGNhc2NhZGUgPSAocGVhayA+IDAuMykuc3VtKCkgPj0gc2VsZi5udW1fcHJvdG9jb2xzICogMC4zCiAgICAgICAgICAgICAgICBwcmVkaWN0aW9uc1tmImNhc2NhZGVfe2h9aCJdLmFwcGVuZChmbG9hdChjYXNjYWRlKSkKCiAgICAgICAgIyBBZ2dyZWdhdGUgYWNyb3NzIHNpbXVsYXRpb25zCiAgICAgICAgcmVzdWx0ID0ge30KICAgICAgICBmb3Iga2V5LCB2YWxzIGluIHByZWRpY3Rpb25zLml0ZW1zKCk6CiAgICAgICAgICAgIHJlc3VsdFtrZXldID0gbnAuYXJyYXkodmFscykubWVhbigpCgogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgcHJlZGljdF9iYXRjaCgKICAgICAgICBzZWxmLCBzdGF0ZXM6IG5wLm5kYXJyYXkKICAgICkgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgICAgICIiIlByZWRpY3QgZm9yIGEgYmF0Y2ggb2Ygc3RhdGVzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBzdGF0ZXM6IFtiYXRjaF9zaXplLCBudW1fcHJvdG9jb2xzXSByaXNrIHN0YXRlcy4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCB3aXRoIGNhc2NhZGUgcHJvYmFiaWxpdHkgYXJyYXlzIHBlciBob3Jpem9uLgogICAgICAgICIiIgogICAgICAgIGJhdGNoX3Jlc3VsdHMgPSB7CiAgICAgICAgICAgIGYiY2FzY2FkZV97aH1oIjogW10gZm9yIGggaW4gc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zCiAgICAgICAgfQoKICAgICAgICBmb3Igc3RhdGUgaW4gc3RhdGVzOgogICAgICAgICAgICBwcmVkID0gc2VsZi5wcmVkaWN0KHN0YXRlKQogICAgICAgICAgICBmb3Iga2V5IGluIGJhdGNoX3Jlc3VsdHM6CiAgICAgICAgICAgICAgICBiYXRjaF9yZXN1bHRzW2tleV0uYXBwZW5kKHByZWRba2V5XSkKCiAgICAgICAgcmV0dXJuIHtrOiBucC5hcnJheSh2KSBmb3IgaywgdiBpbiBiYXRjaF9yZXN1bHRzLml0ZW1zKCl9Cg==", "models/baselines/centrality_model.py": "IiIiCk5ldHdvcmsgQ2VudHJhbGl0eSBiYXNlbGluZS4KUHJlZGljdHMgY2FzY2FkZSByaXNrIHVzaW5nIGdyYXBoLXRoZW9yZXRpYyBjZW50cmFsaXR5IG1ldHJpY3MKY29tYmluZWQgd2l0aCBwcm90b2NvbC1sZXZlbCByaXNrIGluZGljYXRvcnMuCiIiIgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBuZXR3b3JreCBhcyBueApmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IFN0YW5kYXJkU2NhbGVyCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbApmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCgoKY2xhc3MgQ2VudHJhbGl0eU1vZGVsOgogICAgIiIiQ2VudHJhbGl0eS1iYXNlZCBjYXNjYWRlIHByZWRpY3Rvci4KCiAgICBVc2VzIG5ldHdvcmsgY2VudHJhbGl0eSBtZXRyaWNzIChiZXR3ZWVubmVzcywgZWlnZW52ZWN0b3IsIFBhZ2VSYW5rKQogICAgY29tYmluZWQgd2l0aCBiYXNpYyBwcm90b2NvbCBmZWF0dXJlcyBhcyBpbnB1dCB0byBsb2dpc3RpYyByZWdyZXNzaW9uLgoKICAgIEFibGF0aW9uOiBtZWFzdXJlcyB0aGUgdmFsdWUgb2YgZGVlcCBncmFwaCBsZWFybmluZyBvdmVyIHNpbXBsZQogICAgbmV0d29yayBzY2llbmNlIG1ldHJpY3MuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBhZGphY2VuY3lfbWF0cml4OiBucC5uZGFycmF5LAogICAgICAgIHByZWRpY3Rpb25faG9yaXpvbnM6IGxpc3RbaW50XSA9IFsyNCwgNzIsIDE2OCwgNzIwXSwKICAgICk6CiAgICAgICAgc2VsZi5hZGogPSBhZGphY2VuY3lfbWF0cml4CiAgICAgICAgc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zID0gcHJlZGljdGlvbl9ob3Jpem9ucwogICAgICAgIHNlbGYubW9kZWxzID0ge30KICAgICAgICBzZWxmLnNjYWxlciA9IFN0YW5kYXJkU2NhbGVyKCkKICAgICAgICBzZWxmLl9jb21wdXRlX2NlbnRyYWxpdHlfbWV0cmljcygpCgogICAgZGVmIF9jb21wdXRlX2NlbnRyYWxpdHlfbWV0cmljcyhzZWxmKToKICAgICAgICAiIiJDb21wdXRlIHN0YXRpYyBncmFwaCBjZW50cmFsaXR5IG1ldHJpY3MuIiIiCiAgICAgICAgRyA9IG54LmZyb21fbnVtcHlfYXJyYXkoc2VsZi5hZGosIGNyZWF0ZV91c2luZz1ueC5EaUdyYXBoKQogICAgICAgIG4gPSBsZW4oc2VsZi5hZGopCgogICAgICAgIHNlbGYuY2VudHJhbGl0eSA9IHsKICAgICAgICAgICAgImRlZ3JlZSI6IG5wLmFycmF5KAogICAgICAgICAgICAgICAgW254LmRlZ3JlZV9jZW50cmFsaXR5KEcpLmdldChpLCAwKSBmb3IgaSBpbiByYW5nZShuKV0KICAgICAgICAgICAgKSwKICAgICAgICAgICAgImJldHdlZW5uZXNzIjogbnAuYXJyYXkoCiAgICAgICAgICAgICAgICBbbnguYmV0d2Vlbm5lc3NfY2VudHJhbGl0eShHKS5nZXQoaSwgMCkgZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJjbG9zZW5lc3MiOiBucC5hcnJheSgKICAgICAgICAgICAgICAgIFtueC5jbG9zZW5lc3NfY2VudHJhbGl0eShHKS5nZXQoaSwgMCkgZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJwYWdlcmFuayI6IG5wLmFycmF5KAogICAgICAgICAgICAgICAgW254LnBhZ2VyYW5rKEcsIHdlaWdodD0id2VpZ2h0IikuZ2V0KGksIDApIGZvciBpIGluIHJhbmdlKG4pXQogICAgICAgICAgICApLAogICAgICAgIH0KCiAgICAgICAgdHJ5OgogICAgICAgICAgICBlaWdlbiA9IG54LmVpZ2VudmVjdG9yX2NlbnRyYWxpdHlfbnVtcHkoRywgd2VpZ2h0PSJ3ZWlnaHQiKQogICAgICAgICAgICBzZWxmLmNlbnRyYWxpdHlbImVpZ2VudmVjdG9yIl0gPSBucC5hcnJheSgKICAgICAgICAgICAgICAgIFtlaWdlbi5nZXQoaSwgMCkgZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBzZWxmLmNlbnRyYWxpdHlbImVpZ2VudmVjdG9yIl0gPSBucC56ZXJvcyhuKQoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYuY2VudHJhbGl0eVsiY2x1c3RlcmluZyJdID0gbnAuYXJyYXkoCiAgICAgICAgICAgICAgICBbCiAgICAgICAgICAgICAgICAgICAgbnguY2x1c3RlcmluZyhHLnRvX3VuZGlyZWN0ZWQoKSwgd2VpZ2h0PSJ3ZWlnaHQiKS5nZXQoaSwgMCkKICAgICAgICAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShuKQogICAgICAgICAgICAgICAgXQogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgc2VsZi5jZW50cmFsaXR5WyJjbHVzdGVyaW5nIl0gPSBucC56ZXJvcyhuKQoKICAgICAgICBsb2dnZXIuaW5mbygiQ2VudHJhbGl0eSBtZXRyaWNzIGNvbXB1dGVkIikKCiAgICBkZWYgcHJlcGFyZV9mZWF0dXJlcygKICAgICAgICBzZWxmLAogICAgICAgIG5vZGVfZmVhdHVyZXNfc2VxdWVuY2U6IGxpc3RbbnAubmRhcnJheV0sCiAgICAgICAgd2luZG93OiBpbnQgPSAzMCwKICAgICkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJDb21iaW5lIGNlbnRyYWxpdHkgd2l0aCBiYXNpYyBwcm90b2NvbCBmZWF0dXJlcy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgbm9kZV9mZWF0dXJlc19zZXF1ZW5jZTogTGlzdCBvZiBbbnVtX25vZGVzLCBmZWF0dXJlX2RpbV0gYXJyYXlzLgogICAgICAgICAgICB3aW5kb3c6IExvb2tiYWNrIHdpbmRvdy4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRmVhdHVyZSBtYXRyaXggW251bV9zYW1wbGVzLCBudW1fZmVhdHVyZXNdLgogICAgICAgICIiIgogICAgICAgIGFsbF9mZWF0dXJlcyA9IFtdCiAgICAgICAgbiA9IGxlbihzZWxmLmFkaikKCiAgICAgICAgZm9yIHQgaW4gcmFuZ2Uod2luZG93LCBsZW4obm9kZV9mZWF0dXJlc19zZXF1ZW5jZSkpOgogICAgICAgICAgICBmZWF0dXJlcyA9IFtdCgogICAgICAgICAgICAjIEN1cnJlbnQgcHJvdG9jb2wgZmVhdHVyZXMgKGFnZ3JlZ2F0ZWQpCiAgICAgICAgICAgIGN1cnJlbnQgPSBub2RlX2ZlYXR1cmVzX3NlcXVlbmNlW3RdCiAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjdXJyZW50Lm1lYW4oYXhpcz0wKSkgICMgbWVhbiBhY3Jvc3MgcHJvdG9jb2xzCiAgICAgICAgICAgIGZlYXR1cmVzLmV4dGVuZChjdXJyZW50LnN0ZChheGlzPTApKSAgICMgZGlzcGVyc2lvbgoKICAgICAgICAgICAgIyBDZW50cmFsaXR5IG1ldHJpY3MKICAgICAgICAgICAgZm9yIG1ldHJpY19uYW1lLCB2YWx1ZXMgaW4gc2VsZi5jZW50cmFsaXR5Lml0ZW1zKCk6CiAgICAgICAgICAgICAgICBmZWF0dXJlcy5hcHBlbmQodmFsdWVzLm1lYW4oKSkKICAgICAgICAgICAgICAgIGZlYXR1cmVzLmFwcGVuZCh2YWx1ZXMuc3RkKCkpCiAgICAgICAgICAgICAgICBmZWF0dXJlcy5hcHBlbmQodmFsdWVzLm1heCgpKQoKICAgICAgICAgICAgIyBDZW50cmFsaXR5LXdlaWdodGVkIHByb3RvY29sIHJpc2sKICAgICAgICAgICAgY3VycmVudF9yaXNrID0gY3VycmVudC5tZWFuKGF4aXM9MSkgICMgcGVyLXByb3RvY29sIHJpc2sgc2NvcmUKICAgICAgICAgICAgZm9yIG1ldHJpY19uYW1lLCB2YWx1ZXMgaW4gc2VsZi5jZW50cmFsaXR5Lml0ZW1zKCk6CiAgICAgICAgICAgICAgICAjIFdlaWdodGVkIHJpc2s6IGNlbnRyYWxpdHkgKiBjdXJyZW50IHJpc2sKICAgICAgICAgICAgICAgIGZlYXR1cmVzLmFwcGVuZCgodmFsdWVzICogY3VycmVudF9yaXNrKS5zdW0oKSkKCiAgICAgICAgICAgICMgVGVtcG9yYWwgZmVhdHVyZXMKICAgICAgICAgICAgaWYgdCA+PSB3aW5kb3c6CiAgICAgICAgICAgICAgICBwYXN0ID0gbnAuYXJyYXkobm9kZV9mZWF0dXJlc19zZXF1ZW5jZVt0IC0gd2luZG93OnRdKQogICAgICAgICAgICAgICAgZmVhdHVyZXMuZXh0ZW5kKHBhc3QubWVhbihheGlzPSgwLCAxKSkpCiAgICAgICAgICAgICAgICBjaGFuZ2VzID0gbnAuZGlmZihwYXN0LCBheGlzPTApCiAgICAgICAgICAgICAgICBpZiBjaGFuZ2VzLnNoYXBlWzBdID4gMDoKICAgICAgICAgICAgICAgICAgICBmZWF0dXJlcy5leHRlbmQoY2hhbmdlcy5tZWFuKGF4aXM9KDAsIDEpKSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgZmVhdHVyZXMuZXh0ZW5kKG5wLnplcm9zKGN1cnJlbnQuc2hhcGVbMV0pKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgZmVhdHVyZXMuZXh0ZW5kKG5wLnplcm9zKGN1cnJlbnQuc2hhcGVbMV0gKiAyKSkKCiAgICAgICAgICAgIGFsbF9mZWF0dXJlcy5hcHBlbmQoZmVhdHVyZXMpCgogICAgICAgIHJldHVybiBucC5hcnJheShhbGxfZmVhdHVyZXMsIGR0eXBlPW5wLmZsb2F0MzIpCgogICAgZGVmIGZpdCgKICAgICAgICBzZWxmLAogICAgICAgIFg6IG5wLm5kYXJyYXksCiAgICAgICAgeV9kaWN0OiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICApOgogICAgICAgICIiIlRyYWluIGxvZ2lzdGljIHJlZ3Jlc3Npb24gZm9yIGVhY2ggaG9yaXpvbi4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgWDogRmVhdHVyZSBtYXRyaXguCiAgICAgICAgICAgIHlfZGljdDogTGFiZWxzIHBlciBob3Jpem9uLgogICAgICAgICIiIgogICAgICAgIFggPSBucC5uYW5fdG9fbnVtKFgsIG5hbj0wLjAsIHBvc2luZj0xZTYsIG5lZ2luZj0tMWU2KQogICAgICAgIFhfc2NhbGVkID0gc2VsZi5zY2FsZXIuZml0X3RyYW5zZm9ybShYKQoKICAgICAgICBmb3IgaG9yaXpvbl9rZXksIHkgaW4geV9kaWN0Lml0ZW1zKCk6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiVHJhaW5pbmcgY2VudHJhbGl0eSBtb2RlbCBmb3Ige2hvcml6b25fa2V5fSIpCiAgICAgICAgICAgIG1vZGVsID0gTG9naXN0aWNSZWdyZXNzaW9uKAogICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PSJiYWxhbmNlZCIsCiAgICAgICAgICAgICAgICBtYXhfaXRlcj0xMDAwLAogICAgICAgICAgICAgICAgQz0xLjAsCiAgICAgICAgICAgICAgICBzb2x2ZXI9ImxiZmdzIiwKICAgICAgICAgICAgICAgIHJhbmRvbV9zdGF0ZT00MiwKICAgICAgICAgICAgKQogICAgICAgICAgICBtb2RlbC5maXQoWF9zY2FsZWQsIHkpCiAgICAgICAgICAgIHNlbGYubW9kZWxzW2hvcml6b25fa2V5XSA9IG1vZGVsCgogICAgZGVmIHByZWRpY3Qoc2VsZiwgWDogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgICAgICIiIlByZWRpY3QgY2FzY2FkZSBwcm9iYWJpbGl0aWVzLiIiIgogICAgICAgIFggPSBucC5uYW5fdG9fbnVtKFgsIG5hbj0wLjAsIHBvc2luZj0xZTYsIG5lZ2luZj0tMWU2KQogICAgICAgIFhfc2NhbGVkID0gc2VsZi5zY2FsZXIudHJhbnNmb3JtKFgpCiAgICAgICAgcHJlZGljdGlvbnMgPSB7fQogICAgICAgIGZvciBob3Jpem9uX2tleSwgbW9kZWwgaW4gc2VsZi5tb2RlbHMuaXRlbXMoKToKICAgICAgICAgICAgcHJlZGljdGlvbnNbaG9yaXpvbl9rZXldID0gbW9kZWwucHJlZGljdF9wcm9iYShYX3NjYWxlZClbOiwgMV0KICAgICAgICByZXR1cm4gcHJlZGljdGlvbnMK", "data/collectors/cascade_labeler.py": "IiIiCkNhc2NhZGUgZXZlbnQgbGFiZWxlcjogY3JlYXRlcyBncm91bmQgdHJ1dGggbGFiZWxzIGZvciBsaXF1aWRhdGlvbiBjYXNjYWRlCnByZWRpY3Rpb24gZnJvbSBrbm93biBoaXN0b3JpY2FsIGV2ZW50cyBhbmQgVFZMLWJhc2VkIGFub21hbHkgZGV0ZWN0aW9uLgoiIiIKCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lZGVsdGEKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBudW1weSBhcyBucApmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCgoKY2xhc3MgQ2FzY2FkZUxhYmVsZXI6CiAgICAiIiJMYWJlbHMgdGltZSBwZXJpb2RzIGFzIGNhc2NhZGUvbm9uLWNhc2NhZGUgYmFzZWQgb24ga25vd24gZXZlbnRzCiAgICBhbmQgc3RhdGlzdGljYWwgYW5vbWFseSBkZXRlY3Rpb24gb24gVFZMIGRyYXdkb3ducy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2FzY2FkZV9ldmVudHM6IGxpc3RbZGljdF0pOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGNhc2NhZGVfZXZlbnRzOiBMaXN0IG9mIGRpY3RzIHdpdGgga2V5czoKICAgICAgICAgICAgICAgIG5hbWUsIHN0YXJ0LCBwZWFrLCBlbmQsIHNldmVyaXR5LCB0dmxfbG9zc19wY3QKICAgICAgICAiIiIKICAgICAgICBzZWxmLmV2ZW50cyA9IFtdCiAgICAgICAgZm9yIGV2ZW50IGluIGNhc2NhZGVfZXZlbnRzOgogICAgICAgICAgICBzZWxmLmV2ZW50cy5hcHBlbmQoewogICAgICAgICAgICAgICAgIm5hbWUiOiBldmVudFsibmFtZSJdLAogICAgICAgICAgICAgICAgInN0YXJ0IjogcGQuVGltZXN0YW1wKGV2ZW50WyJzdGFydCJdKSwKICAgICAgICAgICAgICAgICJwZWFrIjogcGQuVGltZXN0YW1wKGV2ZW50WyJwZWFrIl0pLAogICAgICAgICAgICAgICAgImVuZCI6IHBkLlRpbWVzdGFtcChldmVudFsiZW5kIl0pLAogICAgICAgICAgICAgICAgInNldmVyaXR5IjogZXZlbnRbInNldmVyaXR5Il0sCiAgICAgICAgICAgICAgICAidHZsX2xvc3NfcGN0IjogZXZlbnRbInR2bF9sb3NzX3BjdCJdLAogICAgICAgICAgICB9KQogICAgICAgIGxvZ2dlci5pbmZvKGYiQ2FzY2FkZUxhYmVsZXIgaW5pdGlhbGl6ZWQgd2l0aCB7bGVuKHNlbGYuZXZlbnRzKX0ga25vd24gZXZlbnRzIikKCiAgICBkZWYgbGFiZWxfa25vd25fZXZlbnRzKAogICAgICAgIHNlbGYsCiAgICAgICAgZGF0ZXM6IHBkLkRhdGV0aW1lSW5kZXgsCiAgICAgICAgcHJlZGljdGlvbl9ob3Jpem9uczogbGlzdFtpbnRdID0gWzI0LCA3MiwgMTY4LCA3MjBdLAogICAgKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiQ3JlYXRlIGJpbmFyeSBsYWJlbHMgZm9yIGVhY2ggZGF0ZSBhbmQgcHJlZGljdGlvbiBob3Jpem9uLgoKICAgICAgICBGb3IgZWFjaCBob3Jpem9uIGgsIGxhYmVsW3RdID0gMSBpZiBhIGNhc2NhZGUgb2NjdXJzIHdpdGhpbiB0aGUgbmV4dCBoIGhvdXJzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBkYXRlczogRGF0ZXRpbWVJbmRleCBvZiBhbGwgdGltZXN0YW1wcyB0byBsYWJlbC4KICAgICAgICAgICAgcHJlZGljdGlvbl9ob3Jpem9uczogTGlzdCBvZiBob3Jpem9ucyBpbiBob3VycyBbMSwgNiwgMjQsIDE2OF0uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIERhdGFGcmFtZSB3aXRoIGNvbHVtbnM6IGRhdGUsIGNhc2NhZGVfe2h9aCBmb3IgZWFjaCBob3Jpem9uLAogICAgICAgICAgICBjYXNjYWRlX3NldmVyaXR5LCBjYXNjYWRlX25hbWUuCiAgICAgICAgIiIiCiAgICAgICAgbGFiZWxzID0gcGQuRGF0YUZyYW1lKHsiZGF0ZSI6IGRhdGVzfSkKCiAgICAgICAgZm9yIGggaW4gcHJlZGljdGlvbl9ob3Jpem9uczoKICAgICAgICAgICAgY29sID0gZiJjYXNjYWRlX3tofWgiCiAgICAgICAgICAgIGxhYmVsc1tjb2xdID0gMAoKICAgICAgICBsYWJlbHNbImNhc2NhZGVfc2V2ZXJpdHkiXSA9ICJub25lIgogICAgICAgIGxhYmVsc1siY2FzY2FkZV9uYW1lIl0gPSAiIgogICAgICAgIGxhYmVsc1siY2FzY2FkZV9hY3RpdmUiXSA9IDAKICAgICAgICBsYWJlbHNbInR2bF9sb3NzX3BjdCJdID0gMC4wCgogICAgICAgIGZvciBldmVudCBpbiBzZWxmLmV2ZW50czoKICAgICAgICAgICAgIyBNYXJrIGNhc2NhZGUtYWN0aXZlIHBlcmlvZAogICAgICAgICAgICBhY3RpdmVfbWFzayA9IChsYWJlbHNbImRhdGUiXSA+PSBldmVudFsic3RhcnQiXSkgJiAoCiAgICAgICAgICAgICAgICBsYWJlbHNbImRhdGUiXSA8PSBldmVudFsiZW5kIl0KICAgICAgICAgICAgKQogICAgICAgICAgICBsYWJlbHMubG9jW2FjdGl2ZV9tYXNrLCAiY2FzY2FkZV9hY3RpdmUiXSA9IDEKICAgICAgICAgICAgbGFiZWxzLmxvY1thY3RpdmVfbWFzaywgImNhc2NhZGVfc2V2ZXJpdHkiXSA9IGV2ZW50WyJzZXZlcml0eSJdCiAgICAgICAgICAgIGxhYmVscy5sb2NbYWN0aXZlX21hc2ssICJjYXNjYWRlX25hbWUiXSA9IGV2ZW50WyJuYW1lIl0KICAgICAgICAgICAgbGFiZWxzLmxvY1thY3RpdmVfbWFzaywgInR2bF9sb3NzX3BjdCJdID0gZXZlbnRbInR2bF9sb3NzX3BjdCJdCgogICAgICAgICAgICAjIEZvciBwcmVkaWN0aW9uIGxhYmVsczogbWFyayB0aGUgUFJFLWNhc2NhZGUgd2luZG93CiAgICAgICAgICAgIGZvciBoIGluIHByZWRpY3Rpb25faG9yaXpvbnM6CiAgICAgICAgICAgICAgICBob3Jpem9uX3RkID0gdGltZWRlbHRhKGhvdXJzPWgpCiAgICAgICAgICAgICAgICBwcmVfbWFzayA9IChsYWJlbHNbImRhdGUiXSA+PSBldmVudFsic3RhcnQiXSAtIGhvcml6b25fdGQpICYgKAogICAgICAgICAgICAgICAgICAgIGxhYmVsc1siZGF0ZSJdIDwgZXZlbnRbInN0YXJ0Il0KICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGxhYmVscy5sb2NbcHJlX21hc2ssIGYiY2FzY2FkZV97aH1oIl0gPSAxCgogICAgICAgICAgICAjIEFsc28gbWFyayB0aGUgYWN0aXZlIHBlcmlvZCBpdHNlbGYKICAgICAgICAgICAgZm9yIGggaW4gcHJlZGljdGlvbl9ob3Jpem9uczoKICAgICAgICAgICAgICAgIGxhYmVscy5sb2NbYWN0aXZlX21hc2ssIGYiY2FzY2FkZV97aH1oIl0gPSAxCgogICAgICAgIHJldHVybiBsYWJlbHMKCiAgICBkZWYgZGV0ZWN0X3R2bF9hbm9tYWxpZXMoCiAgICAgICAgc2VsZiwKICAgICAgICB0dmxfZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICB6X3RocmVzaG9sZDogZmxvYXQgPSAtMi41LAogICAgICAgIGRyYXdkb3duX3RocmVzaG9sZDogZmxvYXQgPSAtMC4xMCwKICAgICAgICBtaW5fZHVyYXRpb25fZGF5czogaW50ID0gMiwKICAgICkgLT4gbGlzdFtkaWN0XToKICAgICAgICAiIiJEZXRlY3QgcG90ZW50aWFsIGNhc2NhZGUgZXZlbnRzIGZyb20gVFZMIGRyYXdkb3duIGFub21hbGllcy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgdHZsX2RmOiBEYXRhRnJhbWUgd2l0aCBjb2x1bW5zIFtkYXRlLCB0dmxfdXNkXS4KICAgICAgICAgICAgel90aHJlc2hvbGQ6IFotc2NvcmUgdGhyZXNob2xkIGZvciBUVkwgY2hhbmdlIChuZWdhdGl2ZSkuCiAgICAgICAgICAgIGRyYXdkb3duX3RocmVzaG9sZDogTWluaW11bSBkcmF3ZG93biB0byBxdWFsaWZ5LgogICAgICAgICAgICBtaW5fZHVyYXRpb25fZGF5czogTWluaW11bSBudW1iZXIgb2YgY29uc2VjdXRpdmUgYW5vbWFseSBkYXlzLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBMaXN0IG9mIGRldGVjdGVkIGFub21hbHkgZXZlbnRzLgogICAgICAgICIiIgogICAgICAgIGRmID0gdHZsX2RmLmNvcHkoKS5zb3J0X3ZhbHVlcygiZGF0ZSIpCiAgICAgICAgZGZbInR2bF9yZXR1cm4iXSA9IGRmWyJ0dmxfdXNkIl0ucGN0X2NoYW5nZSgpCiAgICAgICAgZGZbInR2bF9yZXR1cm5fN2QiXSA9IGRmWyJ0dmxfdXNkIl0ucGN0X2NoYW5nZSg3KQogICAgICAgIGRmWyJyb2xsaW5nX2hpZ2giXSA9IGRmWyJ0dmxfdXNkIl0ucm9sbGluZygzMCkubWF4KCkKICAgICAgICBkZlsiZHJhd2Rvd24iXSA9IGRmWyJ0dmxfdXNkIl0gLyBkZlsicm9sbGluZ19oaWdoIl0gLSAxCgogICAgICAgICMgWi1zY29yZSBvZiBkYWlseSByZXR1cm5zCiAgICAgICAgbWVhbl9yZXQgPSBkZlsidHZsX3JldHVybiJdLnJvbGxpbmcoOTApLm1lYW4oKQogICAgICAgIHN0ZF9yZXQgPSBkZlsidHZsX3JldHVybiJdLnJvbGxpbmcoOTApLnN0ZCgpCiAgICAgICAgZGZbInpfc2NvcmUiXSA9IChkZlsidHZsX3JldHVybiJdIC0gbWVhbl9yZXQpIC8gc3RkX3JldAoKICAgICAgICAjIElkZW50aWZ5IGFub21hbG91cyBkYXlzCiAgICAgICAgZGZbImlzX2Fub21hbHkiXSA9ICgKICAgICAgICAgICAgKGRmWyJ6X3Njb3JlIl0gPCB6X3RocmVzaG9sZCkKICAgICAgICAgICAgfCAoZGZbImRyYXdkb3duIl0gPCBkcmF3ZG93bl90aHJlc2hvbGQpCiAgICAgICAgKS5hc3R5cGUoaW50KQoKICAgICAgICAjIEdyb3VwIGNvbnNlY3V0aXZlIGFub21hbG91cyBkYXlzIGludG8gZXZlbnRzCiAgICAgICAgZGZbImFub21hbHlfZ3JvdXAiXSA9ICgKICAgICAgICAgICAgZGZbImlzX2Fub21hbHkiXS5kaWZmKCkubmUoMCkuY3Vtc3VtKCkgKiBkZlsiaXNfYW5vbWFseSJdCiAgICAgICAgKQoKICAgICAgICBldmVudHMgPSBbXQogICAgICAgIGZvciBncm91cF9pZCBpbiBkZltkZlsiYW5vbWFseV9ncm91cCJdID4gMF1bImFub21hbHlfZ3JvdXAiXS51bmlxdWUoKToKICAgICAgICAgICAgZ3JvdXAgPSBkZltkZlsiYW5vbWFseV9ncm91cCJdID09IGdyb3VwX2lkXQogICAgICAgICAgICBkdXJhdGlvbiA9IChncm91cFsiZGF0ZSJdLm1heCgpIC0gZ3JvdXBbImRhdGUiXS5taW4oKSkuZGF5cyArIDEKICAgICAgICAgICAgaWYgZHVyYXRpb24gPj0gbWluX2R1cmF0aW9uX2RheXM6CiAgICAgICAgICAgICAgICBldmVudHMuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAibmFtZSI6IGYiZGV0ZWN0ZWRfYW5vbWFseV97Z3JvdXBfaWR9IiwKICAgICAgICAgICAgICAgICAgICAic3RhcnQiOiBncm91cFsiZGF0ZSJdLm1pbigpLAogICAgICAgICAgICAgICAgICAgICJwZWFrIjogZ3JvdXAubG9jW2dyb3VwWyJkcmF3ZG93biJdLmlkeG1pbigpLCAiZGF0ZSJdLAogICAgICAgICAgICAgICAgICAgICJlbmQiOiBncm91cFsiZGF0ZSJdLm1heCgpLAogICAgICAgICAgICAgICAgICAgICJzZXZlcml0eSI6IHNlbGYuX2NsYXNzaWZ5X3NldmVyaXR5KAogICAgICAgICAgICAgICAgICAgICAgICBncm91cFsiZHJhd2Rvd24iXS5taW4oKQogICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgICAgInR2bF9sb3NzX3BjdCI6IGFicyhncm91cFsiZHJhd2Rvd24iXS5taW4oKSksCiAgICAgICAgICAgICAgICAgICAgImR1cmF0aW9uX2RheXMiOiBkdXJhdGlvbiwKICAgICAgICAgICAgICAgICAgICAibWF4X3pfc2NvcmUiOiBncm91cFsiel9zY29yZSJdLm1pbigpLAogICAgICAgICAgICAgICAgfSkKCiAgICAgICAgbG9nZ2VyLmluZm8oZiJEZXRlY3RlZCB7bGVuKGV2ZW50cyl9IHBvdGVudGlhbCBjYXNjYWRlIGV2ZW50cyBmcm9tIFRWTCBkYXRhIikKICAgICAgICByZXR1cm4gZXZlbnRzCgogICAgZGVmIF9jbGFzc2lmeV9zZXZlcml0eShzZWxmLCBtYXhfZHJhd2Rvd246IGZsb2F0KSAtPiBzdHI6CiAgICAgICAgIiIiQ2xhc3NpZnkgZXZlbnQgc2V2ZXJpdHkgYmFzZWQgb24gbWF4aW11bSBkcmF3ZG93bi4iIiIKICAgICAgICBpZiBtYXhfZHJhd2Rvd24gPCAtMC4zMDoKICAgICAgICAgICAgcmV0dXJuICJjYXRhc3Ryb3BoaWMiCiAgICAgICAgZWxpZiBtYXhfZHJhd2Rvd24gPCAtMC4xNToKICAgICAgICAgICAgcmV0dXJuICJzZXZlcmUiCiAgICAgICAgZWxpZiBtYXhfZHJhd2Rvd24gPCAtMC4wODoKICAgICAgICAgICAgcmV0dXJuICJtb2RlcmF0ZSIKICAgICAgICBlbHNlOgogICAgICAgICAgICByZXR1cm4gIm1pbm9yIgoKICAgIGRlZiBjcmVhdGVfbXVsdGlfaG9yaXpvbl9sYWJlbHMoCiAgICAgICAgc2VsZiwKICAgICAgICB0dmxfZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICBwcmVkaWN0aW9uX2hvcml6b25zOiBsaXN0W2ludF0gPSBbMjQsIDcyLCAxNjgsIDcyMF0sCiAgICAgICAgY29tYmluZV9rbm93bl9hbmRfZGV0ZWN0ZWQ6IGJvb2wgPSBUcnVlLAogICAgICAgIHpfdGhyZXNob2xkOiBmbG9hdCA9IC0yLjUsCiAgICAgICAgZHJhd2Rvd25fdGhyZXNob2xkOiBmbG9hdCA9IC0wLjEwLAogICAgICAgIG1pbl9kdXJhdGlvbl9kYXlzOiBpbnQgPSAyLAogICAgKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiQ3JlYXRlIGZpbmFsIG11bHRpLWhvcml6b24gbGFiZWxzIGNvbWJpbmluZyBrbm93biBldmVudHMgYW5kCiAgICAgICAgZGV0ZWN0ZWQgYW5vbWFsaWVzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0dmxfZGY6IERhdGFGcmFtZSB3aXRoIFtkYXRlLCB0dmxfdXNkXSBmb3IgYW5vbWFseSBkZXRlY3Rpb24uCiAgICAgICAgICAgIHByZWRpY3Rpb25faG9yaXpvbnM6IEZvcmVjYXN0IGhvcml6b25zIGluIGhvdXJzLgogICAgICAgICAgICBjb21iaW5lX2tub3duX2FuZF9kZXRlY3RlZDogV2hldGhlciB0byBhbHNvIGRldGVjdCBmcm9tIFRWTC4KICAgICAgICAgICAgel90aHJlc2hvbGQ6IFotc2NvcmUgdGhyZXNob2xkIGZvciBhbm9tYWx5IGRldGVjdGlvbi4KICAgICAgICAgICAgZHJhd2Rvd25fdGhyZXNob2xkOiBEcmF3ZG93biB0aHJlc2hvbGQgZm9yIGFub21hbHkgZGV0ZWN0aW9uLgogICAgICAgICAgICBtaW5fZHVyYXRpb25fZGF5czogTWluaW11bSBjb25zZWN1dGl2ZSBhbm9tYWx5IGRheXMuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIENvbXBsZXRlIGxhYmVsIERhdGFGcmFtZS4KICAgICAgICAiIiIKICAgICAgICBkYXRlcyA9IHBkLkRhdGV0aW1lSW5kZXgodHZsX2RmWyJkYXRlIl0udW5pcXVlKCkpLnNvcnRfdmFsdWVzKCkKICAgICAgICBsYWJlbHMgPSBzZWxmLmxhYmVsX2tub3duX2V2ZW50cyhkYXRlcywgcHJlZGljdGlvbl9ob3Jpem9ucykKCiAgICAgICAgaWYgY29tYmluZV9rbm93bl9hbmRfZGV0ZWN0ZWQ6CiAgICAgICAgICAgIGRldGVjdGVkID0gc2VsZi5kZXRlY3RfdHZsX2Fub21hbGllcygKICAgICAgICAgICAgICAgIHR2bF9kZiwKICAgICAgICAgICAgICAgIHpfdGhyZXNob2xkPXpfdGhyZXNob2xkLAogICAgICAgICAgICAgICAgZHJhd2Rvd25fdGhyZXNob2xkPWRyYXdkb3duX3RocmVzaG9sZCwKICAgICAgICAgICAgICAgIG1pbl9kdXJhdGlvbl9kYXlzPW1pbl9kdXJhdGlvbl9kYXlzLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciBldmVudCBpbiBkZXRlY3RlZDoKICAgICAgICAgICAgICAgICMgT25seSBhZGQgaWYgbm90IGFscmVhZHkgY292ZXJlZCBieSBhIGtub3duIGV2ZW50CiAgICAgICAgICAgICAgICBvdmVybGFwID0gRmFsc2UKICAgICAgICAgICAgICAgIGZvciBrbm93biBpbiBzZWxmLmV2ZW50czoKICAgICAgICAgICAgICAgICAgICBpZiAoCiAgICAgICAgICAgICAgICAgICAgICAgIGV2ZW50WyJzdGFydCJdIDw9IGtub3duWyJlbmQiXQogICAgICAgICAgICAgICAgICAgICAgICBhbmQgZXZlbnRbImVuZCJdID49IGtub3duWyJzdGFydCJdCiAgICAgICAgICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgICAgICAgICAgb3ZlcmxhcCA9IFRydWUKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIG5vdCBvdmVybGFwOgogICAgICAgICAgICAgICAgICAgIHNlbGYuZXZlbnRzLmFwcGVuZChldmVudCkKICAgICAgICAgICAgICAgICAgICBsb2dnZXIuaW5mbyhmIkFkZGVkIGRldGVjdGVkIGV2ZW50OiB7ZXZlbnRbJ25hbWUnXX0iKQoKICAgICAgICAgICAgIyBSZS1sYWJlbCB3aXRoIG5ldyBldmVudHMKICAgICAgICAgICAgbGFiZWxzID0gc2VsZi5sYWJlbF9rbm93bl9ldmVudHMoZGF0ZXMsIHByZWRpY3Rpb25faG9yaXpvbnMpCgogICAgICAgICMgQWRkIGNvbnRpbnVvdXMgcmlzayBzY29yZSAoaGlnaGVyIG5lYXIgY2FzY2FkZSBldmVudHMpCiAgICAgICAgbGFiZWxzWyJyaXNrX3Njb3JlIl0gPSAwLjAKICAgICAgICBmb3IgZXZlbnQgaW4gc2VsZi5ldmVudHM6CiAgICAgICAgICAgIHNldmVyaXR5X21hcCA9IHsKICAgICAgICAgICAgICAgICJjYXRhc3Ryb3BoaWMiOiAxLjAsICJzZXZlcmUiOiAwLjc1LAogICAgICAgICAgICAgICAgIm1vZGVyYXRlIjogMC41LCAibWlub3IiOiAwLjI1LAogICAgICAgICAgICB9CiAgICAgICAgICAgIGJhc2Vfc2NvcmUgPSBzZXZlcml0eV9tYXAuZ2V0KGV2ZW50WyJzZXZlcml0eSJdLCAwLjI1KQogICAgICAgICAgICBmb3IgaWR4LCByb3cgaW4gbGFiZWxzLml0ZXJyb3dzKCk6CiAgICAgICAgICAgICAgICBkaXN0ID0gYWJzKChyb3dbImRhdGUiXSAtIGV2ZW50WyJzdGFydCJdKS5kYXlzKQogICAgICAgICAgICAgICAgaWYgZGlzdCA8PSAzMDoKICAgICAgICAgICAgICAgICAgICBjb250cmlidXRpb24gPSBiYXNlX3Njb3JlICogbnAuZXhwKC0wLjEgKiBkaXN0KQogICAgICAgICAgICAgICAgICAgIGxhYmVscy5sb2NbaWR4LCAicmlza19zY29yZSJdID0gbWF4KAogICAgICAgICAgICAgICAgICAgICAgICBsYWJlbHMubG9jW2lkeCwgInJpc2tfc2NvcmUiXSwgY29udHJpYnV0aW9uCiAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICByZXR1cm4gbGFiZWxzCgogICAgZGVmIGdldF9ldmVudF93aW5kb3dzKAogICAgICAgIHNlbGYsIHByZV9kYXlzOiBpbnQgPSAxNCwgcG9zdF9kYXlzOiBpbnQgPSA3CiAgICApIC0+IGxpc3RbZGljdF06CiAgICAgICAgIiIiR2V0IGV4cGFuZGVkIGV2ZW50IHdpbmRvd3MgZm9yIGNhc2Ugc3R1ZHkgYW5hbHlzaXMuIiIiCiAgICAgICAgd2luZG93cyA9IFtdCiAgICAgICAgZm9yIGV2ZW50IGluIHNlbGYuZXZlbnRzOgogICAgICAgICAgICB3aW5kb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAqKmV2ZW50LAogICAgICAgICAgICAgICAgIndpbmRvd19zdGFydCI6IGV2ZW50WyJzdGFydCJdIC0gdGltZWRlbHRhKGRheXM9cHJlX2RheXMpLAogICAgICAgICAgICAgICAgIndpbmRvd19lbmQiOiBldmVudFsiZW5kIl0gKyB0aW1lZGVsdGEoZGF5cz1wb3N0X2RheXMpLAogICAgICAgICAgICB9KQogICAgICAgIHJldHVybiB3aW5kb3dzCg==", "data/collectors/coingecko_collector.py": "IiIiCkNvaW5HZWNrbyBBUEkgY29sbGVjdG9yIGZvciB0b2tlbiBwcmljZXMsIG1hcmtldCBkYXRhLCBhbmQgY29ycmVsYXRpb25zLgpGcmVlIGRlbW8gdGllcjogMTAtMzAgY2FsbHMvbWluLCBubyBBUEkga2V5IHJlcXVpcmVkIGZvciBiYXNpYyBlbmRwb2ludHMuCiIiIgoKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHJlcXVlc3RzCmZyb20gbG9ndXJ1IGltcG9ydCBsb2dnZXIKCgpCQVNFX1VSTCA9ICJodHRwczovL2FwaS5jb2luZ2Vja28uY29tL2FwaS92MyIKCgpjbGFzcyBDb2luR2Vja29Db2xsZWN0b3I6CiAgICAiIiJDb2xsZWN0cyB0b2tlbiBwcmljZSBhbmQgbWFya2V0IGRhdGEgZnJvbSBDb2luR2Vja28ncyBmcmVlIEFQSS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcmF3X2Rpcjogc3RyID0gImRhdGEvcmF3IiwgcmF0ZV9saW1pdDogZmxvYXQgPSAyLjUpOgogICAgICAgIHNlbGYucmF3X2RpciA9IFBhdGgocmF3X2RpcikKICAgICAgICBzZWxmLnJhd19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHNlbGYucmF0ZV9saW1pdCA9IHJhdGVfbGltaXQgICMgc2Vjb25kcyBiZXR3ZWVuIHJlcXVlc3RzCiAgICAgICAgc2VsZi5zZXNzaW9uID0gcmVxdWVzdHMuU2Vzc2lvbigpCgogICAgZGVmIF9nZXQoc2VsZiwgZW5kcG9pbnQ6IHN0ciwgcGFyYW1zOiBPcHRpb25hbFtkaWN0XSA9IE5vbmUpIC0+IGRpY3Q6CiAgICAgICAgIiIiUmF0ZS1saW1pdGVkIEdFVCByZXF1ZXN0LiIiIgogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKHNlbGYucmF0ZV9saW1pdCkKICAgICAgICAgICAgICAgIHJlc3AgPSBzZWxmLnNlc3Npb24uZ2V0KAogICAgICAgICAgICAgICAgICAgIGYie0JBU0VfVVJMfS97ZW5kcG9pbnR9IiwgcGFyYW1zPXBhcmFtcywgdGltZW91dD0zMAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgaWYgcmVzcC5zdGF0dXNfY29kZSA9PSA0Mjk6CiAgICAgICAgICAgICAgICAgICAgd2FpdCA9IGludChyZXNwLmhlYWRlcnMuZ2V0KCJSZXRyeS1BZnRlciIsIDYwKSkKICAgICAgICAgICAgICAgICAgICBsb2dnZXIud2FybmluZyhmIlJhdGUgbGltaXRlZC4gV2FpdGluZyB7d2FpdH1zLi4uIikKICAgICAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKHdhaXQpCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJlc3AucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgICAgICAgICByZXR1cm4gcmVzcC5qc29uKCkKICAgICAgICAgICAgZXhjZXB0IHJlcXVlc3RzLmV4Y2VwdGlvbnMuUmVxdWVzdEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJSZXF1ZXN0IGZhaWxlZCAoYXR0ZW1wdCB7YXR0ZW1wdCArIDF9KToge2V9IikKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMiAqKiBhdHRlbXB0KQogICAgICAgIHJhaXNlIENvbm5lY3Rpb25FcnJvcihmIkZhaWxlZCB0byBmZXRjaCB7ZW5kcG9pbnR9IGFmdGVyIDMgYXR0ZW1wdHMiKQoKICAgICMgTWFwcGluZyBvZiBEZUZpIHByb3RvY29sIG5hbWVzIHRvIHRoZWlyIENvaW5HZWNrbyB0b2tlbiBJRHMKICAgIFBST1RPQ09MX1RPS0VOX01BUCA9IHsKICAgICAgICAiYWF2ZS12MyI6ICJhYXZlIiwKICAgICAgICAiYWF2ZS12MiI6ICJhYXZlIiwKICAgICAgICAiY29tcG91bmQtdjMiOiAiY29tcG91bmQtZ292ZXJuYW5jZS10b2tlbiIsCiAgICAgICAgImNvbXBvdW5kLXYyIjogImNvbXBvdW5kLWdvdmVybmFuY2UtdG9rZW4iLAogICAgICAgICJtYWtlcmRhbyI6ICJtYWtlciIsCiAgICAgICAgInVuaXN3YXAtdjMiOiAidW5pc3dhcCIsCiAgICAgICAgInVuaXN3YXAtdjIiOiAidW5pc3dhcCIsCiAgICAgICAgImN1cnZlLWRleCI6ICJjdXJ2ZS1kYW8tdG9rZW4iLAogICAgICAgICJsaWRvIjogImxpZG8tZGFvIiwKICAgICAgICAicm9ja2V0LXBvb2wiOiAicm9ja2V0LXBvb2wiLAogICAgICAgICJjb252ZXgtZmluYW5jZSI6ICJjb252ZXgtZmluYW5jZSIsCiAgICAgICAgInllYXJuLWZpbmFuY2UiOiAieWVhcm4tZmluYW5jZSIsCiAgICAgICAgImZyYXgiOiAiZnJheC1zaGFyZSIsCiAgICAgICAgImluc3RhZGFwcCI6ICJpbnN0YWRhcHAiLAogICAgICAgICJtb3JwaG8iOiAibW9ycGhvIiwKICAgIH0KCiAgICAjIEtleSB0b2tlbnMgdG8gdHJhY2sgZm9yIHRoZSBEZUZpIGVjb3N5c3RlbQogICAgS0VZX1RPS0VOUyA9IFsKICAgICAgICAiZXRoZXJldW0iLCAiYml0Y29pbiIsICJ0ZXRoZXIiLCAidXNkLWNvaW4iLCAiZGFpIiwKICAgICAgICAid3JhcHBlZC1iaXRjb2luIiwgInN0YWtlZC1ldGhlciIsICJmcmF4IiwgInJvY2tldC1wb29sLWV0aCIsCiAgICAgICAgImNoYWlubGluayIsICJhYXZlIiwgImNvbXBvdW5kLWdvdmVybmFuY2UtdG9rZW4iLCAibWFrZXIiLAogICAgICAgICJ1bmlzd2FwIiwgImN1cnZlLWRhby10b2tlbiIsICJsaWRvLWRhbyIsICJjb252ZXgtZmluYW5jZSIsCiAgICBdCgogICAgZGVmIGNvbGxlY3RfdG9rZW5fcHJpY2VfaGlzdG9yeSgKICAgICAgICBzZWxmLAogICAgICAgIHRva2VuX2lkOiBzdHIsCiAgICAgICAgdnNfY3VycmVuY3k6IHN0ciA9ICJ1c2QiLAogICAgICAgIGRheXM6IGludCA9IDM2NSwKICAgICkgLT4gcGQuRGF0YUZyYW1lOgogICAgICAgICIiIkNvbGxlY3QgaGlzdG9yaWNhbCBPSExDIHByaWNlIGRhdGEgZm9yIGEgdG9rZW4uIiIiCiAgICAgICAgbG9nZ2VyLmluZm8oZiJDb2xsZWN0aW5nIHByaWNlIGhpc3RvcnkgZm9yIHt0b2tlbl9pZH0gKHtkYXlzfSBkYXlzKSIpCiAgICAgICAgZGF0YSA9IHNlbGYuX2dldCgKICAgICAgICAgICAgZiJjb2lucy97dG9rZW5faWR9L21hcmtldF9jaGFydCIsCiAgICAgICAgICAgIHBhcmFtcz17InZzX2N1cnJlbmN5IjogdnNfY3VycmVuY3ksICJkYXlzIjogZGF5cywgImludGVydmFsIjogImRhaWx5In0sCiAgICAgICAgKQoKICAgICAgICBwcmljZXMgPSBkYXRhLmdldCgicHJpY2VzIiwgW10pCiAgICAgICAgdm9sdW1lcyA9IGRhdGEuZ2V0KCJ0b3RhbF92b2x1bWVzIiwgW10pCiAgICAgICAgbWFya2V0X2NhcHMgPSBkYXRhLmdldCgibWFya2V0X2NhcHMiLCBbXSkKCiAgICAgICAgcmVjb3JkcyA9IFtdCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKHByaWNlcykpOgogICAgICAgICAgICByZWNvcmQgPSB7CiAgICAgICAgICAgICAgICAidG9rZW4iOiB0b2tlbl9pZCwKICAgICAgICAgICAgICAgICJkYXRlIjogZGF0ZXRpbWUuZnJvbXRpbWVzdGFtcChwcmljZXNbaV1bMF0gLyAxMDAwKSwKICAgICAgICAgICAgICAgICJwcmljZV91c2QiOiBwcmljZXNbaV1bMV0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgaWYgaSA8IGxlbih2b2x1bWVzKToKICAgICAgICAgICAgICAgIHJlY29yZFsidm9sdW1lX3VzZCJdID0gdm9sdW1lc1tpXVsxXQogICAgICAgICAgICBpZiBpIDwgbGVuKG1hcmtldF9jYXBzKToKICAgICAgICAgICAgICAgIHJlY29yZFsibWFya2V0X2NhcF91c2QiXSA9IG1hcmtldF9jYXBzW2ldWzFdCiAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKCiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUocmVjb3JkcykKICAgICAgICBpZiBub3QgZGYuZW1wdHk6CiAgICAgICAgICAgIGRmWyJkYXRlIl0gPSBwZC50b19kYXRldGltZShkZlsiZGF0ZSJdKS5kdC5ub3JtYWxpemUoKQogICAgICAgICAgICBkZiA9IGRmLmRyb3BfZHVwbGljYXRlcyhzdWJzZXQ9WyJ0b2tlbiIsICJkYXRlIl0pCiAgICAgICAgcmV0dXJuIGRmCgogICAgZGVmIGNvbGxlY3RfYWxsX3Rva2VuX3ByaWNlcygKICAgICAgICBzZWxmLCBkYXlzOiBpbnQgPSAxNDYwLCAgIyB+NCB5ZWFycwogICAgKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiQ29sbGVjdCBwcmljZSBoaXN0b3J5IGZvciBhbGwga2V5IERlRmkgdG9rZW5zLiIiIgogICAgICAgIGFsbF9kZnMgPSBbXQogICAgICAgIGZvciB0b2tlbl9pZCBpbiBzZWxmLktFWV9UT0tFTlM6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRmID0gc2VsZi5jb2xsZWN0X3Rva2VuX3ByaWNlX2hpc3RvcnkodG9rZW5faWQsIGRheXM9ZGF5cykKICAgICAgICAgICAgICAgIGFsbF9kZnMuYXBwZW5kKGRmKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJGYWlsZWQgdG8gY29sbGVjdCBwcmljZXMgZm9yIHt0b2tlbl9pZH06IHtlfSIpCiAgICAgICAgcmVzdWx0ID0gcGQuY29uY2F0KGFsbF9kZnMsIGlnbm9yZV9pbmRleD1UcnVlKSBpZiBhbGxfZGZzIGVsc2UgcGQuRGF0YUZyYW1lKCkKICAgICAgICBpZiBub3QgcmVzdWx0LmVtcHR5OgogICAgICAgICAgICByZXN1bHQudG9fcGFycXVldChzZWxmLnJhd19kaXIgLyAidG9rZW5fcHJpY2VzLnBhcnF1ZXQiLCBpbmRleD1GYWxzZSkKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgZGVmIGNvbGxlY3RfZ2xvYmFsX21hcmtldF9kYXRhKHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgIiIiQ29sbGVjdCBnbG9iYWwgY3J5cHRvIG1hcmtldCBzdGF0aXN0aWNzLiIiIgogICAgICAgIGxvZ2dlci5pbmZvKCJDb2xsZWN0aW5nIGdsb2JhbCBtYXJrZXQgZGF0YSIpCiAgICAgICAgZGF0YSA9IHNlbGYuX2dldCgiZ2xvYmFsIikKICAgICAgICByZXR1cm4gZGF0YS5nZXQoImRhdGEiLCB7fSkKCiAgICBkZWYgY29tcHV0ZV9wcmljZV9jb3JyZWxhdGlvbl9tYXRyaXgoCiAgICAgICAgc2VsZiwKICAgICAgICBwcmljZV9kZjogcGQuRGF0YUZyYW1lLAogICAgICAgIHdpbmRvdzogaW50ID0gMzAsCiAgICApIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICAiIiJDb21wdXRlIHJvbGxpbmcgcGFpcndpc2UgcHJpY2UgY29ycmVsYXRpb24gbWF0cml4LiIiIgogICAgICAgIHBpdm90ID0gcHJpY2VfZGYucGl2b3RfdGFibGUoCiAgICAgICAgICAgIGluZGV4PSJkYXRlIiwgY29sdW1ucz0idG9rZW4iLCB2YWx1ZXM9InByaWNlX3VzZCIKICAgICAgICApCiAgICAgICAgIyBDb21wdXRlIGxvZyByZXR1cm5zCiAgICAgICAgcmV0dXJucyA9IG5wLmxvZyhwaXZvdCAvIHBpdm90LnNoaWZ0KDEpKS5kcm9wbmEoKQogICAgICAgICMgUm9sbGluZyBjb3JyZWxhdGlvbgogICAgICAgIGNvcnIgPSByZXR1cm5zLnJvbGxpbmcod2luZG93PXdpbmRvdykuY29ycigpCiAgICAgICAgcmV0dXJuIGNvcnIKCiAgICBkZWYgY29tcHV0ZV9yZXR1cm5fZmVhdHVyZXMoc2VsZiwgcHJpY2VfZGY6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgICAgICIiIkNvbXB1dGUgcmV0dXJuLWJhc2VkIGZlYXR1cmVzIGZvciBlYWNoIHRva2VuLiIiIgogICAgICAgIGZlYXR1cmVzID0gW10KICAgICAgICBmb3IgdG9rZW5faWQsIGdyb3VwIGluIHByaWNlX2RmLmdyb3VwYnkoInRva2VuIik6CiAgICAgICAgICAgIGdyb3VwID0gZ3JvdXAuc29ydF92YWx1ZXMoImRhdGUiKS5jb3B5KCkKICAgICAgICAgICAgZ3JvdXBbImxvZ19yZXR1cm4iXSA9IG5wLmxvZygKICAgICAgICAgICAgICAgIGdyb3VwWyJwcmljZV91c2QiXSAvIGdyb3VwWyJwcmljZV91c2QiXS5zaGlmdCgxKQogICAgICAgICAgICApCiAgICAgICAgICAgIGdyb3VwWyJ2b2xhdGlsaXR5XzdkIl0gPSBncm91cFsibG9nX3JldHVybiJdLnJvbGxpbmcoNykuc3RkKCkKICAgICAgICAgICAgZ3JvdXBbInZvbGF0aWxpdHlfMzBkIl0gPSBncm91cFsibG9nX3JldHVybiJdLnJvbGxpbmcoMzApLnN0ZCgpCiAgICAgICAgICAgIGdyb3VwWyJyZXR1cm5fN2QiXSA9IGdyb3VwWyJwcmljZV91c2QiXS5wY3RfY2hhbmdlKDcpCiAgICAgICAgICAgIGdyb3VwWyJyZXR1cm5fMzBkIl0gPSBncm91cFsicHJpY2VfdXNkIl0ucGN0X2NoYW5nZSgzMCkKICAgICAgICAgICAgZ3JvdXBbInZvbHVtZV9tYV83ZCJdID0gZ3JvdXBbInZvbHVtZV91c2QiXS5yb2xsaW5nKDcpLm1lYW4oKQogICAgICAgICAgICBncm91cFsidm9sdW1lX3JhdGlvIl0gPSAoCiAgICAgICAgICAgICAgICBncm91cFsidm9sdW1lX3VzZCJdIC8gZ3JvdXBbInZvbHVtZV9tYV83ZCJdCiAgICAgICAgICAgICkKICAgICAgICAgICAgZ3JvdXBbImRyYXdkb3duIl0gPSAoCiAgICAgICAgICAgICAgICBncm91cFsicHJpY2VfdXNkIl0gLyBncm91cFsicHJpY2VfdXNkIl0uY3VtbWF4KCkgLSAxCiAgICAgICAgICAgICkKICAgICAgICAgICAgZmVhdHVyZXMuYXBwZW5kKGdyb3VwKQoKICAgICAgICByZXN1bHQgPSBwZC5jb25jYXQoZmVhdHVyZXMsIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIHJldHVybiByZXN1bHQK", "data/processing/graph_constructor.py": "...", "data/processing/feature_engineer.py": "...", "training/losses.py": "IiIiCkN1c3RvbSBsb3NzIGZ1bmN0aW9ucyBmb3IgY2FzY2FkZSBwcmVkaWN0aW9uLgoKSW1wbGVtZW50czoKICAtIEZvY2FsTG9zczogYWRkcmVzc2VzIGV4dHJlbWUgY2xhc3MgaW1iYWxhbmNlIChjYXNjYWRlcyBhcmUgcmFyZSkKICAtIENhc2NhZGVMb3NzOiBtdWx0aS10YXNrIGxvc3MgY29tYmluaW5nIGNsYXNzaWZpY2F0aW9uICsgc2V2ZXJpdHkgKyBwcm9wYWdhdGlvbgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmNsYXNzIEZvY2FsTG9zcyhubi5Nb2R1bGUpOgogICAgIiIiRm9jYWwgTG9zcyBmb3IgaGFuZGxpbmcgZXh0cmVtZSBjbGFzcyBpbWJhbGFuY2UgaW4gY2FzY2FkZSBwcmVkaWN0aW9uLgoKICAgIEZMKHBfdCkgPSAtYWxwaGFfdCAqICgxIC0gcF90KV5nYW1tYSAqIGxvZyhwX3QpCgogICAgVXNlcyBPTkxZIGZvY2FsIG1vZHVsYXRpb24gKGdhbW1hKSBhbmQgYWxwaGEgYmFsYW5jaW5nIOKAlCBubyBhZGRpdGlvbmFsCiAgICBwb3Nfd2VpZ2h0IG11bHRpcGxpZXIsIHdoaWNoIHdvdWxkIGNyZWF0ZSB0cmlwbGUtcmVkdW5kYW50IHdlaWdodGluZwogICAgYW5kIHVuc3RhYmxlIGdyYWRpZW50cy4KCiAgICBSZWZlcmVuY2U6IExpbiBldCBhbC4sICJGb2NhbCBMb3NzIGZvciBEZW5zZSBPYmplY3QgRGV0ZWN0aW9uIiwgSUNDViAyMDE3LgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZ2FtbWE6IGZsb2F0ID0gMi4wLAogICAgICAgIGFscGhhOiBmbG9hdCA9IDAuNzUsCiAgICAgICAgcG9zX3dlaWdodDogZmxvYXQgPSAxLjAsICAjIGtlcHQgZm9yIEFQSSBjb21wYXQ7IG9ubHkgdXNlZCBpbiBCQ0UKICAgICAgICByZWR1Y3Rpb246IHN0ciA9ICJtZWFuIiwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5nYW1tYSA9IGdhbW1hCiAgICAgICAgc2VsZi5hbHBoYSA9IGFscGhhICAjIHdlaWdodCBmb3IgUE9TSVRJVkUgY2xhc3MgKDAuNzUgPSAzOjEgcG9zIGJpYXMpCiAgICAgICAgc2VsZi5yZWR1Y3Rpb24gPSByZWR1Y3Rpb24KCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLCBsb2dpdHM6IHRvcmNoLlRlbnNvciwgdGFyZ2V0czogdG9yY2guVGVuc29yCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiIKICAgICAgICBBcmdzOgogICAgICAgICAgICBsb2dpdHM6IFJhdyBtb2RlbCBvdXRwdXRzIChiZWZvcmUgc2lnbW9pZCkuCiAgICAgICAgICAgIHRhcmdldHM6IEJpbmFyeSBsYWJlbHMgKDAgb3IgMSkuCiAgICAgICAgIiIiCiAgICAgICAgcHJvYnMgPSB0b3JjaC5zaWdtb2lkKGxvZ2l0cykKICAgICAgICB0YXJnZXRzID0gdGFyZ2V0cy5mbG9hdCgpCgogICAgICAgICMgQkNFIGxvc3MgcGVyIGVsZW1lbnQKICAgICAgICBiY2UgPSBGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKAogICAgICAgICAgICBsb2dpdHMsIHRhcmdldHMsIHJlZHVjdGlvbj0ibm9uZSIKICAgICAgICApCgogICAgICAgICMgRm9jYWwgbW9kdWxhdGlvbjogZG93bi13ZWlnaHQgZWFzeSBleGFtcGxlcwogICAgICAgIHBfdCA9IHByb2JzICogdGFyZ2V0cyArICgxIC0gcHJvYnMpICogKDEgLSB0YXJnZXRzKQogICAgICAgIGZvY2FsX3dlaWdodCA9ICgxIC0gcF90KSAqKiBzZWxmLmdhbW1hCgogICAgICAgICMgQWxwaGEgYmFsYW5jaW5nOiBoaWdoZXIgYWxwaGEgPSBtb3JlIHdlaWdodCBvbiBwb3NpdGl2ZXMKICAgICAgICBhbHBoYV90ID0gc2VsZi5hbHBoYSAqIHRhcmdldHMgKyAoMSAtIHNlbGYuYWxwaGEpICogKDEgLSB0YXJnZXRzKQoKICAgICAgICBsb3NzID0gYWxwaGFfdCAqIGZvY2FsX3dlaWdodCAqIGJjZQoKICAgICAgICBpZiBzZWxmLnJlZHVjdGlvbiA9PSAibWVhbiI6CiAgICAgICAgICAgIHJldHVybiBsb3NzLm1lYW4oKQogICAgICAgIGVsaWYgc2VsZi5yZWR1Y3Rpb24gPT0gInN1bSI6CiAgICAgICAgICAgIHJldHVybiBsb3NzLnN1bSgpCiAgICAgICAgcmV0dXJuIGxvc3MKCgpjbGFzcyBDYXNjYWRlTG9zcyhubi5Nb2R1bGUpOgogICAgIiIiTXVsdGktdGFzayBsb3NzIGZvciBjYXNjYWRlIHByZWRpY3Rpb24gY29tYmluaW5nOgogICAgICAxLiBNdWx0aS1ob3Jpem9uIGNsYXNzaWZpY2F0aW9uIChmb2NhbCBsb3NzKQogICAgICAyLiBTZXZlcml0eSBlc3RpbWF0aW9uIChNU0UpCiAgICAgIDMuIFByb3BhZ2F0aW9uIHBhdGggcHJlZGljdGlvbiAoQkNFKQoKICAgIFRvdGFsIGxvc3MgPSBzdW0gb2Ygd2VpZ2h0ZWQgY29tcG9uZW50IGxvc3Nlcy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHByZWRpY3Rpb25faG9yaXpvbnM6IGxpc3RbaW50XSA9IFsyNCwgNzIsIDE2OCwgNzIwXSwKICAgICAgICBmb2NhbF9nYW1tYTogZmxvYXQgPSAyLjAsCiAgICAgICAgcG9zX3dlaWdodDogZmxvYXQgPSAxLjAsCiAgICAgICAgc2V2ZXJpdHlfd2VpZ2h0OiBmbG9hdCA9IDAuMywKICAgICAgICBwcm9wYWdhdGlvbl93ZWlnaHQ6IGZsb2F0ID0gMC4yLAogICAgICAgIGhvcml6b25fd2VpZ2h0czogZGljdFtpbnQsIGZsb2F0XSA9IE5vbmUsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucHJlZGljdGlvbl9ob3Jpem9ucyA9IHByZWRpY3Rpb25faG9yaXpvbnMKICAgICAgICBzZWxmLnNldmVyaXR5X3dlaWdodCA9IHNldmVyaXR5X3dlaWdodAogICAgICAgIHNlbGYucHJvcGFnYXRpb25fd2VpZ2h0ID0gcHJvcGFnYXRpb25fd2VpZ2h0CgogICAgICAgICMgRGlmZmVyZW50IHdlaWdodHMgZm9yIGRpZmZlcmVudCBob3Jpem9ucwogICAgICAgIGlmIGhvcml6b25fd2VpZ2h0cyBpcyBOb25lOgogICAgICAgICAgICBzZWxmLmhvcml6b25fd2VpZ2h0cyA9IHtoOiAxLjAgZm9yIGggaW4gcHJlZGljdGlvbl9ob3Jpem9uc30KICAgICAgICAgICAgIyBTaG9ydC10ZXJtIHByZWRpY3Rpb25zIHdlaWdodGVkIGhpZ2hlcgogICAgICAgICAgICBpZiAyNCBpbiBzZWxmLmhvcml6b25fd2VpZ2h0czoKICAgICAgICAgICAgICAgIHNlbGYuaG9yaXpvbl93ZWlnaHRzWzI0XSA9IDEuNQogICAgICAgICAgICBpZiA3MiBpbiBzZWxmLmhvcml6b25fd2VpZ2h0czoKICAgICAgICAgICAgICAgIHNlbGYuaG9yaXpvbl93ZWlnaHRzWzcyXSA9IDEuMgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuaG9yaXpvbl93ZWlnaHRzID0gaG9yaXpvbl93ZWlnaHRzCgogICAgICAgICMgRm9jYWwgbG9zcyBmb3IgZWFjaCBob3Jpem9uCiAgICAgICAgc2VsZi5mb2NhbF9sb3NzZXMgPSB7CiAgICAgICAgICAgIGg6IEZvY2FsTG9zcyhnYW1tYT1mb2NhbF9nYW1tYSwgYWxwaGE9MC43NSkKICAgICAgICAgICAgZm9yIGggaW4gcHJlZGljdGlvbl9ob3Jpem9ucwogICAgICAgIH0KCiAgICAgICAgIyBTZXZlcml0eSBsb3NzCiAgICAgICAgc2VsZi5zZXZlcml0eV9sb3NzID0gbm4uTVNFTG9zcygpCgogICAgICAgICMgUHJvcGFnYXRpb24gbG9zcwogICAgICAgIHNlbGYucHJvcGFnYXRpb25fbG9zcyA9IG5uLkJDRVdpdGhMb2dpdHNMb3NzKCkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSwKICAgICAgICB0YXJnZXRzOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSwKICAgICkgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcHJlZGljdGlvbnM6IE1vZGVsIG91dHB1dHMgZGljdCB3aXRoIGtleXM6CiAgICAgICAgICAgICAgICBjYXNjYWRlX3tofWgsIHNldmVyaXR5LCBwcm9wYWdhdGlvbi4KICAgICAgICAgICAgdGFyZ2V0czogR3JvdW5kIHRydXRoIGRpY3Qgd2l0aCBtYXRjaGluZyBrZXlzLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IHdpdGggdG90YWxfbG9zcyBhbmQgY29tcG9uZW50IGxvc3Nlcy4KICAgICAgICAiIiIKICAgICAgICBsb3NzZXMgPSB7fQogICAgICAgIHRvdGFsID0gdG9yY2gudGVuc29yKDAuMCwgZGV2aWNlPW5leHQoaXRlcihwcmVkaWN0aW9ucy52YWx1ZXMoKSkpLmRldmljZSkKCiAgICAgICAgIyAxLiBNdWx0aS1ob3Jpem9uIGNhc2NhZGUgY2xhc3NpZmljYXRpb24gbG9zc2VzCiAgICAgICAgZm9yIGggaW4gc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zOgogICAgICAgICAgICBrZXkgPSBmImNhc2NhZGVfe2h9aCIKICAgICAgICAgICAgaWYga2V5IGluIHByZWRpY3Rpb25zIGFuZCBrZXkgaW4gdGFyZ2V0czoKICAgICAgICAgICAgICAgIHByZWQgPSBwcmVkaWN0aW9uc1trZXldCiAgICAgICAgICAgICAgICB0Z3QgPSB0YXJnZXRzW2tleV0KICAgICAgICAgICAgICAgIGlmIHByZWQuZGltKCkgPT0gMDoKICAgICAgICAgICAgICAgICAgICBwcmVkID0gcHJlZC51bnNxdWVlemUoMCkKICAgICAgICAgICAgICAgIGlmIHRndC5kaW0oKSA9PSAwOgogICAgICAgICAgICAgICAgICAgIHRndCA9IHRndC51bnNxdWVlemUoMCkKICAgICAgICAgICAgICAgIGxvc3MgPSBzZWxmLmZvY2FsX2xvc3Nlc1toXShwcmVkLCB0Z3QpCiAgICAgICAgICAgICAgICB3ZWlnaHRlZCA9IHNlbGYuaG9yaXpvbl93ZWlnaHRzLmdldChoLCAxLjApICogbG9zcwogICAgICAgICAgICAgICAgbG9zc2VzW2YibG9zc197a2V5fSJdID0gbG9zcwogICAgICAgICAgICAgICAgdG90YWwgPSB0b3RhbCArIHdlaWdodGVkCgogICAgICAgICMgMi4gU2V2ZXJpdHkgZXN0aW1hdGlvbiBsb3NzCiAgICAgICAgaWYgInNldmVyaXR5IiBpbiBwcmVkaWN0aW9ucyBhbmQgInNldmVyaXR5IiBpbiB0YXJnZXRzOgogICAgICAgICAgICBzZXZfbG9zcyA9IHNlbGYuc2V2ZXJpdHlfbG9zcygKICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zWyJzZXZlcml0eSJdLCB0YXJnZXRzWyJzZXZlcml0eSJdCiAgICAgICAgICAgICkKICAgICAgICAgICAgbG9zc2VzWyJsb3NzX3NldmVyaXR5Il0gPSBzZXZfbG9zcwogICAgICAgICAgICB0b3RhbCA9IHRvdGFsICsgc2VsZi5zZXZlcml0eV93ZWlnaHQgKiBzZXZfbG9zcwoKICAgICAgICAjIDMuIFByb3BhZ2F0aW9uIHBhdGggbG9zcwogICAgICAgIGlmICJwcm9wYWdhdGlvbiIgaW4gcHJlZGljdGlvbnMgYW5kICJwcm9wYWdhdGlvbiIgaW4gdGFyZ2V0czoKICAgICAgICAgICAgcHJvcF9sb3NzID0gc2VsZi5wcm9wYWdhdGlvbl9sb3NzKAogICAgICAgICAgICAgICAgcHJlZGljdGlvbnNbInByb3BhZ2F0aW9uIl0sIHRhcmdldHNbInByb3BhZ2F0aW9uIl0KICAgICAgICAgICAgKQogICAgICAgICAgICBsb3NzZXNbImxvc3NfcHJvcGFnYXRpb24iXSA9IHByb3BfbG9zcwogICAgICAgICAgICB0b3RhbCA9IHRvdGFsICsgc2VsZi5wcm9wYWdhdGlvbl93ZWlnaHQgKiBwcm9wX2xvc3MKCiAgICAgICAgbG9zc2VzWyJ0b3RhbF9sb3NzIl0gPSB0b3RhbAogICAgICAgIHJldHVybiBsb3NzZXMKCgpjbGFzcyBNb25vdG9uaWNpdHlSZWd1bGFyaXphdGlvbihubi5Nb2R1bGUpOgogICAgIiIiRW5mb3JjZSBQKGxvbmdlciBob3Jpem9uKSA+PSBQKHNob3J0ZXIgaG9yaXpvbikuCgogICAgUGVuYWxpemVzIGNhc2VzIHdoZXJlIGEgc2hvcnRlci1ob3Jpem9uIGNhc2NhZGUgcHJvYmFiaWxpdHkgZXhjZWVkcwogICAgYSBsb25nZXItaG9yaXpvbiBwcm9iYWJpbGl0eSwgc2luY2UgYSBjYXNjYWRlIHdpdGhpbiA3IGRheXMgaW1wbGllcwogICAgYSBjYXNjYWRlIHdpdGhpbiAzMCBkYXlzLgoKICAgIFVzZXMgYSBoaW5nZSBsb3NzIG9uIHRoZSByYXcgbG9naXRzOiBSZUxVKHNob3J0ZXJfbG9naXQgLSBsb25nZXJfbG9naXQpLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByZWRpY3Rpb25faG9yaXpvbnM6IGxpc3RbaW50XSwgd2VpZ2h0OiBmbG9hdCA9IDAuNSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ob3Jpem9ucyA9IHNvcnRlZChwcmVkaWN0aW9uX2hvcml6b25zKQogICAgICAgIHNlbGYud2VpZ2h0ID0gd2VpZ2h0CgogICAgZGVmIGZvcndhcmQoc2VsZiwgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgZGV2aWNlID0gbmV4dChpdGVyKHByZWRpY3Rpb25zLnZhbHVlcygpKSkuZGV2aWNlCiAgICAgICAgbG9zcyA9IHRvcmNoLnRlbnNvcigwLjAsIGRldmljZT1kZXZpY2UpCiAgICAgICAgc29ydGVkX2tleXMgPSBbZiJjYXNjYWRlX3tofWgiIGZvciBoIGluIHNlbGYuaG9yaXpvbnNdCiAgICAgICAgYXZhaWxhYmxlID0gW2sgZm9yIGsgaW4gc29ydGVkX2tleXMgaWYgayBpbiBwcmVkaWN0aW9uc10KICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oYXZhaWxhYmxlKSAtIDEpOgogICAgICAgICAgICBzaG9ydGVyID0gcHJlZGljdGlvbnNbYXZhaWxhYmxlW2ldXQogICAgICAgICAgICBsb25nZXIgPSBwcmVkaWN0aW9uc1thdmFpbGFibGVbaSArIDFdXQogICAgICAgICAgICB2aW9sYXRpb25zID0gdG9yY2gucmVsdShzaG9ydGVyIC0gbG9uZ2VyKQogICAgICAgICAgICBsb3NzID0gbG9zcyArIHZpb2xhdGlvbnMubWVhbigpCiAgICAgICAgcmV0dXJuIHNlbGYud2VpZ2h0ICogbG9zcwo=", "training/trainer.py": "IiIiClRyYWluaW5nIHBpcGVsaW5lIGZvciB0aGUgVEdOIGFuZCBiYXNlbGluZSBtb2RlbHMuCgpJbXBsZW1lbnRzOgogIC0gVGVtcG9yYWwgdHJhaW4vdmFsL3Rlc3Qgc3BsaXR0aW5nIChubyBkYXRhIGxlYWthZ2UpCiAgLSBLLWZvbGQgdGVtcG9yYWwgY3Jvc3MtdmFsaWRhdGlvbgogIC0gRWFybHkgc3RvcHBpbmcgd2l0aCBwYXRpZW5jZQogIC0gTGVhcm5pbmcgcmF0ZSBzY2hlZHVsaW5nCiAgLSBHcmFkaWVudCBjbGlwcGluZwogIC0gQ2hlY2twb2ludCBzYXZpbmcKIiIiCgppbXBvcnQgb3MKaW1wb3J0IHRpbWUKaW1wb3J0IGNvcHkKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIgaW1wb3J0IENvc2luZUFubmVhbGluZ0xSCmZyb20gbG9ndXJ1IGltcG9ydCBsb2dnZXIKCmZyb20gLmxvc3NlcyBpbXBvcnQgQ2FzY2FkZUxvc3MKCgpjbGFzcyBUcmFpbmVyOgogICAgIiIiVHJhaW5zIHRoZSBUR04gbW9kZWwgb24gdGVtcG9yYWwgZ3JhcGggc25hcHNob3Qgc2VxdWVuY2VzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsOiBubi5Nb2R1bGUsCiAgICAgICAgY29uZmlnOiBkaWN0LAogICAgICAgIGRldmljZTogc3RyID0gImNwdSIsCiAgICAgICAgb3V0cHV0X2Rpcjogc3RyID0gIm91dHB1dHMvY2hlY2twb2ludHMiLAogICAgKToKICAgICAgICBzZWxmLm1vZGVsID0gbW9kZWwudG8oZGV2aWNlKQogICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2UKICAgICAgICBzZWxmLm91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICAgICAgc2VsZi5vdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAgICAgdHJhaW5fY2ZnID0gY29uZmlnLmdldCgidHJhaW5pbmciLCB7fSkKCiAgICAgICAgIyBMb3NzIGZ1bmN0aW9uCiAgICAgICAgc2VsZi5jcml0ZXJpb24gPSBDYXNjYWRlTG9zcygKICAgICAgICAgICAgcHJlZGljdGlvbl9ob3Jpem9ucz10cmFpbl9jZmcuZ2V0KAogICAgICAgICAgICAgICAgInByZWRpY3Rpb25faG9yaXpvbnMiLCBbMjQsIDcyLCAxNjgsIDcyMF0KICAgICAgICAgICAgKSwKICAgICAgICAgICAgZm9jYWxfZ2FtbWE9dHJhaW5fY2ZnLmdldCgiZm9jYWxfbG9zc19nYW1tYSIsIDIuMCksCiAgICAgICAgKQoKICAgICAgICAjIE9wdGltaXplcgogICAgICAgIHNlbGYub3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcoCiAgICAgICAgICAgIG1vZGVsLnBhcmFtZXRlcnMoKSwKICAgICAgICAgICAgbHI9dHJhaW5fY2ZnLmdldCgibGVhcm5pbmdfcmF0ZSIsIDNlLTQpLAogICAgICAgICAgICB3ZWlnaHRfZGVjYXk9dHJhaW5fY2ZnLmdldCgid2VpZ2h0X2RlY2F5IiwgMWUtNCksCiAgICAgICAgKQoKICAgICAgICAjIFNjaGVkdWxlcgogICAgICAgIHNjaGVkX2NmZyA9IHRyYWluX2NmZy5nZXQoInNjaGVkdWxlciIsIHt9KQogICAgICAgIHNlbGYuc2NoZWR1bGVyID0gQ29zaW5lQW5uZWFsaW5nTFIoCiAgICAgICAgICAgIHNlbGYub3B0aW1pemVyLAogICAgICAgICAgICBUX21heD1zY2hlZF9jZmcuZ2V0KCJUX21heCIsIDIwMCksCiAgICAgICAgICAgIGV0YV9taW49c2NoZWRfY2ZnLmdldCgiZXRhX21pbiIsIDFlLTYpLAogICAgICAgICkKCiAgICAgICAgIyBUcmFpbmluZyBwYXJhbXMKICAgICAgICBzZWxmLmVwb2NocyA9IHRyYWluX2NmZy5nZXQoImVwb2NocyIsIDIwMCkKICAgICAgICBzZWxmLnBhdGllbmNlID0gdHJhaW5fY2ZnLmdldCgicGF0aWVuY2UiLCAyMCkKICAgICAgICBzZWxmLmdyYWRfY2xpcCA9IDEuMAoKICAgICAgICAjIFRyYWNraW5nCiAgICAgICAgc2VsZi50cmFpbl9sb3NzZXMgPSBbXQogICAgICAgIHNlbGYudmFsX2xvc3NlcyA9IFtdCiAgICAgICAgc2VsZi5iZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICAgICAgc2VsZi5iZXN0X21vZGVsX3N0YXRlID0gTm9uZQogICAgICAgIHNlbGYuZXBvY2hzX25vX2ltcHJvdmUgPSAwCgogICAgZGVmIHRlbXBvcmFsX3NwbGl0KAogICAgICAgIHNlbGYsCiAgICAgICAgZGF0YTogbGlzdFtkaWN0XSwKICAgICAgICBsYWJlbHM6IGxpc3RbZGljdF0sCiAgICAgICAgdmFsX3JhdGlvOiBmbG9hdCA9IDAuMTUsCiAgICAgICAgdGVzdF9yYXRpbzogZmxvYXQgPSAwLjE1LAogICAgKSAtPiB0dXBsZToKICAgICAgICAiIiJTcGxpdCB0ZW1wb3JhbCBkYXRhIGNocm9ub2xvZ2ljYWxseSAobm8gZGF0YSBsZWFrYWdlKS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgKHRyYWluX2RhdGEsIHRyYWluX2xhYmVscywgdmFsX2RhdGEsIHZhbF9sYWJlbHMsCiAgICAgICAgICAgICB0ZXN0X2RhdGEsIHRlc3RfbGFiZWxzKQogICAgICAgICIiIgogICAgICAgIG4gPSBsZW4oZGF0YSkKICAgICAgICB0ZXN0X3N0YXJ0ID0gaW50KG4gKiAoMSAtIHRlc3RfcmF0aW8pKQogICAgICAgIHZhbF9zdGFydCA9IGludChuICogKDEgLSB0ZXN0X3JhdGlvIC0gdmFsX3JhdGlvKSkKCiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZGF0YVs6dmFsX3N0YXJ0XSwgbGFiZWxzWzp2YWxfc3RhcnRdLAogICAgICAgICAgICBkYXRhW3ZhbF9zdGFydDp0ZXN0X3N0YXJ0XSwgbGFiZWxzW3ZhbF9zdGFydDp0ZXN0X3N0YXJ0XSwKICAgICAgICAgICAgZGF0YVt0ZXN0X3N0YXJ0Ol0sIGxhYmVsc1t0ZXN0X3N0YXJ0Ol0sCiAgICAgICAgKQoKICAgIGRlZiBwcmVwYXJlX3NuYXBzaG90X2RpY3QoCiAgICAgICAgc2VsZiwKICAgICAgICBub2RlX2ZlYXR1cmVzOiBucC5uZGFycmF5LAogICAgICAgIGVkZ2VfaW5kZXhfZGljdDogZGljdFtzdHIsIHRvcmNoLlRlbnNvcl0sCiAgICAgICAgdGltZXN0YW1wOiBmbG9hdCwKICAgICAgICBlZGdlX2F0dHJfZGljdDogT3B0aW9uYWxbZGljdF0gPSBOb25lLAogICAgKSAtPiBkaWN0OgogICAgICAgICIiIkNvbnZlcnQgcmF3IGRhdGEgaW50byB0aGUgZm9ybWF0IGV4cGVjdGVkIGJ5IFRHTi5mb3J3YXJkKCkuIiIiCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm5vZGVfZmVhdHVyZXMiOiB0b3JjaC50ZW5zb3IoCiAgICAgICAgICAgICAgICBub2RlX2ZlYXR1cmVzLCBkdHlwZT10b3JjaC5mbG9hdDMyCiAgICAgICAgICAgICkudG8oc2VsZi5kZXZpY2UpLAogICAgICAgICAgICAiZWRnZV9pbmRleF9kaWN0IjogewogICAgICAgICAgICAgICAgazogdi50byhzZWxmLmRldmljZSkgZm9yIGssIHYgaW4gZWRnZV9pbmRleF9kaWN0Lml0ZW1zKCkKICAgICAgICAgICAgfSwKICAgICAgICAgICAgInRpbWVzdGFtcCI6IHRvcmNoLnRlbnNvcigKICAgICAgICAgICAgICAgIFt0aW1lc3RhbXBdICogbm9kZV9mZWF0dXJlcy5zaGFwZVswXSwgZHR5cGU9dG9yY2guZmxvYXQzMgogICAgICAgICAgICApLnRvKHNlbGYuZGV2aWNlKSwKICAgICAgICAgICAgImVkZ2VfYXR0cl9kaWN0IjogKAogICAgICAgICAgICAgICAge2s6IHYudG8oc2VsZi5kZXZpY2UpIGZvciBrLCB2IGluIGVkZ2VfYXR0cl9kaWN0Lml0ZW1zKCl9CiAgICAgICAgICAgICAgICBpZiBlZGdlX2F0dHJfZGljdAogICAgICAgICAgICAgICAgZWxzZSBOb25lCiAgICAgICAgICAgICksCiAgICAgICAgfQoKICAgIGRlZiBwcmVwYXJlX3RhcmdldF9kaWN0KHNlbGYsIGxhYmVsX3JvdzogZGljdCkgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgIiIiQ29udmVydCBsYWJlbCByb3cgaW50byB0YXJnZXQgdGVuc29yIGRpY3QuIiIiCiAgICAgICAgdGFyZ2V0cyA9IHt9CiAgICAgICAgZm9yIGtleSwgdmFsIGluIGxhYmVsX3Jvdy5pdGVtcygpOgogICAgICAgICAgICBpZiBrZXkuc3RhcnRzd2l0aCgiY2FzY2FkZV8iKSBvciBrZXkgaW4gKCJzZXZlcml0eSIsICJyaXNrX3Njb3JlIik6CiAgICAgICAgICAgICAgICB0Z3Rfa2V5ID0gInNldmVyaXR5IiBpZiBrZXkgaW4gKCJzZXZlcml0eSIsICJyaXNrX3Njb3JlIikgZWxzZSBrZXkKICAgICAgICAgICAgICAgIHRhcmdldHNbdGd0X2tleV0gPSB0b3JjaC50ZW5zb3IoCiAgICAgICAgICAgICAgICAgICAgZmxvYXQodmFsKSwgZHR5cGU9dG9yY2guZmxvYXQzMgogICAgICAgICAgICAgICAgKS50byhzZWxmLmRldmljZSkKICAgICAgICByZXR1cm4gdGFyZ2V0cwoKICAgIGRlZiB0cmFpbl9lcG9jaCgKICAgICAgICBzZWxmLAogICAgICAgIHRyYWluX2RhdGE6IGxpc3RbZGljdF0sCiAgICAgICAgdHJhaW5fbGFiZWxzOiBsaXN0W2RpY3RdLAogICAgICAgIHRicHR0X3dpbmRvdzogaW50ID0gMTAsCiAgICApIC0+IGZsb2F0OgogICAgICAgICIiIlRyYWluIGZvciBvbmUgZXBvY2ggb3ZlciB0aGUgdGVtcG9yYWwgc2VxdWVuY2UuCgogICAgICAgIFVzZXMgd2luZG93ZWQgVEJQVFQ6IGFjY3VtdWxhdGVzIGxvc3Mgb3ZlciB0YnB0dF93aW5kb3cgc3RlcHMsCiAgICAgICAgYmFja3Byb3BzIG9uY2UsIHRoZW4gZGV0YWNoZXMgbWVtb3J5IGF0IHRoZSB3aW5kb3cgYm91bmRhcnkuCiAgICAgICAgTWVtb3J5IGlzIE5PVCByZXNldCBhdCBlcG9jaCBzdGFydCDigJQgaXQgcGVyc2lzdHMgYWNyb3NzIGVwb2Nocy4KICAgICAgICAiIiIKICAgICAgICBzZWxmLm1vZGVsLnRyYWluKCkKICAgICAgICBlcG9jaF9sb3NzID0gMC4wCiAgICAgICAgbl9iYXRjaGVzID0gMAogICAgICAgIHdpbmRvd19sb3NzID0gdG9yY2gudGVuc29yKDAuMCwgZGV2aWNlPXNlbGYuZGV2aWNlKQogICAgICAgIHdpbmRvd19jb3VudCA9IDAKCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKHRyYWluX2RhdGEpKToKICAgICAgICAgICAgc25hcHNob3QgPSB0cmFpbl9kYXRhW2ldCiAgICAgICAgICAgIHRhcmdldHMgPSB0cmFpbl9sYWJlbHNbaV0KCiAgICAgICAgICAgICMgRm9yd2FyZCBwYXNzCiAgICAgICAgICAgIHByZWRpY3Rpb25zID0gc2VsZi5tb2RlbCgKICAgICAgICAgICAgICAgIG5vZGVfZmVhdHVyZXM9c25hcHNob3RbIm5vZGVfZmVhdHVyZXMiXSwKICAgICAgICAgICAgICAgIGVkZ2VfaW5kZXhfZGljdD1zbmFwc2hvdFsiZWRnZV9pbmRleF9kaWN0Il0sCiAgICAgICAgICAgICAgICB0aW1lc3RhbXBzPXNuYXBzaG90WyJ0aW1lc3RhbXAiXSwKICAgICAgICAgICAgICAgIGVkZ2VfYXR0cl9kaWN0PXNuYXBzaG90LmdldCgiZWRnZV9hdHRyX2RpY3QiKSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyBDb21wdXRlIGxvc3MKICAgICAgICAgICAgbG9zc2VzID0gc2VsZi5jcml0ZXJpb24ocHJlZGljdGlvbnMsIHRhcmdldHMpCiAgICAgICAgICAgIGxvc3MgPSBsb3NzZXNbInRvdGFsX2xvc3MiXQogICAgICAgICAgICB3aW5kb3dfbG9zcyA9IHdpbmRvd19sb3NzICsgbG9zcwogICAgICAgICAgICB3aW5kb3dfY291bnQgKz0gMQogICAgICAgICAgICBlcG9jaF9sb3NzICs9IGxvc3MuaXRlbSgpCiAgICAgICAgICAgIG5fYmF0Y2hlcyArPSAxCgogICAgICAgICAgICAjIFdpbmRvd2VkIFRCUFRUOiBiYWNrcHJvcCBhdCB3aW5kb3cgYm91bmRhcmllcwogICAgICAgICAgICBpZiB3aW5kb3dfY291bnQgPj0gdGJwdHRfd2luZG93IG9yIGkgPT0gbGVuKHRyYWluX2RhdGEpIC0gMToKICAgICAgICAgICAgICAgIGF2Z19sb3NzID0gd2luZG93X2xvc3MgLyB3aW5kb3dfY291bnQKICAgICAgICAgICAgICAgIHNlbGYub3B0aW1pemVyLnplcm9fZ3JhZCgpCiAgICAgICAgICAgICAgICBhdmdfbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8oCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbC5wYXJhbWV0ZXJzKCksIHNlbGYuZ3JhZF9jbGlwCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBzZWxmLm9wdGltaXplci5zdGVwKCkKICAgICAgICAgICAgICAgIGlmIGhhc2F0dHIoc2VsZi5tb2RlbCwgIm1lbW9yeSIpOgogICAgICAgICAgICAgICAgICAgIHNlbGYubW9kZWwubWVtb3J5LmRldGFjaF9tZW1vcnkoKQogICAgICAgICAgICAgICAgd2luZG93X2xvc3MgPSB0b3JjaC50ZW5zb3IoMC4wLCBkZXZpY2U9c2VsZi5kZXZpY2UpCiAgICAgICAgICAgICAgICB3aW5kb3dfY291bnQgPSAwCgogICAgICAgIHJldHVybiBlcG9jaF9sb3NzIC8gbWF4KG5fYmF0Y2hlcywgMSkKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgdmFsaWRhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICB2YWxfZGF0YTogbGlzdFtkaWN0XSwKICAgICAgICB2YWxfbGFiZWxzOiBsaXN0W2RpY3RdLAogICAgICAgIHdhcm11cF9kYXRhOiBsaXN0W2RpY3RdID0gTm9uZSwKICAgICkgLT4gdHVwbGVbZmxvYXQsIGRpY3RdOgogICAgICAgICIiIlZhbGlkYXRlIG9uIGhlbGQtb3V0IHRlbXBvcmFsIGRhdGEuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHZhbF9kYXRhOiBWYWxpZGF0aW9uIHNuYXBzaG90cy4KICAgICAgICAgICAgdmFsX2xhYmVsczogVmFsaWRhdGlvbiBsYWJlbHMuCiAgICAgICAgICAgIHdhcm11cF9kYXRhOiBJZiBwcm92aWRlZCwgcnVuIHRocm91Z2ggdGhlc2UgZmlyc3QgdG8gYnVpbGQgbWVtb3J5IHN0YXRlLgogICAgICAgICIiIgogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgdmFsX2xvc3MgPSAwLjAKICAgICAgICBhbGxfcHJlZGljdGlvbnMgPSB7CiAgICAgICAgICAgIGYiY2FzY2FkZV97aH1oIjogW10gZm9yIGggaW4gc2VsZi5jb25maWcuZ2V0KAogICAgICAgICAgICAgICAgInRyYWluaW5nIiwge30KICAgICAgICAgICAgKS5nZXQoInByZWRpY3Rpb25faG9yaXpvbnMiLCBbMjQsIDcyLCAxNjgsIDcyMF0pCiAgICAgICAgfQogICAgICAgIGFsbF90YXJnZXRzID0ge2s6IFtdIGZvciBrIGluIGFsbF9wcmVkaWN0aW9uc30KICAgICAgICBuID0gMAoKICAgICAgICAjIFdhcm0gdXAgbWVtb3J5IG9uIHRyYWluaW5nIGRhdGEKICAgICAgICBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJyZXNldF9tZW1vcnkiKToKICAgICAgICAgICAgc2VsZi5tb2RlbC5yZXNldF9tZW1vcnkoKQoKICAgICAgICBpZiB3YXJtdXBfZGF0YToKICAgICAgICAgICAgZm9yIHNuYXBzaG90IGluIHdhcm11cF9kYXRhOgogICAgICAgICAgICAgICAgc2VsZi5tb2RlbCgKICAgICAgICAgICAgICAgICAgICBub2RlX2ZlYXR1cmVzPXNuYXBzaG90WyJub2RlX2ZlYXR1cmVzIl0sCiAgICAgICAgICAgICAgICAgICAgZWRnZV9pbmRleF9kaWN0PXNuYXBzaG90WyJlZGdlX2luZGV4X2RpY3QiXSwKICAgICAgICAgICAgICAgICAgICB0aW1lc3RhbXBzPXNuYXBzaG90WyJ0aW1lc3RhbXAiXSwKICAgICAgICAgICAgICAgICAgICBlZGdlX2F0dHJfZGljdD1zbmFwc2hvdC5nZXQoImVkZ2VfYXR0cl9kaWN0IiksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJtZW1vcnkiKToKICAgICAgICAgICAgICAgICAgICBzZWxmLm1vZGVsLm1lbW9yeS5kZXRhY2hfbWVtb3J5KCkKCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKHZhbF9kYXRhKSk6CiAgICAgICAgICAgIHNuYXBzaG90ID0gdmFsX2RhdGFbaV0KICAgICAgICAgICAgdGFyZ2V0cyA9IHZhbF9sYWJlbHNbaV0KCiAgICAgICAgICAgIHByZWRpY3Rpb25zID0gc2VsZi5tb2RlbCgKICAgICAgICAgICAgICAgIG5vZGVfZmVhdHVyZXM9c25hcHNob3RbIm5vZGVfZmVhdHVyZXMiXSwKICAgICAgICAgICAgICAgIGVkZ2VfaW5kZXhfZGljdD1zbmFwc2hvdFsiZWRnZV9pbmRleF9kaWN0Il0sCiAgICAgICAgICAgICAgICB0aW1lc3RhbXBzPXNuYXBzaG90WyJ0aW1lc3RhbXAiXSwKICAgICAgICAgICAgICAgIGVkZ2VfYXR0cl9kaWN0PXNuYXBzaG90LmdldCgiZWRnZV9hdHRyX2RpY3QiKSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgbG9zc2VzID0gc2VsZi5jcml0ZXJpb24ocHJlZGljdGlvbnMsIHRhcmdldHMpCiAgICAgICAgICAgIHZhbF9sb3NzICs9IGxvc3Nlc1sidG90YWxfbG9zcyJdLml0ZW0oKQoKICAgICAgICAgICAgZm9yIGtleSBpbiBhbGxfcHJlZGljdGlvbnM6CiAgICAgICAgICAgICAgICBpZiBrZXkgaW4gcHJlZGljdGlvbnM6CiAgICAgICAgICAgICAgICAgICAgcHJlZF92YWwgPSB0b3JjaC5zaWdtb2lkKHByZWRpY3Rpb25zW2tleV0pLmNwdSgpLml0ZW0oKQogICAgICAgICAgICAgICAgICAgIGFsbF9wcmVkaWN0aW9uc1trZXldLmFwcGVuZChwcmVkX3ZhbCkKICAgICAgICAgICAgICAgIGlmIGtleSBpbiB0YXJnZXRzOgogICAgICAgICAgICAgICAgICAgIGFsbF90YXJnZXRzW2tleV0uYXBwZW5kKHRhcmdldHNba2V5XS5jcHUoKS5pdGVtKCkpCgogICAgICAgICAgICBuICs9IDEKCiAgICAgICAgICAgIGlmIGhhc2F0dHIoc2VsZi5tb2RlbCwgIm1lbW9yeSIpOgogICAgICAgICAgICAgICAgc2VsZi5tb2RlbC5tZW1vcnkuZGV0YWNoX21lbW9yeSgpCgogICAgICAgIGF2Z19sb3NzID0gdmFsX2xvc3MgLyBtYXgobiwgMSkKICAgICAgICByZXR1cm4gYXZnX2xvc3MsIGFsbF9wcmVkaWN0aW9ucywgYWxsX3RhcmdldHMKCiAgICBkZWYgdHJhaW4oCiAgICAgICAgc2VsZiwKICAgICAgICB0cmFpbl9kYXRhOiBsaXN0W2RpY3RdLAogICAgICAgIHRyYWluX2xhYmVsczogbGlzdFtkaWN0XSwKICAgICAgICB2YWxfZGF0YTogbGlzdFtkaWN0XSwKICAgICAgICB2YWxfbGFiZWxzOiBsaXN0W2RpY3RdLAogICAgKSAtPiBkaWN0OgogICAgICAgICIiIkZ1bGwgdHJhaW5pbmcgbG9vcCB3aXRoIGVhcmx5IHN0b3BwaW5nLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IHdpdGggdHJhaW5pbmcgaGlzdG9yeSBhbmQgYmVzdCBtZXRyaWNzLgogICAgICAgICIiIgogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICBmIlN0YXJ0aW5nIHRyYWluaW5nOiB7c2VsZi5lcG9jaHN9IGVwb2NocywgIgogICAgICAgICAgICBmInBhdGllbmNlPXtzZWxmLnBhdGllbmNlfSwgZGV2aWNlPXtzZWxmLmRldmljZX0iCiAgICAgICAgKQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICBmIlRyYWluOiB7bGVuKHRyYWluX2RhdGEpfSBzbmFwc2hvdHMsIFZhbDoge2xlbih2YWxfZGF0YSl9IHNuYXBzaG90cyIKICAgICAgICApCgogICAgICAgIHN0YXJ0X3RpbWUgPSB0aW1lLnRpbWUoKQoKICAgICAgICAjIFJlc2V0IG1lbW9yeSBvbmx5IG9uY2UgYXQgdHJhaW5pbmcgc3RhcnQKICAgICAgICBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJyZXNldF9tZW1vcnkiKToKICAgICAgICAgICAgc2VsZi5tb2RlbC5yZXNldF9tZW1vcnkoKQoKICAgICAgICBmb3IgZXBvY2ggaW4gcmFuZ2Uoc2VsZi5lcG9jaHMpOgogICAgICAgICAgICAjIFRyYWluIChtZW1vcnkgcGVyc2lzdHMgYWNyb3NzIGVwb2NocykKICAgICAgICAgICAgdHJhaW5fbG9zcyA9IHNlbGYudHJhaW5fZXBvY2godHJhaW5fZGF0YSwgdHJhaW5fbGFiZWxzKQogICAgICAgICAgICBzZWxmLnRyYWluX2xvc3Nlcy5hcHBlbmQodHJhaW5fbG9zcykKCiAgICAgICAgICAgICMgVmFsaWRhdGUgKHdpdGggd2FybXVwIG9uIHRyYWluaW5nIGRhdGEpCiAgICAgICAgICAgIHZhbF9sb3NzLCB2YWxfcHJlZHMsIHZhbF90Z3RzID0gc2VsZi52YWxpZGF0ZSgKICAgICAgICAgICAgICAgIHZhbF9kYXRhLCB2YWxfbGFiZWxzLCB3YXJtdXBfZGF0YT10cmFpbl9kYXRhCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2VsZi52YWxfbG9zc2VzLmFwcGVuZCh2YWxfbG9zcykKCiAgICAgICAgICAgICMgTGVhcm5pbmcgcmF0ZSBzdGVwCiAgICAgICAgICAgIHNlbGYuc2NoZWR1bGVyLnN0ZXAoKQogICAgICAgICAgICBjdXJyZW50X2xyID0gc2VsZi5vcHRpbWl6ZXIucGFyYW1fZ3JvdXBzWzBdWyJsciJdCgogICAgICAgICAgICAjIEVhcmx5IHN0b3BwaW5nIGNoZWNrCiAgICAgICAgICAgIGlmIHZhbF9sb3NzIDwgc2VsZi5iZXN0X3ZhbF9sb3NzOgogICAgICAgICAgICAgICAgc2VsZi5iZXN0X3ZhbF9sb3NzID0gdmFsX2xvc3MKICAgICAgICAgICAgICAgIHNlbGYuYmVzdF9tb2RlbF9zdGF0ZSA9IGNvcHkuZGVlcGNvcHkoc2VsZi5tb2RlbC5zdGF0ZV9kaWN0KCkpCiAgICAgICAgICAgICAgICBzZWxmLmVwb2Noc19ub19pbXByb3ZlID0gMAoKICAgICAgICAgICAgICAgICMgU2F2ZSBjaGVja3BvaW50CiAgICAgICAgICAgICAgICB0b3JjaC5zYXZlKAogICAgICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAgICAgImVwb2NoIjogZXBvY2gsCiAgICAgICAgICAgICAgICAgICAgICAgICJtb2RlbF9zdGF0ZV9kaWN0Ijogc2VsZi5iZXN0X21vZGVsX3N0YXRlLAogICAgICAgICAgICAgICAgICAgICAgICAib3B0aW1pemVyX3N0YXRlX2RpY3QiOiBzZWxmLm9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICJ2YWxfbG9zcyI6IHZhbF9sb3NzLAogICAgICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICAgICAgc2VsZi5vdXRwdXRfZGlyIC8gImJlc3RfbW9kZWwucHQiLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc2VsZi5lcG9jaHNfbm9faW1wcm92ZSArPSAxCgogICAgICAgICAgICBpZiAoZXBvY2ggKyAxKSAlIDEwID09IDAgb3IgZXBvY2ggPT0gMDoKICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAgICAgICAgIGYiRXBvY2gge2Vwb2NoICsgMX0ve3NlbGYuZXBvY2hzfSB8ICIKICAgICAgICAgICAgICAgICAgICBmIlRyYWluIExvc3M6IHt0cmFpbl9sb3NzOi42Zn0gfCAiCiAgICAgICAgICAgICAgICAgICAgZiJWYWwgTG9zczoge3ZhbF9sb3NzOi42Zn0gfCAiCiAgICAgICAgICAgICAgICAgICAgZiJMUjoge2N1cnJlbnRfbHI6LjJlfSB8ICIKICAgICAgICAgICAgICAgICAgICBmIk5vIEltcHJvdmU6IHtzZWxmLmVwb2Noc19ub19pbXByb3ZlfS97c2VsZi5wYXRpZW5jZX0iCiAgICAgICAgICAgICAgICApCgogICAgICAgICAgICBpZiBzZWxmLmVwb2Noc19ub19pbXByb3ZlID49IHNlbGYucGF0aWVuY2U6CiAgICAgICAgICAgICAgICBsb2dnZXIuaW5mbyhmIkVhcmx5IHN0b3BwaW5nIGF0IGVwb2NoIHtlcG9jaCArIDF9IikKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgICMgUmVzdG9yZSBiZXN0IG1vZGVsCiAgICAgICAgaWYgc2VsZi5iZXN0X21vZGVsX3N0YXRlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLm1vZGVsLmxvYWRfc3RhdGVfZGljdChzZWxmLmJlc3RfbW9kZWxfc3RhdGUpCgogICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHN0YXJ0X3RpbWUKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgZiJUcmFpbmluZyBjb21wbGV0ZSBpbiB7ZWxhcHNlZDouMWZ9cy4gIgogICAgICAgICAgICBmIkJlc3QgdmFsIGxvc3M6IHtzZWxmLmJlc3RfdmFsX2xvc3M6LjZmfSIKICAgICAgICApCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ0cmFpbl9sb3NzZXMiOiBzZWxmLnRyYWluX2xvc3NlcywKICAgICAgICAgICAgInZhbF9sb3NzZXMiOiBzZWxmLnZhbF9sb3NzZXMsCiAgICAgICAgICAgICJiZXN0X3ZhbF9sb3NzIjogc2VsZi5iZXN0X3ZhbF9sb3NzLAogICAgICAgICAgICAiYmVzdF9lcG9jaCI6IGxlbihzZWxmLnRyYWluX2xvc3NlcykgLSBzZWxmLmVwb2Noc19ub19pbXByb3ZlLAogICAgICAgICAgICAidG90YWxfZXBvY2hzIjogbGVuKHNlbGYudHJhaW5fbG9zc2VzKSwKICAgICAgICAgICAgInRyYWluaW5nX3RpbWUiOiBlbGFwc2VkLAogICAgICAgIH0KCiAgICBkZWYgdGVtcG9yYWxfY3Jvc3NfdmFsaWRhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICBhbGxfZGF0YTogbGlzdFtkaWN0XSwKICAgICAgICBhbGxfbGFiZWxzOiBsaXN0W2RpY3RdLAogICAgICAgIG5fZm9sZHM6IGludCA9IDUsCiAgICApIC0+IGxpc3RbZGljdF06CiAgICAgICAgIiIiRXhwYW5kaW5nIHdpbmRvdyB0ZW1wb3JhbCBjcm9zcy12YWxpZGF0aW9uLgoKICAgICAgICBVc2VzIGV4cGFuZGluZyB0cmFpbmluZyB3aW5kb3dzIHRvIHJlc3BlY3QgdGVtcG9yYWwgb3JkZXJpbmc6CiAgICAgICAgICBGb2xkIDE6IFRyYWluIFswOm4xXSwgVmFsIFtuMTpuMl0KICAgICAgICAgIEZvbGQgMjogVHJhaW4gWzA6bjJdLCBWYWwgW24yOm4zXQogICAgICAgICAgLi4uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIExpc3Qgb2YgcGVyLWZvbGQgcmVzdWx0cy4KICAgICAgICAiIiIKICAgICAgICBuID0gbGVuKGFsbF9kYXRhKQogICAgICAgIGZvbGRfc2l6ZSA9IG4gLy8gKG5fZm9sZHMgKyAxKQogICAgICAgIHJlc3VsdHMgPSBbXQoKICAgICAgICBmb3IgZm9sZCBpbiByYW5nZShuX2ZvbGRzKToKICAgICAgICAgICAgdHJhaW5fZW5kID0gZm9sZF9zaXplICogKGZvbGQgKyAxKQogICAgICAgICAgICB2YWxfZW5kID0gbWluKHRyYWluX2VuZCArIGZvbGRfc2l6ZSwgbikKCiAgICAgICAgICAgIGlmIHZhbF9lbmQgPD0gdHJhaW5fZW5kOgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgIHRyYWluX2QgPSBhbGxfZGF0YVs6dHJhaW5fZW5kXQogICAgICAgICAgICB0cmFpbl9sID0gYWxsX2xhYmVsc1s6dHJhaW5fZW5kXQogICAgICAgICAgICB2YWxfZCA9IGFsbF9kYXRhW3RyYWluX2VuZDp2YWxfZW5kXQogICAgICAgICAgICB2YWxfbCA9IGFsbF9sYWJlbHNbdHJhaW5fZW5kOnZhbF9lbmRdCgogICAgICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgICAgIGYiRm9sZCB7Zm9sZCArIDF9L3tuX2ZvbGRzfTogIgogICAgICAgICAgICAgICAgZiJUcmFpbiBbezB9Ont0cmFpbl9lbmR9XSwgVmFsIFt7dHJhaW5fZW5kfTp7dmFsX2VuZH1dIgogICAgICAgICAgICApCgogICAgICAgICAgICAjIFJlc2V0IG1vZGVsIGFuZCBvcHRpbWl6ZXIgZm9yIGVhY2ggZm9sZAogICAgICAgICAgICBzZWxmLm1vZGVsLmFwcGx5KHNlbGYuX3Jlc2V0X3dlaWdodHMpCiAgICAgICAgICAgIHNlbGYub3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcoCiAgICAgICAgICAgICAgICBzZWxmLm1vZGVsLnBhcmFtZXRlcnMoKSwKICAgICAgICAgICAgICAgIGxyPXNlbGYuY29uZmlnLmdldCgidHJhaW5pbmciLCB7fSkuZ2V0KCJsZWFybmluZ19yYXRlIiwgM2UtNCksCiAgICAgICAgICAgICAgICB3ZWlnaHRfZGVjYXk9c2VsZi5jb25maWcuZ2V0KCJ0cmFpbmluZyIsIHt9KS5nZXQoCiAgICAgICAgICAgICAgICAgICAgIndlaWdodF9kZWNheSIsIDFlLTQKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2VsZi5iZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICAgICAgICAgIHNlbGYuZXBvY2hzX25vX2ltcHJvdmUgPSAwCiAgICAgICAgICAgIHNlbGYudHJhaW5fbG9zc2VzID0gW10KICAgICAgICAgICAgc2VsZi52YWxfbG9zc2VzID0gW10KCiAgICAgICAgICAgIGZvbGRfcmVzdWx0ID0gc2VsZi50cmFpbih0cmFpbl9kLCB0cmFpbl9sLCB2YWxfZCwgdmFsX2wpCiAgICAgICAgICAgIGZvbGRfcmVzdWx0WyJmb2xkIl0gPSBmb2xkICsgMQoKICAgICAgICAgICAgIyBFdmFsdWF0ZSBvbiB2YWxpZGF0aW9uIHNldAogICAgICAgICAgICB2YWxfbG9zcywgdmFsX3ByZWRzLCB2YWxfdGd0cyA9IHNlbGYudmFsaWRhdGUodmFsX2QsIHZhbF9sKQogICAgICAgICAgICBmb2xkX3Jlc3VsdFsidmFsX3ByZWRpY3Rpb25zIl0gPSB2YWxfcHJlZHMKICAgICAgICAgICAgZm9sZF9yZXN1bHRbInZhbF90YXJnZXRzIl0gPSB2YWxfdGd0cwoKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoZm9sZF9yZXN1bHQpCgogICAgICAgIHJldHVybiByZXN1bHRzCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9yZXNldF93ZWlnaHRzKG0pOgogICAgICAgICIiIlJlc2V0IG1vZGVsIHdlaWdodHMgZm9yIGNyb3NzLXZhbGlkYXRpb24gZm9sZHMuIiIiCiAgICAgICAgaWYgaGFzYXR0cihtLCAicmVzZXRfcGFyYW1ldGVycyIpOgogICAgICAgICAgICBtLnJlc2V0X3BhcmFtZXRlcnMoKQo=", "evaluation/metrics.py": "IiIiCkV2YWx1YXRpb24gbWV0cmljcyBmb3IgY2FzY2FkZSBwcmVkaWN0aW9uLgoKQ29tcHV0ZXMgY2xhc3NpZmljYXRpb24gbWV0cmljcywgY2FsaWJyYXRpb24sIGFuZCBjYXNjYWRlLXNwZWNpZmljCm1ldHJpY3MgbGlrZSBsZWFkIHRpbWUgYW5kIGVhcmx5IHdhcm5pbmcgc2NvcmUuCiIiIgoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCAoCiAgICByb2NfYXVjX3Njb3JlLAogICAgYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUsCiAgICBmMV9zY29yZSwKICAgIHByZWNpc2lvbl9zY29yZSwKICAgIHJlY2FsbF9zY29yZSwKICAgIGJyaWVyX3Njb3JlX2xvc3MsCiAgICBjb25mdXNpb25fbWF0cml4LAogICAgbWF0dGhld3NfY29ycmNvZWYsCiAgICBwcmVjaXNpb25fcmVjYWxsX2N1cnZlLAogICAgcm9jX2N1cnZlLAogICAgY2xhc3NpZmljYXRpb25fcmVwb3J0LAopCmZyb20gbG9ndXJ1IGltcG9ydCBsb2dnZXIKCgpjbGFzcyBNZXRyaWNzQ2FsY3VsYXRvcjoKICAgICIiIkNvbXB1dGVzIGNvbXByZWhlbnNpdmUgZXZhbHVhdGlvbiBtZXRyaWNzIGZvciBjYXNjYWRlIHByZWRpY3Rpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJlZGljdGlvbl9ob3Jpem9uczogbGlzdFtpbnRdID0gWzI0LCA3MiwgMTY4LCA3MjBdLAogICAgICAgIHRocmVzaG9sZDogZmxvYXQgPSAwLjUsCiAgICApOgogICAgICAgIHNlbGYucHJlZGljdGlvbl9ob3Jpem9ucyA9IHByZWRpY3Rpb25faG9yaXpvbnMKICAgICAgICBzZWxmLnRocmVzaG9sZCA9IHRocmVzaG9sZAoKICAgIGRlZiBjb21wdXRlX2FsbF9tZXRyaWNzKAogICAgICAgIHNlbGYsCiAgICAgICAgeV90cnVlOiBucC5uZGFycmF5LAogICAgICAgIHlfcHJvYjogbnAubmRhcnJheSwKICAgICAgICB0aHJlc2hvbGQ6IGZsb2F0ID0gTm9uZSwKICAgICkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJDb21wdXRlIGFsbCBtZXRyaWNzIGZvciBhIHNpbmdsZSBob3Jpem9uLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB5X3RydWU6IEJpbmFyeSBsYWJlbHMgW25fc2FtcGxlc10uCiAgICAgICAgICAgIHlfcHJvYjogUHJlZGljdGVkIHByb2JhYmlsaXRpZXMgW25fc2FtcGxlc10uCiAgICAgICAgICAgIHRocmVzaG9sZDogQ2xhc3NpZmljYXRpb24gdGhyZXNob2xkLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IG9mIG1ldHJpY19uYW1lIC0+IHZhbHVlLgogICAgICAgICIiIgogICAgICAgIGlmIHRocmVzaG9sZCBpcyBOb25lOgogICAgICAgICAgICB0aHJlc2hvbGQgPSBzZWxmLnRocmVzaG9sZAoKICAgICAgICB5X3RydWUgPSBucC5hc2FycmF5KHlfdHJ1ZSkuZmxhdHRlbigpCiAgICAgICAgeV9wcm9iID0gbnAuYXNhcnJheSh5X3Byb2IpLmZsYXR0ZW4oKQoKICAgICAgICAjIEVuc3VyZSB2YWxpZAogICAgICAgIG1hc2sgPSBucC5pc2Zpbml0ZSh5X3Byb2IpICYgbnAuaXNmaW5pdGUoeV90cnVlKQogICAgICAgIHlfdHJ1ZSA9IHlfdHJ1ZVttYXNrXQogICAgICAgIHlfcHJvYiA9IHlfcHJvYlttYXNrXQoKICAgICAgICBpZiBsZW4oeV90cnVlKSA9PSAwIG9yIHlfdHJ1ZS5zdW0oKSA9PSAwIG9yIHlfdHJ1ZS5zdW0oKSA9PSBsZW4oeV90cnVlKToKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIkNhbm5vdCBjb21wdXRlIG1ldHJpY3M6IG5vIHBvc2l0aXZlIG9yIG5vIG5lZ2F0aXZlIHNhbXBsZXMiKQogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHlfbWV0cmljcygpCgogICAgICAgIHlfcHJlZCA9ICh5X3Byb2IgPj0gdGhyZXNob2xkKS5hc3R5cGUoaW50KQoKICAgICAgICBtZXRyaWNzID0ge30KCiAgICAgICAgIyBEaXNjcmltaW5hdGlvbiBtZXRyaWNzCiAgICAgICAgbWV0cmljc1siYXVyb2MiXSA9IHJvY19hdWNfc2NvcmUoeV90cnVlLCB5X3Byb2IpCiAgICAgICAgbWV0cmljc1siYXVwcmMiXSA9IGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHlfdHJ1ZSwgeV9wcm9iKQoKICAgICAgICAjIENsYXNzaWZpY2F0aW9uIG1ldHJpY3MKICAgICAgICBtZXRyaWNzWyJmMSJdID0gZjFfc2NvcmUoeV90cnVlLCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkKICAgICAgICBtZXRyaWNzWyJwcmVjaXNpb24iXSA9IHByZWNpc2lvbl9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKQogICAgICAgIG1ldHJpY3NbInJlY2FsbCJdID0gcmVjYWxsX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCB6ZXJvX2RpdmlzaW9uPTApCiAgICAgICAgbWV0cmljc1sibWNjIl0gPSBtYXR0aGV3c19jb3JyY29lZih5X3RydWUsIHlfcHJlZCkKCiAgICAgICAgIyBDYWxpYnJhdGlvbgogICAgICAgIG1ldHJpY3NbImJyaWVyX3Njb3JlIl0gPSBicmllcl9zY29yZV9sb3NzKHlfdHJ1ZSwgeV9wcm9iKQoKICAgICAgICAjIENvbmZ1c2lvbiBtYXRyaXggZWxlbWVudHMKICAgICAgICB0biwgZnAsIGZuLCB0cCA9IGNvbmZ1c2lvbl9tYXRyaXgoCiAgICAgICAgICAgIHlfdHJ1ZSwgeV9wcmVkLCBsYWJlbHM9WzAsIDFdCiAgICAgICAgKS5yYXZlbCgpCiAgICAgICAgbWV0cmljc1sidHJ1ZV9wb3NpdGl2ZXMiXSA9IGludCh0cCkKICAgICAgICBtZXRyaWNzWyJmYWxzZV9wb3NpdGl2ZXMiXSA9IGludChmcCkKICAgICAgICBtZXRyaWNzWyJ0cnVlX25lZ2F0aXZlcyJdID0gaW50KHRuKQogICAgICAgIG1ldHJpY3NbImZhbHNlX25lZ2F0aXZlcyJdID0gaW50KGZuKQogICAgICAgIG1ldHJpY3NbInNwZWNpZmljaXR5Il0gPSB0biAvICh0biArIGZwKSBpZiAodG4gKyBmcCkgPiAwIGVsc2UgMAogICAgICAgIG1ldHJpY3NbIm5wdiJdID0gdG4gLyAodG4gKyBmbikgaWYgKHRuICsgZm4pID4gMCBlbHNlIDAKCiAgICAgICAgIyBPcHRpbWFsIHRocmVzaG9sZCAoWW91ZGVuJ3MgSikKICAgICAgICBmcHIsIHRwciwgdGhyZXNob2xkcyA9IHJvY19jdXJ2ZSh5X3RydWUsIHlfcHJvYikKICAgICAgICBqX3Njb3JlcyA9IHRwciAtIGZwcgogICAgICAgIG9wdGltYWxfaWR4ID0gbnAuYXJnbWF4KGpfc2NvcmVzKQogICAgICAgIG1ldHJpY3NbIm9wdGltYWxfdGhyZXNob2xkIl0gPSBmbG9hdCh0aHJlc2hvbGRzW29wdGltYWxfaWR4XSkKICAgICAgICBtZXRyaWNzWyJ5b3VkZW5faiJdID0gZmxvYXQoal9zY29yZXNbb3B0aW1hbF9pZHhdKQoKICAgICAgICAjIFBvc2l0aXZlIHJhdGUKICAgICAgICBtZXRyaWNzWyJwb3NpdGl2ZV9yYXRlIl0gPSB5X3RydWUubWVhbigpCiAgICAgICAgbWV0cmljc1sicHJlZGljdGVkX3Bvc2l0aXZlX3JhdGUiXSA9IHlfcHJlZC5tZWFuKCkKICAgICAgICBtZXRyaWNzWyJuX3NhbXBsZXMiXSA9IGxlbih5X3RydWUpCgogICAgICAgIHJldHVybiBtZXRyaWNzCgogICAgZGVmIGNvbXB1dGVfbXVsdGlfaG9yaXpvbl9tZXRyaWNzKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBucC5uZGFycmF5XSwKICAgICAgICB0YXJnZXRzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICApIC0+IGRpY3Rbc3RyLCBkaWN0W3N0ciwgZmxvYXRdXToKICAgICAgICAiIiJDb21wdXRlIG1ldHJpY3MgZm9yIGFsbCBwcmVkaWN0aW9uIGhvcml6b25zLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBwcmVkaWN0aW9uczogRGljdCBvZiBob3Jpem9uX2tleSAtPiBwcm9iYWJpbGl0aWVzLgogICAgICAgICAgICB0YXJnZXRzOiBEaWN0IG9mIGhvcml6b25fa2V5IC0+IGJpbmFyeSBsYWJlbHMuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIE5lc3RlZCBkaWN0OiBob3Jpem9uX2tleSAtPiBtZXRyaWNfbmFtZSAtPiB2YWx1ZS4KICAgICAgICAiIiIKICAgICAgICBhbGxfbWV0cmljcyA9IHt9CiAgICAgICAgZm9yIGggaW4gc2VsZi5wcmVkaWN0aW9uX2hvcml6b25zOgogICAgICAgICAgICBrZXkgPSBmImNhc2NhZGVfe2h9aCIKICAgICAgICAgICAgaWYga2V5IGluIHByZWRpY3Rpb25zIGFuZCBrZXkgaW4gdGFyZ2V0czoKICAgICAgICAgICAgICAgIHlfcHJvYiA9IG5wLmFycmF5KHByZWRpY3Rpb25zW2tleV0pCiAgICAgICAgICAgICAgICB5X3RydWUgPSBucC5hcnJheSh0YXJnZXRzW2tleV0pCiAgICAgICAgICAgICAgICBtZXRyaWNzID0gc2VsZi5jb21wdXRlX2FsbF9tZXRyaWNzKHlfdHJ1ZSwgeV9wcm9iKQogICAgICAgICAgICAgICAgYWxsX21ldHJpY3Nba2V5XSA9IG1ldHJpY3MKICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAgICAgICAgIGYiICB7a2V5fTogQVVST0M9e21ldHJpY3NbJ2F1cm9jJ106LjRmfSwgIgogICAgICAgICAgICAgICAgICAgIGYiQVVQUkM9e21ldHJpY3NbJ2F1cHJjJ106LjRmfSwgIgogICAgICAgICAgICAgICAgICAgIGYiRjE9e21ldHJpY3NbJ2YxJ106LjRmfSIKICAgICAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gYWxsX21ldHJpY3MKCiAgICBkZWYgY29tcHV0ZV9sZWFkX3RpbWUoCiAgICAgICAgc2VsZiwKICAgICAgICB5X3RydWU6IG5wLm5kYXJyYXksCiAgICAgICAgeV9wcm9iOiBucC5uZGFycmF5LAogICAgICAgIHRpbWVzdGFtcHM6IG5wLm5kYXJyYXksCiAgICAgICAgY2FzY2FkZV9zdGFydF90aW1lczogbGlzdCwKICAgICAgICB0aHJlc2hvbGQ6IGZsb2F0ID0gTm9uZSwKICAgICkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJDb21wdXRlIGxlYWQgdGltZTogaG93IGZhciBpbiBhZHZhbmNlIHRoZSBtb2RlbCBkZXRlY3RzIGNhc2NhZGVzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB5X3RydWU6IEJpbmFyeSBsYWJlbHMuCiAgICAgICAgICAgIHlfcHJvYjogUHJlZGljdGVkIHByb2JhYmlsaXRpZXMuCiAgICAgICAgICAgIHRpbWVzdGFtcHM6IERhdGV0aW1lIGFycmF5IGFsaWduZWQgd2l0aCBwcmVkaWN0aW9ucy4KICAgICAgICAgICAgY2FzY2FkZV9zdGFydF90aW1lczogTGlzdCBvZiBjYXNjYWRlIHN0YXJ0IHRpbWVzdGFtcHMuCiAgICAgICAgICAgIHRocmVzaG9sZDogRGV0ZWN0aW9uIHRocmVzaG9sZC4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCB3aXRoIG1lYW4vbWVkaWFuL21pbiBsZWFkIHRpbWUgaW4gaG91cnMuCiAgICAgICAgIiIiCiAgICAgICAgaWYgdGhyZXNob2xkIGlzIE5vbmU6CiAgICAgICAgICAgIHRocmVzaG9sZCA9IHNlbGYudGhyZXNob2xkCgogICAgICAgIHlfcHJlZCA9IChucC5hcnJheSh5X3Byb2IpID49IHRocmVzaG9sZCkuYXN0eXBlKGludCkKICAgICAgICBsZWFkX3RpbWVzID0gW10KCiAgICAgICAgZm9yIGNhc2NhZGVfc3RhcnQgaW4gY2FzY2FkZV9zdGFydF90aW1lczoKICAgICAgICAgICAgIyBGaW5kIHRoZSBmaXJzdCBwcmVkaWN0aW9uIGJlZm9yZSBjYXNjYWRlIHRoYXQgcmFpc2VkIGFsYXJtCiAgICAgICAgICAgIHByZV9jYXNjYWRlID0gWwogICAgICAgICAgICAgICAgaSBmb3IgaSwgdCBpbiBlbnVtZXJhdGUodGltZXN0YW1wcykKICAgICAgICAgICAgICAgIGlmIHQgPCBjYXNjYWRlX3N0YXJ0IGFuZCB5X3ByZWRbaV0gPT0gMQogICAgICAgICAgICBdCiAgICAgICAgICAgIGlmIHByZV9jYXNjYWRlOgogICAgICAgICAgICAgICAgZmlyc3RfYWxhcm1faWR4ID0gbWluKHByZV9jYXNjYWRlLCBrZXk9bGFtYmRhIGk6IGFicygKICAgICAgICAgICAgICAgICAgICB0aW1lc3RhbXBzW2ldIC0gY2FzY2FkZV9zdGFydAogICAgICAgICAgICAgICAgKSkKICAgICAgICAgICAgICAgICMgRm9yIHRoZSBjbG9zZXN0IGFsYXJtLCBjb21wdXRlIGxlYWQgdGltZQogICAgICAgICAgICAgICAgbGFzdF9hbGFybSA9IG1heCgKICAgICAgICAgICAgICAgICAgICBpIGZvciBpIGluIHByZV9jYXNjYWRlIGlmIHRpbWVzdGFtcHNbaV0gPCBjYXNjYWRlX3N0YXJ0CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBsZWFkX3RpbWVfaG91cnMgPSAoCiAgICAgICAgICAgICAgICAgICAgY2FzY2FkZV9zdGFydCAtIHRpbWVzdGFtcHNbbGFzdF9hbGFybV0KICAgICAgICAgICAgICAgICkudG90YWxfc2Vjb25kcygpIC8gMzYwMAogICAgICAgICAgICAgICAgbGVhZF90aW1lcy5hcHBlbmQobGVhZF90aW1lX2hvdXJzKQoKICAgICAgICBpZiBub3QgbGVhZF90aW1lczoKICAgICAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgICAgICJtZWFuX2xlYWRfdGltZV9ob3VycyI6IDAsCiAgICAgICAgICAgICAgICAibWVkaWFuX2xlYWRfdGltZV9ob3VycyI6IDAsCiAgICAgICAgICAgICAgICAibWluX2xlYWRfdGltZV9ob3VycyI6IDAsCiAgICAgICAgICAgICAgICAibWF4X2xlYWRfdGltZV9ob3VycyI6IDAsCiAgICAgICAgICAgICAgICAiZGV0ZWN0aW9uX3JhdGUiOiAwLAogICAgICAgICAgICB9CgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJtZWFuX2xlYWRfdGltZV9ob3VycyI6IG5wLm1lYW4obGVhZF90aW1lcyksCiAgICAgICAgICAgICJtZWRpYW5fbGVhZF90aW1lX2hvdXJzIjogbnAubWVkaWFuKGxlYWRfdGltZXMpLAogICAgICAgICAgICAibWluX2xlYWRfdGltZV9ob3VycyI6IG5wLm1pbihsZWFkX3RpbWVzKSwKICAgICAgICAgICAgIm1heF9sZWFkX3RpbWVfaG91cnMiOiBucC5tYXgobGVhZF90aW1lcyksCiAgICAgICAgICAgICJkZXRlY3Rpb25fcmF0ZSI6IGxlbihsZWFkX3RpbWVzKSAvIGxlbihjYXNjYWRlX3N0YXJ0X3RpbWVzKSwKICAgICAgICB9CgogICAgZGVmIGZpbmRfb3B0aW1hbF90aHJlc2hvbGQoCiAgICAgICAgc2VsZiwgeV90cnVlOiBucC5uZGFycmF5LCB5X3Byb2I6IG5wLm5kYXJyYXksIG1ldHJpYzogc3RyID0gImYxIgogICAgKSAtPiBmbG9hdDoKICAgICAgICAiIiJGaW5kIG9wdGltYWwgY2xhc3NpZmljYXRpb24gdGhyZXNob2xkLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB5X3RydWU6IEJpbmFyeSBsYWJlbHMuCiAgICAgICAgICAgIHlfcHJvYjogUHJlZGljdGVkIHByb2JhYmlsaXRpZXMuCiAgICAgICAgICAgIG1ldHJpYzogTWV0cmljIHRvIG9wdGltaXplICgiZjEiLCAieW91ZGVuIiwgInByZWNpc2lvbl9yZWNhbGwiKS4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgT3B0aW1hbCB0aHJlc2hvbGQgdmFsdWUuCiAgICAgICAgIiIiCiAgICAgICAgdGhyZXNob2xkcyA9IG5wLmFyYW5nZSgwLjA1LCAwLjk1LCAwLjAxKQogICAgICAgIGJlc3Rfc2NvcmUgPSAtMQogICAgICAgIGJlc3RfdGhyZXNob2xkID0gMC41CgogICAgICAgIGZvciB0IGluIHRocmVzaG9sZHM6CiAgICAgICAgICAgIHlfcHJlZCA9ICh5X3Byb2IgPj0gdCkuYXN0eXBlKGludCkKICAgICAgICAgICAgaWYgbWV0cmljID09ICJmMSI6CiAgICAgICAgICAgICAgICBzY29yZSA9IGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCB6ZXJvX2RpdmlzaW9uPTApCiAgICAgICAgICAgIGVsaWYgbWV0cmljID09ICJ5b3VkZW4iOgogICAgICAgICAgICAgICAgdG4sIGZwLCBmbiwgdHAgPSBjb25mdXNpb25fbWF0cml4KAogICAgICAgICAgICAgICAgICAgIHlfdHJ1ZSwgeV9wcmVkLCBsYWJlbHM9WzAsIDFdCiAgICAgICAgICAgICAgICApLnJhdmVsKCkKICAgICAgICAgICAgICAgIHRwciA9IHRwIC8gKHRwICsgZm4pIGlmICh0cCArIGZuKSA+IDAgZWxzZSAwCiAgICAgICAgICAgICAgICBmcHIgPSBmcCAvIChmcCArIHRuKSBpZiAoZnAgKyB0bikgPiAwIGVsc2UgMAogICAgICAgICAgICAgICAgc2NvcmUgPSB0cHIgLSBmcHIKICAgICAgICAgICAgZWxpZiBtZXRyaWMgPT0gInByZWNpc2lvbl9yZWNhbGwiOgogICAgICAgICAgICAgICAgcCA9IHByZWNpc2lvbl9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKQogICAgICAgICAgICAgICAgciA9IHJlY2FsbF9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKQogICAgICAgICAgICAgICAgc2NvcmUgPSAyICogcCAqIHIgLyAocCArIHIpIGlmIChwICsgcikgPiAwIGVsc2UgMAogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc2NvcmUgPSBmMV9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKQoKICAgICAgICAgICAgaWYgc2NvcmUgPiBiZXN0X3Njb3JlOgogICAgICAgICAgICAgICAgYmVzdF9zY29yZSA9IHNjb3JlCiAgICAgICAgICAgICAgICBiZXN0X3RocmVzaG9sZCA9IHQKCiAgICAgICAgcmV0dXJuIGJlc3RfdGhyZXNob2xkCgogICAgZGVmIF9lbXB0eV9tZXRyaWNzKHNlbGYpIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAgICAgIiIiUmV0dXJuIGVtcHR5IG1ldHJpY3MgZGljdCB3aGVuIGNvbXB1dGF0aW9uIGlzIGltcG9zc2libGUuIiIiCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImF1cm9jIjogMC41LCAiYXVwcmMiOiAwLjAsICJmMSI6IDAuMCwgInByZWNpc2lvbiI6IDAuMCwKICAgICAgICAgICAgInJlY2FsbCI6IDAuMCwgIm1jYyI6IDAuMCwgImJyaWVyX3Njb3JlIjogMC4yNSwKICAgICAgICAgICAgInRydWVfcG9zaXRpdmVzIjogMCwgImZhbHNlX3Bvc2l0aXZlcyI6IDAsCiAgICAgICAgICAgICJ0cnVlX25lZ2F0aXZlcyI6IDAsICJmYWxzZV9uZWdhdGl2ZXMiOiAwLAogICAgICAgICAgICAic3BlY2lmaWNpdHkiOiAwLjAsICJucHYiOiAwLjAsICJvcHRpbWFsX3RocmVzaG9sZCI6IDAuNSwKICAgICAgICAgICAgInlvdWRlbl9qIjogMC4wLCAicG9zaXRpdmVfcmF0ZSI6IDAuMCwKICAgICAgICAgICAgInByZWRpY3RlZF9wb3NpdGl2ZV9yYXRlIjogMC4wLCAibl9zYW1wbGVzIjogMCwKICAgICAgICB9Cg==", "evaluation/statistical_tests.py": "IiIiClN0YXRpc3RpY2FsIHRlc3RzIGZvciBtb2RlbCBjb21wYXJpc29uLgoKSW1wbGVtZW50czoKICAtIERpZWJvbGQtTWFyaWFubyB0ZXN0IChmb3JlY2FzdCBjb21wYXJpc29uKQogIC0gTWNOZW1hcidzIHRlc3QgKGNsYXNzaWZpY2F0aW9uIGNvbXBhcmlzb24pCiAgLSBQYWlyZWQgdC10ZXN0IChtZXRyaWMgY29tcGFyaXNvbiBhY3Jvc3MgZm9sZHMpCiAgLSBXaWxjb3hvbiBzaWduZWQtcmFuayB0ZXN0IChub24tcGFyYW1ldHJpYyBwYWlyZWQgY29tcGFyaXNvbikKICAtIEJvb3RzdHJhcCBjb25maWRlbmNlIGludGVydmFscwogIC0gQm9uZmVycm9uaSBjb3JyZWN0aW9uIGZvciBtdWx0aXBsZSBjb21wYXJpc29ucwoiIiIKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNjaXB5IGltcG9ydCBzdGF0cwpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKZnJvbSBsb2d1cnUgaW1wb3J0IGxvZ2dlcgoKCmNsYXNzIFN0YXRpc3RpY2FsVGVzdFN1aXRlOgogICAgIiIiU3VpdGUgb2Ygc3RhdGlzdGljYWwgdGVzdHMgZm9yIHJpZ29yb3VzIG1vZGVsIGNvbXBhcmlzb24uIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgY29uZmlkZW5jZV9sZXZlbDogZmxvYXQgPSAwLjk1LAogICAgICAgIGJvb3RzdHJhcF9pdGVyYXRpb25zOiBpbnQgPSAxMDAwMCwKICAgICk6CiAgICAgICAgc2VsZi5jb25maWRlbmNlX2xldmVsID0gY29uZmlkZW5jZV9sZXZlbAogICAgICAgIHNlbGYuYWxwaGEgPSAxIC0gY29uZmlkZW5jZV9sZXZlbAogICAgICAgIHNlbGYuYm9vdHN0cmFwX2l0ZXJhdGlvbnMgPSBib290c3RyYXBfaXRlcmF0aW9ucwoKICAgIGRlZiBkaWVib2xkX21hcmlhbm9fdGVzdCgKICAgICAgICBzZWxmLAogICAgICAgIHlfdHJ1ZTogbnAubmRhcnJheSwKICAgICAgICBwcmVkXzE6IG5wLm5kYXJyYXksCiAgICAgICAgcHJlZF8yOiBucC5uZGFycmF5LAogICAgICAgIGxvc3NfZm46IHN0ciA9ICJzcXVhcmVkIiwKICAgICAgICBoOiBpbnQgPSAxLAogICAgKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgICIiIkRpZWJvbGQtTWFyaWFubyB0ZXN0IGZvciBjb21wYXJpbmcgZm9yZWNhc3QgYWNjdXJhY3kuCgogICAgICAgIFRlc3RzIEgwOiBFW2RfdF0gPSAwIHdoZXJlIGRfdCA9IEwoZTFfdCkgLSBMKGUyX3QpLgogICAgICAgIFJlamVjdGlvbiBtZWFucyBtb2RlbCAxIGFuZCBtb2RlbCAyIGhhdmUgc2lnbmlmaWNhbnRseSBkaWZmZXJlbnQKICAgICAgICBmb3JlY2FzdCBhY2N1cmFjeS4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgeV90cnVlOiBUcnVlIHZhbHVlcyBbbl0uCiAgICAgICAgICAgIHByZWRfMTogUHJlZGljdGlvbnMgZnJvbSBtb2RlbCAxIFtuXS4KICAgICAgICAgICAgcHJlZF8yOiBQcmVkaWN0aW9ucyBmcm9tIG1vZGVsIDIgW25dLgogICAgICAgICAgICBsb3NzX2ZuOiAic3F1YXJlZCIgb3IgImFic29sdXRlIi4KICAgICAgICAgICAgaDogRm9yZWNhc3QgaG9yaXpvbiBmb3IgSEFDIGNvcnJlY3Rpb24uCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIERpY3Qgd2l0aCB0ZXN0X3N0YXRpc3RpYywgcF92YWx1ZSwgc2lnbmlmaWNhbnQsIHByZWZlcnJlZF9tb2RlbC4KICAgICAgICAiIiIKICAgICAgICBlMSA9IHlfdHJ1ZSAtIHByZWRfMQogICAgICAgIGUyID0geV90cnVlIC0gcHJlZF8yCgogICAgICAgIGlmIGxvc3NfZm4gPT0gInNxdWFyZWQiOgogICAgICAgICAgICBkID0gZTEgKiogMiAtIGUyICoqIDIKICAgICAgICBlbGlmIGxvc3NfZm4gPT0gImFic29sdXRlIjoKICAgICAgICAgICAgZCA9IG5wLmFicyhlMSkgLSBucC5hYnMoZTIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZCA9IGUxICoqIDIgLSBlMiAqKiAyCgogICAgICAgIG4gPSBsZW4oZCkKICAgICAgICBkX21lYW4gPSBkLm1lYW4oKQoKICAgICAgICAjIEhBQyAoSGV0ZXJvc2tlZGFzdGljaXR5IGFuZCBBdXRvY29ycmVsYXRpb24gQ29uc2lzdGVudCkgdmFyaWFuY2UKICAgICAgICBnYW1tYV8wID0gbnAudmFyKGQsIGRkb2Y9MSkKICAgICAgICBnYW1tYV9zdW0gPSAwCiAgICAgICAgZm9yIGsgaW4gcmFuZ2UoMSwgaCk6CiAgICAgICAgICAgIGdhbW1hX2sgPSBucC5jb3YoZFtrOl0sIGRbOi1rXSlbMCwgMV0gaWYgbGVuKGQpID4gayBlbHNlIDAKICAgICAgICAgICAgZ2FtbWFfc3VtICs9IDIgKiBnYW1tYV9rCgogICAgICAgIHZhcl9kID0gKGdhbW1hXzAgKyBnYW1tYV9zdW0pIC8gbgoKICAgICAgICBpZiB2YXJfZCA8PSAwOgogICAgICAgICAgICByZXR1cm4gewogICAgICAgICAgICAgICAgInRlc3Rfc3RhdGlzdGljIjogMC4wLAogICAgICAgICAgICAgICAgInBfdmFsdWUiOiAxLjAsCiAgICAgICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJwcmVmZXJyZWRfbW9kZWwiOiAibmVpdGhlciIsCiAgICAgICAgICAgIH0KCiAgICAgICAgZG1fc3RhdCA9IGRfbWVhbiAvIG5wLnNxcnQodmFyX2QpCiAgICAgICAgcF92YWx1ZSA9IDIgKiAoMSAtIHN0YXRzLm5vcm0uY2RmKGFicyhkbV9zdGF0KSkpCgogICAgICAgIHByZWZlcnJlZCA9ICJuZWl0aGVyIgogICAgICAgIGlmIHBfdmFsdWUgPCBzZWxmLmFscGhhOgogICAgICAgICAgICBwcmVmZXJyZWQgPSAibW9kZWxfMSIgaWYgZF9tZWFuIDwgMCBlbHNlICJtb2RlbF8yIgoKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAidGVzdF9zdGF0aXN0aWMiOiBmbG9hdChkbV9zdGF0KSwKICAgICAgICAgICAgInBfdmFsdWUiOiBmbG9hdChwX3ZhbHVlKSwKICAgICAgICAgICAgInNpZ25pZmljYW50IjogcF92YWx1ZSA8IHNlbGYuYWxwaGEsCiAgICAgICAgICAgICJwcmVmZXJyZWRfbW9kZWwiOiBwcmVmZXJyZWQsCiAgICAgICAgICAgICJtZWFuX2xvc3NfZGlmZiI6IGZsb2F0KGRfbWVhbiksCiAgICAgICAgfQoKICAgIGRlZiBtY25lbWFyX3Rlc3QoCiAgICAgICAgc2VsZiwKICAgICAgICB5X3RydWU6IG5wLm5kYXJyYXksCiAgICAgICAgcHJlZF8xOiBucC5uZGFycmF5LAogICAgICAgIHByZWRfMjogbnAubmRhcnJheSwKICAgICAgICB0aHJlc2hvbGQ6IGZsb2F0ID0gMC41LAogICAgKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgICIiIk1jTmVtYXIncyB0ZXN0IGZvciBjb21wYXJpbmcgdHdvIGNsYXNzaWZpZXJzLgoKICAgICAgICBUZXN0cyB3aGV0aGVyIHR3byBjbGFzc2lmaWVycyBoYXZlIHRoZSBzYW1lIGVycm9yIHJhdGUuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHlfdHJ1ZTogVHJ1ZSBiaW5hcnkgbGFiZWxzLgogICAgICAgICAgICBwcmVkXzE6IFByb2JhYmlsaXRpZXMgb3IgYmluYXJ5IHByZWRpY3Rpb25zIGZyb20gbW9kZWwgMS4KICAgICAgICAgICAgcHJlZF8yOiBQcm9iYWJpbGl0aWVzIG9yIGJpbmFyeSBwcmVkaWN0aW9ucyBmcm9tIG1vZGVsIDIuCiAgICAgICAgICAgIHRocmVzaG9sZDogQ2xhc3NpZmljYXRpb24gdGhyZXNob2xkLgogICAgICAgICIiIgogICAgICAgIGMxID0gKG5wLmFycmF5KHByZWRfMSkgPj0gdGhyZXNob2xkKS5hc3R5cGUoaW50KQogICAgICAgIGMyID0gKG5wLmFycmF5KHByZWRfMikgPj0gdGhyZXNob2xkKS5hc3R5cGUoaW50KQogICAgICAgIHkgPSBucC5hcnJheSh5X3RydWUpLmFzdHlwZShpbnQpCgogICAgICAgICMgQ29ycmVjdC9pbmNvcnJlY3QgZm9yIGVhY2ggbW9kZWwKICAgICAgICBjb3JyZWN0XzEgPSAoYzEgPT0geSkKICAgICAgICBjb3JyZWN0XzIgPSAoYzIgPT0geSkKCiAgICAgICAgIyBDb250aW5nZW5jeSB0YWJsZQogICAgICAgICMgYjogbW9kZWwgMSBjb3JyZWN0LCBtb2RlbCAyIGluY29ycmVjdAogICAgICAgICMgYzogbW9kZWwgMSBpbmNvcnJlY3QsIG1vZGVsIDIgY29ycmVjdAogICAgICAgIGIgPSBucC5zdW0oY29ycmVjdF8xICYgfmNvcnJlY3RfMikKICAgICAgICBjID0gbnAuc3VtKH5jb3JyZWN0XzEgJiBjb3JyZWN0XzIpCgogICAgICAgICMgTWNOZW1hcidzIHRlc3Qgd2l0aCBjb250aW51aXR5IGNvcnJlY3Rpb24KICAgICAgICBpZiBiICsgYyA9PSAwOgogICAgICAgICAgICByZXR1cm4gewogICAgICAgICAgICAgICAgInRlc3Rfc3RhdGlzdGljIjogMC4wLAogICAgICAgICAgICAgICAgInBfdmFsdWUiOiAxLjAsCiAgICAgICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJiIjogaW50KGIpLAogICAgICAgICAgICAgICAgImMiOiBpbnQoYyksCiAgICAgICAgICAgIH0KCiAgICAgICAgY2hpMiA9IChhYnMoYiAtIGMpIC0gMSkgKiogMiAvIChiICsgYykKICAgICAgICBwX3ZhbHVlID0gMSAtIHN0YXRzLmNoaTIuY2RmKGNoaTIsIGRmPTEpCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ0ZXN0X3N0YXRpc3RpYyI6IGZsb2F0KGNoaTIpLAogICAgICAgICAgICAicF92YWx1ZSI6IGZsb2F0KHBfdmFsdWUpLAogICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBwX3ZhbHVlIDwgc2VsZi5hbHBoYSwKICAgICAgICAgICAgImIiOiBpbnQoYiksCiAgICAgICAgICAgICJjIjogaW50KGMpLAogICAgICAgICAgICAicHJlZmVycmVkX21vZGVsIjogKAogICAgICAgICAgICAgICAgIm1vZGVsXzEiIGlmIGIgPiBjIGVsc2UgIm1vZGVsXzIiIGlmIGMgPiBiIGVsc2UgIm5laXRoZXIiCiAgICAgICAgICAgICksCiAgICAgICAgfQoKICAgIGRlZiBwYWlyZWRfdHRlc3QoCiAgICAgICAgc2VsZiwKICAgICAgICBzY29yZXNfMTogbnAubmRhcnJheSwKICAgICAgICBzY29yZXNfMjogbnAubmRhcnJheSwKICAgICkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJQYWlyZWQgdC10ZXN0IGNvbXBhcmluZyBtZXRyaWMgc2NvcmVzIGFjcm9zcyBDViBmb2xkcy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgc2NvcmVzXzE6IE1ldHJpYyB2YWx1ZXMgZnJvbSBtb2RlbCAxIGFjcm9zcyBmb2xkcy4KICAgICAgICAgICAgc2NvcmVzXzI6IE1ldHJpYyB2YWx1ZXMgZnJvbSBtb2RlbCAyIGFjcm9zcyBmb2xkcy4KICAgICAgICAiIiIKICAgICAgICBzY29yZXNfMSA9IG5wLmFycmF5KHNjb3Jlc18xKQogICAgICAgIHNjb3Jlc18yID0gbnAuYXJyYXkoc2NvcmVzXzIpCgogICAgICAgIGlmIGxlbihzY29yZXNfMSkgPCAyOgogICAgICAgICAgICByZXR1cm4gewogICAgICAgICAgICAgICAgInRlc3Rfc3RhdGlzdGljIjogMC4wLAogICAgICAgICAgICAgICAgInBfdmFsdWUiOiAxLjAsCiAgICAgICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJtZWFuX2RpZmYiOiAwLjAsCiAgICAgICAgICAgIH0KCiAgICAgICAgdF9zdGF0LCBwX3ZhbHVlID0gc3RhdHMudHRlc3RfcmVsKHNjb3Jlc18xLCBzY29yZXNfMikKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInRlc3Rfc3RhdGlzdGljIjogZmxvYXQodF9zdGF0KSwKICAgICAgICAgICAgInBfdmFsdWUiOiBmbG9hdChwX3ZhbHVlKSwKICAgICAgICAgICAgInNpZ25pZmljYW50IjogcF92YWx1ZSA8IHNlbGYuYWxwaGEsCiAgICAgICAgICAgICJtZWFuX2RpZmYiOiBmbG9hdChucC5tZWFuKHNjb3Jlc18xIC0gc2NvcmVzXzIpKSwKICAgICAgICAgICAgInN0ZF9kaWZmIjogZmxvYXQobnAuc3RkKHNjb3Jlc18xIC0gc2NvcmVzXzIpKSwKICAgICAgICAgICAgInByZWZlcnJlZF9tb2RlbCI6ICgKICAgICAgICAgICAgICAgICJtb2RlbF8xIiBpZiBucC5tZWFuKHNjb3Jlc18xKSA+IG5wLm1lYW4oc2NvcmVzXzIpCiAgICAgICAgICAgICAgICBlbHNlICJtb2RlbF8yIgogICAgICAgICAgICApLAogICAgICAgIH0KCiAgICBkZWYgd2lsY294b25fc2lnbmVkX3JhbmsoCiAgICAgICAgc2VsZiwKICAgICAgICBzY29yZXNfMTogbnAubmRhcnJheSwKICAgICAgICBzY29yZXNfMjogbnAubmRhcnJheSwKICAgICkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJXaWxjb3hvbiBzaWduZWQtcmFuayB0ZXN0IChub24tcGFyYW1ldHJpYyBwYWlyZWQgY29tcGFyaXNvbikuCgogICAgICAgIE1vcmUgcm9idXN0IHRoYW4gcGFpcmVkIHQtdGVzdCB3aGVuIG5vcm1hbGl0eSBhc3N1bXB0aW9uIGlzIHZpb2xhdGVkLgogICAgICAgICIiIgogICAgICAgIHNjb3Jlc18xID0gbnAuYXJyYXkoc2NvcmVzXzEpCiAgICAgICAgc2NvcmVzXzIgPSBucC5hcnJheShzY29yZXNfMikKCiAgICAgICAgaWYgbGVuKHNjb3Jlc18xKSA8IDY6CiAgICAgICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICAgICAidGVzdF9zdGF0aXN0aWMiOiAwLjAsCiAgICAgICAgICAgICAgICAicF92YWx1ZSI6IDEuMCwKICAgICAgICAgICAgICAgICJzaWduaWZpY2FudCI6IEZhbHNlLAogICAgICAgICAgICAgICAgIm5vdGUiOiAiVG9vIGZldyBzYW1wbGVzIGZvciBXaWxjb3hvbiB0ZXN0IiwKICAgICAgICAgICAgfQoKICAgICAgICBkaWZmID0gc2NvcmVzXzEgLSBzY29yZXNfMgogICAgICAgIGlmIG5wLmFsbChkaWZmID09IDApOgogICAgICAgICAgICByZXR1cm4gewogICAgICAgICAgICAgICAgInRlc3Rfc3RhdGlzdGljIjogMC4wLAogICAgICAgICAgICAgICAgInBfdmFsdWUiOiAxLjAsCiAgICAgICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBGYWxzZSwKICAgICAgICAgICAgfQoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHN0YXQsIHBfdmFsdWUgPSBzdGF0cy53aWxjb3hvbihzY29yZXNfMSwgc2NvcmVzXzIpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICAgICAidGVzdF9zdGF0aXN0aWMiOiAwLjAsCiAgICAgICAgICAgICAgICAicF92YWx1ZSI6IDEuMCwKICAgICAgICAgICAgICAgICJzaWduaWZpY2FudCI6IEZhbHNlLAogICAgICAgICAgICB9CgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ0ZXN0X3N0YXRpc3RpYyI6IGZsb2F0KHN0YXQpLAogICAgICAgICAgICAicF92YWx1ZSI6IGZsb2F0KHBfdmFsdWUpLAogICAgICAgICAgICAic2lnbmlmaWNhbnQiOiBwX3ZhbHVlIDwgc2VsZi5hbHBoYSwKICAgICAgICAgICAgIm1lZGlhbl9kaWZmIjogZmxvYXQobnAubWVkaWFuKGRpZmYpKSwKICAgICAgICB9CgogICAgZGVmIGJvb3RzdHJhcF9jb25maWRlbmNlX2ludGVydmFsKAogICAgICAgIHNlbGYsCiAgICAgICAgeV90cnVlOiBucC5uZGFycmF5LAogICAgICAgIHlfcHJvYjogbnAubmRhcnJheSwKICAgICAgICBtZXRyaWNfZm46IGNhbGxhYmxlLAogICAgICAgIG5fYm9vdHN0cmFwOiBpbnQgPSBOb25lLAogICAgKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgICIiIkJvb3RzdHJhcCBjb25maWRlbmNlIGludGVydmFsIGZvciBhbnkgbWV0cmljLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB5X3RydWU6IFRydWUgbGFiZWxzLgogICAgICAgICAgICB5X3Byb2I6IFByZWRpY3RlZCBwcm9iYWJpbGl0aWVzLgogICAgICAgICAgICBtZXRyaWNfZm46IEZ1bmN0aW9uKHlfdHJ1ZSwgeV9wcm9iKSAtPiBmbG9hdC4KICAgICAgICAgICAgbl9ib290c3RyYXA6IE51bWJlciBvZiBib290c3RyYXAgc2FtcGxlcy4KICAgICAgICAiIiIKICAgICAgICBpZiBuX2Jvb3RzdHJhcCBpcyBOb25lOgogICAgICAgICAgICBuX2Jvb3RzdHJhcCA9IHNlbGYuYm9vdHN0cmFwX2l0ZXJhdGlvbnMKCiAgICAgICAgcm5nID0gbnAucmFuZG9tLlJhbmRvbVN0YXRlKDQyKQogICAgICAgIG4gPSBsZW4oeV90cnVlKQogICAgICAgIHNjb3JlcyA9IFtdCgogICAgICAgIGZvciBfIGluIHJhbmdlKG5fYm9vdHN0cmFwKToKICAgICAgICAgICAgaWR4ID0gcm5nLnJhbmRpbnQoMCwgbiwgc2l6ZT1uKQogICAgICAgICAgICBib290X3RydWUgPSB5X3RydWVbaWR4XQogICAgICAgICAgICBib290X3Byb2IgPSB5X3Byb2JbaWR4XQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzY29yZSA9IG1ldHJpY19mbihib290X3RydWUsIGJvb3RfcHJvYikKICAgICAgICAgICAgICAgIGlmIG5wLmlzZmluaXRlKHNjb3JlKToKICAgICAgICAgICAgICAgICAgICBzY29yZXMuYXBwZW5kKHNjb3JlKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgaWYgbm90IHNjb3JlczoKICAgICAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgICAgICJtZWFuIjogMC4wLCAic3RkIjogMC4wLAogICAgICAgICAgICAgICAgImNpX2xvd2VyIjogMC4wLCAiY2lfdXBwZXIiOiAwLjAsCiAgICAgICAgICAgIH0KCiAgICAgICAgc2NvcmVzID0gbnAuYXJyYXkoc2NvcmVzKQogICAgICAgIGNpX2xvd2VyID0gbnAucGVyY2VudGlsZShzY29yZXMsICgxIC0gc2VsZi5jb25maWRlbmNlX2xldmVsKSAvIDIgKiAxMDApCiAgICAgICAgY2lfdXBwZXIgPSBucC5wZXJjZW50aWxlKHNjb3JlcywgKDEgKyBzZWxmLmNvbmZpZGVuY2VfbGV2ZWwpIC8gMiAqIDEwMCkKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm1lYW4iOiBmbG9hdChzY29yZXMubWVhbigpKSwKICAgICAgICAgICAgInN0ZCI6IGZsb2F0KHNjb3Jlcy5zdGQoKSksCiAgICAgICAgICAgICJjaV9sb3dlciI6IGZsb2F0KGNpX2xvd2VyKSwKICAgICAgICAgICAgImNpX3VwcGVyIjogZmxvYXQoY2lfdXBwZXIpLAogICAgICAgIH0KCiAgICBkZWYgYm9uZmVycm9uaV9jb3JyZWN0aW9uKAogICAgICAgIHNlbGYsIHBfdmFsdWVzOiBsaXN0W2Zsb2F0XSwgYWxwaGE6IGZsb2F0ID0gTm9uZQogICAgKSAtPiBkaWN0W3N0ciwgbGlzdF06CiAgICAgICAgIiIiQXBwbHkgQm9uZmVycm9uaSBjb3JyZWN0aW9uIGZvciBtdWx0aXBsZSBjb21wYXJpc29ucy4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcF92YWx1ZXM6IExpc3Qgb2YgcC12YWx1ZXMgZnJvbSBtdWx0aXBsZSB0ZXN0cy4KICAgICAgICAgICAgYWxwaGE6IFNpZ25pZmljYW5jZSBsZXZlbC4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCB3aXRoIGNvcnJlY3RlZF9hbHBoYSwgYWRqdXN0ZWRfcF92YWx1ZXMsIHNpZ25pZmljYW50IGZsYWdzLgogICAgICAgICIiIgogICAgICAgIGlmIGFscGhhIGlzIE5vbmU6CiAgICAgICAgICAgIGFscGhhID0gc2VsZi5hbHBoYQoKICAgICAgICBtID0gbGVuKHBfdmFsdWVzKQogICAgICAgIGNvcnJlY3RlZF9hbHBoYSA9IGFscGhhIC8gbQogICAgICAgIGFkanVzdGVkID0gW21pbihwICogbSwgMS4wKSBmb3IgcCBpbiBwX3ZhbHVlc10KCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImNvcnJlY3RlZF9hbHBoYSI6IGNvcnJlY3RlZF9hbHBoYSwKICAgICAgICAgICAgImFkanVzdGVkX3BfdmFsdWVzIjogYWRqdXN0ZWQsCiAgICAgICAgICAgICJzaWduaWZpY2FudCI6IFtwIDwgY29ycmVjdGVkX2FscGhhIGZvciBwIGluIHBfdmFsdWVzXSwKICAgICAgICAgICAgIm51bV9jb21wYXJpc29ucyI6IG0sCiAgICAgICAgfQoKICAgIGRlZiBydW5fZnVsbF9jb21wYXJpc29uKAogICAgICAgIHNlbGYsCiAgICAgICAgeV90cnVlOiBucC5uZGFycmF5LAogICAgICAgIG1vZGVsX3ByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICAgICAgZm9sZF9zY29yZXM6IE9wdGlvbmFsW2RpY3Rbc3RyLCBsaXN0XV0gPSBOb25lLAogICAgICAgIHJlZmVyZW5jZV9tb2RlbDogc3RyID0gIlRHTiIsCiAgICApIC0+IGRpY3Q6CiAgICAgICAgIiIiUnVuIGFsbCBzdGF0aXN0aWNhbCB0ZXN0cyBjb21wYXJpbmcgbW9kZWxzIGFnYWluc3QgcmVmZXJlbmNlLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB5X3RydWU6IEdyb3VuZCB0cnV0aC4KICAgICAgICAgICAgbW9kZWxfcHJlZGljdGlvbnM6IERpY3Qgb2YgbW9kZWxfbmFtZSAtPiBwcmVkaWN0aW9ucy4KICAgICAgICAgICAgZm9sZF9zY29yZXM6IE9wdGlvbmFsIGRpY3Qgb2YgbW9kZWxfbmFtZSAtPiBsaXN0IG9mIGZvbGQgc2NvcmVzLgogICAgICAgICAgICByZWZlcmVuY2VfbW9kZWw6IE5hbWUgb2YgdGhlIHJlZmVyZW5jZSBtb2RlbC4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgQ29tcHJlaGVuc2l2ZSBjb21wYXJpc29uIHJlc3VsdHMuCiAgICAgICAgIiIiCiAgICAgICAgcmVzdWx0cyA9IHt9CiAgICAgICAgcmVmX3ByZWRzID0gbW9kZWxfcHJlZGljdGlvbnMuZ2V0KHJlZmVyZW5jZV9tb2RlbCkKCiAgICAgICAgaWYgcmVmX3ByZWRzIGlzIE5vbmU6CiAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIlJlZmVyZW5jZSBtb2RlbCAne3JlZmVyZW5jZV9tb2RlbH0nIG5vdCBmb3VuZCIpCiAgICAgICAgICAgIHJldHVybiByZXN1bHRzCgogICAgICAgIGFsbF9wX3ZhbHVlcyA9IFtdCgogICAgICAgIGZvciBtb2RlbF9uYW1lLCBwcmVkcyBpbiBtb2RlbF9wcmVkaWN0aW9ucy5pdGVtcygpOgogICAgICAgICAgICBpZiBtb2RlbF9uYW1lID09IHJlZmVyZW5jZV9tb2RlbDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBjb21wYXJpc29uID0geyJ2cyI6IGYie3JlZmVyZW5jZV9tb2RlbH0gdnMge21vZGVsX25hbWV9In0KCiAgICAgICAgICAgICMgRGllYm9sZC1NYXJpYW5vCiAgICAgICAgICAgIGNvbXBhcmlzb25bImRpZWJvbGRfbWFyaWFubyJdID0gc2VsZi5kaWVib2xkX21hcmlhbm9fdGVzdCgKICAgICAgICAgICAgICAgIHlfdHJ1ZSwgcmVmX3ByZWRzLCBwcmVkcwogICAgICAgICAgICApCiAgICAgICAgICAgIGFsbF9wX3ZhbHVlcy5hcHBlbmQoY29tcGFyaXNvblsiZGllYm9sZF9tYXJpYW5vIl1bInBfdmFsdWUiXSkKCiAgICAgICAgICAgICMgTWNOZW1hcidzCiAgICAgICAgICAgIGNvbXBhcmlzb25bIm1jbmVtYXIiXSA9IHNlbGYubWNuZW1hcl90ZXN0KAogICAgICAgICAgICAgICAgeV90cnVlLCByZWZfcHJlZHMsIHByZWRzCiAgICAgICAgICAgICkKICAgICAgICAgICAgYWxsX3BfdmFsdWVzLmFwcGVuZChjb21wYXJpc29uWyJtY25lbWFyIl1bInBfdmFsdWUiXSkKCiAgICAgICAgICAgICMgUGFpcmVkIHQtdGVzdCBvbiBmb2xkIHNjb3JlcwogICAgICAgICAgICBpZiBmb2xkX3Njb3JlcyBhbmQgbW9kZWxfbmFtZSBpbiBmb2xkX3Njb3JlczoKICAgICAgICAgICAgICAgIHJlZl9zY29yZXMgPSBmb2xkX3Njb3Jlcy5nZXQocmVmZXJlbmNlX21vZGVsLCBbXSkKICAgICAgICAgICAgICAgIG1vZF9zY29yZXMgPSBmb2xkX3Njb3Jlcy5nZXQobW9kZWxfbmFtZSwgW10pCiAgICAgICAgICAgICAgICBpZiByZWZfc2NvcmVzIGFuZCBtb2Rfc2NvcmVzOgogICAgICAgICAgICAgICAgICAgIGNvbXBhcmlzb25bInBhaXJlZF90dGVzdCJdID0gc2VsZi5wYWlyZWRfdHRlc3QoCiAgICAgICAgICAgICAgICAgICAgICAgIG5wLmFycmF5KHJlZl9zY29yZXMpLCBucC5hcnJheShtb2Rfc2NvcmVzKQogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICAgICBjb21wYXJpc29uWyJ3aWxjb3hvbiJdID0gc2VsZi53aWxjb3hvbl9zaWduZWRfcmFuaygKICAgICAgICAgICAgICAgICAgICAgICAgbnAuYXJyYXkocmVmX3Njb3JlcyksIG5wLmFycmF5KG1vZF9zY29yZXMpCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgIGFsbF9wX3ZhbHVlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbXBhcmlzb25bInBhaXJlZF90dGVzdCJdWyJwX3ZhbHVlIl0KICAgICAgICAgICAgICAgICAgICApCgogICAgICAgICAgICByZXN1bHRzW21vZGVsX25hbWVdID0gY29tcGFyaXNvbgoKICAgICAgICAjIEJvbmZlcnJvbmkgY29ycmVjdGlvbgogICAgICAgIGlmIGFsbF9wX3ZhbHVlczoKICAgICAgICAgICAgcmVzdWx0c1siYm9uZmVycm9uaSJdID0gc2VsZi5ib25mZXJyb25pX2NvcnJlY3Rpb24oYWxsX3BfdmFsdWVzKQoKICAgICAgICByZXR1cm4gcmVzdWx0cwo=", "evaluation/visualization.py": "IiIiClB1YmxpY2F0aW9uLXF1YWxpdHkgdmlzdWFsaXphdGlvbiBmb3IgSUVFRSBUQ1NTIHBhcGVyLgoKR2VuZXJhdGVzIGFsbCBmaWd1cmVzIG5lZWRlZCBmb3IgdGhlIHBhcGVyOgogIDEuIFJPQyBhbmQgUFIgY3VydmVzIChtdWx0aS1ob3Jpem9uKQogIDIuIFRyYWluaW5nIGxvc3MgY3VydmVzCiAgMy4gTW9kZWwgY29tcGFyaXNvbiBiYXIgY2hhcnRzCiAgNC4gQWJsYXRpb24gaGVhdG1hcAogIDUuIENhc2Ugc3R1ZHkgdGltZWxpbmUgKGNhc2NhZGUgZXZlbnRzKQogIDYuIENvbXBvc2FiaWxpdHkgZ3JhcGggdmlzdWFsaXphdGlvbgogIDcuIEZlYXR1cmUgaW1wb3J0YW5jZSBhbmFseXNpcwogIDguIENvbmZpZGVuY2UgaW50ZXJ2YWwgcGxvdHMKIiIiCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgbWF0cGxvdGxpYi5ncmlkc3BlYyBhcyBncmlkc3BlYwpmcm9tIG1hdHBsb3RsaWIucGF0Y2hlcyBpbXBvcnQgRmFuY3lCYm94UGF0Y2gKaW1wb3J0IHNlYWJvcm4gYXMgc25zCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfY3VydmUsIHByZWNpc2lvbl9yZWNhbGxfY3VydmUsIGF1Ywpmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCgoKIyBJRUVFLXN0eWxlIGZvcm1hdHRpbmcKcGx0LnJjUGFyYW1zLnVwZGF0ZSh7CiAgICAiZm9udC5mYW1pbHkiOiAic2VyaWYiLAogICAgImZvbnQuc2VyaWYiOiBbIlRpbWVzIE5ldyBSb21hbiIsICJUaW1lcyIsICJEZWphVnUgU2VyaWYiXSwKICAgICJmb250LnNpemUiOiAxMCwKICAgICJheGVzLmxhYmVsc2l6ZSI6IDExLAogICAgImF4ZXMudGl0bGVzaXplIjogMTIsCiAgICAieHRpY2subGFiZWxzaXplIjogOSwKICAgICJ5dGljay5sYWJlbHNpemUiOiA5LAogICAgImxlZ2VuZC5mb250c2l6ZSI6IDksCiAgICAiZmlndXJlLmRwaSI6IDMwMCwKICAgICJzYXZlZmlnLmRwaSI6IDMwMCwKICAgICJzYXZlZmlnLmJib3giOiAidGlnaHQiLAogICAgInNhdmVmaWcucGFkX2luY2hlcyI6IDAuMDUsCiAgICAiYXhlcy5ncmlkIjogVHJ1ZSwKICAgICJncmlkLmFscGhhIjogMC4zLAogICAgImF4ZXMuc3BpbmVzLnRvcCI6IEZhbHNlLAogICAgImF4ZXMuc3BpbmVzLnJpZ2h0IjogRmFsc2UsCn0pCgpDT0xPUlMgPSB7CiAgICAiVEdOIjogIiMyMTk2RjMiLAogICAgIlN0YXRpYyBHTk4iOiAiI0ZGOTgwMCIsCiAgICAiTFNUTSI6ICIjNENBRjUwIiwKICAgICJYR0Jvb3N0IjogIiNGNDQzMzYiLAogICAgIlNJUiI6ICIjOUMyN0IwIiwKICAgICJDZW50cmFsaXR5IjogIiM3OTU1NDgiLAp9CgpIT1JJWk9OX0NPTE9SUyA9IHsKICAgICIxZCI6ICIjRTUzOTM1IiwKICAgICIzZCI6ICIjRkI4QzAwIiwKICAgICI3ZCI6ICIjNDNBMDQ3IiwKICAgICIzMGQiOiAiIzFFODhFNSIsCn0KCgpjbGFzcyBQYXBlclZpc3VhbGl6ZXI6CiAgICAiIiJHZW5lcmF0ZXMgYWxsIHB1YmxpY2F0aW9uIGZpZ3VyZXMgZm9yIHRoZSBJRUVFIFRDU1MgcGFwZXIuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG91dHB1dF9kaXI6IHN0ciA9ICJvdXRwdXRzL2ZpZ3VyZXMiKToKICAgICAgICBzZWxmLm91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICAgICAgc2VsZi5vdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBkZWYgcGxvdF9yb2NfY3VydmVzKAogICAgICAgIHNlbGYsCiAgICAgICAgeV90cnVlOiBucC5uZGFycmF5LAogICAgICAgIG1vZGVsX3ByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICAgICAgaG9yaXpvbl9sYWJlbDogc3RyID0gIjI0aCIsCiAgICAgICAgZmlsZW5hbWU6IHN0ciA9ICJyb2NfY3VydmVzLnBkZiIsCiAgICApOgogICAgICAgICIiIlBsb3QgUk9DIGN1cnZlcyBjb21wYXJpbmcgYWxsIG1vZGVscy4iIiIKICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDQuNSwgNCkpCgogICAgICAgIGZvciBtb2RlbF9uYW1lLCB5X3Byb2IgaW4gbW9kZWxfcHJlZGljdGlvbnMuaXRlbXMoKToKICAgICAgICAgICAgZnByLCB0cHIsIF8gPSByb2NfY3VydmUoeV90cnVlLCB5X3Byb2IpCiAgICAgICAgICAgIGF1cm9jID0gYXVjKGZwciwgdHByKQogICAgICAgICAgICBjb2xvciA9IENPTE9SUy5nZXQobW9kZWxfbmFtZSwgIiM2NjY2NjYiKQogICAgICAgICAgICBsaW5ld2lkdGggPSAyLjUgaWYgbW9kZWxfbmFtZSA9PSAiVEdOIiBlbHNlIDEuNQogICAgICAgICAgICBsaW5lc3R5bGUgPSAiLSIgaWYgbW9kZWxfbmFtZSA9PSAiVEdOIiBlbHNlICItLSIKICAgICAgICAgICAgYXgucGxvdCgKICAgICAgICAgICAgICAgIGZwciwgdHByLAogICAgICAgICAgICAgICAgbGFiZWw9ZiJ7bW9kZWxfbmFtZX0gKEFVQz17YXVyb2M6LjNmfSkiLAogICAgICAgICAgICAgICAgY29sb3I9Y29sb3IsCiAgICAgICAgICAgICAgICBsaW5ld2lkdGg9bGluZXdpZHRoLAogICAgICAgICAgICAgICAgbGluZXN0eWxlPWxpbmVzdHlsZSwKICAgICAgICAgICAgKQoKICAgICAgICBheC5wbG90KFswLCAxXSwgWzAsIDFdLCAiay0tIiwgYWxwaGE9MC4zLCBsaW5ld2lkdGg9MC44KQogICAgICAgIGF4LnNldF94bGFiZWwoIkZhbHNlIFBvc2l0aXZlIFJhdGUiKQogICAgICAgIGF4LnNldF95bGFiZWwoIlRydWUgUG9zaXRpdmUgUmF0ZSIpCiAgICAgICAgYXguc2V0X3RpdGxlKGYiUk9DIEN1cnZlcyDigJQge2hvcml6b25fbGFiZWx9IEhvcml6b24iKQogICAgICAgIGF4LmxlZ2VuZChsb2M9Imxvd2VyIHJpZ2h0IiwgZnJhbWVhbHBoYT0wLjkpCiAgICAgICAgYXguc2V0X3hsaW0oWzAsIDFdKQogICAgICAgIGF4LnNldF95bGltKFswLCAxLjAyXSkKCiAgICAgICAgZmlnLnNhdmVmaWcoc2VsZi5vdXRwdXRfZGlyIC8gZmlsZW5hbWUpCiAgICAgICAgcGx0LmNsb3NlKGZpZykKICAgICAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIFJPQyBjdXJ2ZXMgdG8ge2ZpbGVuYW1lfSIpCgogICAgZGVmIHBsb3RfcHJfY3VydmVzKAogICAgICAgIHNlbGYsCiAgICAgICAgeV90cnVlOiBucC5uZGFycmF5LAogICAgICAgIG1vZGVsX3ByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICAgICAgaG9yaXpvbl9sYWJlbDogc3RyID0gIjI0aCIsCiAgICAgICAgZmlsZW5hbWU6IHN0ciA9ICJwcl9jdXJ2ZXMucGRmIiwKICAgICk6CiAgICAgICAgIiIiUGxvdCBQcmVjaXNpb24tUmVjYWxsIGN1cnZlcyBjb21wYXJpbmcgYWxsIG1vZGVscy4iIiIKICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDQuNSwgNCkpCgogICAgICAgIGJhc2VsaW5lX3ByID0geV90cnVlLm1lYW4oKQoKICAgICAgICBmb3IgbW9kZWxfbmFtZSwgeV9wcm9iIGluIG1vZGVsX3ByZWRpY3Rpb25zLml0ZW1zKCk6CiAgICAgICAgICAgIHByZWMsIHJlYywgXyA9IHByZWNpc2lvbl9yZWNhbGxfY3VydmUoeV90cnVlLCB5X3Byb2IpCiAgICAgICAgICAgIGFwID0gYXVjKHJlYywgcHJlYykKICAgICAgICAgICAgY29sb3IgPSBDT0xPUlMuZ2V0KG1vZGVsX25hbWUsICIjNjY2NjY2IikKICAgICAgICAgICAgbGluZXdpZHRoID0gMi41IGlmIG1vZGVsX25hbWUgPT0gIlRHTiIgZWxzZSAxLjUKICAgICAgICAgICAgbGluZXN0eWxlID0gIi0iIGlmIG1vZGVsX25hbWUgPT0gIlRHTiIgZWxzZSAiLS0iCiAgICAgICAgICAgIGF4LnBsb3QoCiAgICAgICAgICAgICAgICByZWMsIHByZWMsCiAgICAgICAgICAgICAgICBsYWJlbD1mInttb2RlbF9uYW1lfSAoQVA9e2FwOi4zZn0pIiwKICAgICAgICAgICAgICAgIGNvbG9yPWNvbG9yLAogICAgICAgICAgICAgICAgbGluZXdpZHRoPWxpbmV3aWR0aCwKICAgICAgICAgICAgICAgIGxpbmVzdHlsZT1saW5lc3R5bGUsCiAgICAgICAgICAgICkKCiAgICAgICAgYXguYXhobGluZShiYXNlbGluZV9wciwgY29sb3I9ImdyYXkiLCBsaW5lc3R5bGU9IjoiLCBhbHBoYT0wLjUsCiAgICAgICAgICAgICAgICAgICAgbGFiZWw9ZiJSYW5kb20gKHtiYXNlbGluZV9wcjouM2Z9KSIpCiAgICAgICAgYXguc2V0X3hsYWJlbCgiUmVjYWxsIikKICAgICAgICBheC5zZXRfeWxhYmVsKCJQcmVjaXNpb24iKQogICAgICAgIGF4LnNldF90aXRsZShmIlByZWNpc2lvbi1SZWNhbGwgQ3VydmVzIOKAlCB7aG9yaXpvbl9sYWJlbH0gSG9yaXpvbiIpCiAgICAgICAgYXgubGVnZW5kKGxvYz0idXBwZXIgcmlnaHQiLCBmcmFtZWFscGhhPTAuOSkKICAgICAgICBheC5zZXRfeGxpbShbMCwgMV0pCiAgICAgICAgYXguc2V0X3lsaW0oWzAsIDEuMDJdKQoKICAgICAgICBmaWcuc2F2ZWZpZyhzZWxmLm91dHB1dF9kaXIgLyBmaWxlbmFtZSkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgUFIgY3VydmVzIHRvIHtmaWxlbmFtZX0iKQoKICAgIGRlZiBwbG90X211bHRpX2hvcml6b25fY29tcGFyaXNvbigKICAgICAgICBzZWxmLAogICAgICAgIGFsbF9tZXRyaWNzOiBkaWN0W3N0ciwgZGljdFtzdHIsIGRpY3RdXSwKICAgICAgICBtZXRyaWNfbmFtZTogc3RyID0gImF1cm9jIiwKICAgICAgICBmaWxlbmFtZTogc3RyID0gIm11bHRpX2hvcml6b25fY29tcGFyaXNvbi5wZGYiLAogICAgKToKICAgICAgICAiIiJCYXIgY2hhcnQgY29tcGFyaW5nIG1vZGVscyBhY3Jvc3MgYWxsIHByZWRpY3Rpb24gaG9yaXpvbnMuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGFsbF9tZXRyaWNzOiBtb2RlbF9uYW1lIC0+IGhvcml6b25fa2V5IC0+IG1ldHJpY3NfZGljdC4KICAgICAgICAiIiIKICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDcsIDQpKQoKICAgICAgICBob3Jpem9ucyA9IFsiY2FzY2FkZV8yNGgiLCAiY2FzY2FkZV83MmgiLCAiY2FzY2FkZV8xNjhoIiwgImNhc2NhZGVfNzIwaCJdCiAgICAgICAgaG9yaXpvbl9sYWJlbHMgPSBbIjFkIiwgIjNkIiwgIjdkIiwgIjMwZCJdCiAgICAgICAgbW9kZWxzID0gbGlzdChhbGxfbWV0cmljcy5rZXlzKCkpCiAgICAgICAgbl9tb2RlbHMgPSBsZW4obW9kZWxzKQogICAgICAgIG5faG9yaXpvbnMgPSBsZW4oaG9yaXpvbnMpCgogICAgICAgIGJhcl93aWR0aCA9IDAuOCAvIG5fbW9kZWxzCiAgICAgICAgeCA9IG5wLmFyYW5nZShuX2hvcml6b25zKQoKICAgICAgICBmb3IgaSwgbW9kZWwgaW4gZW51bWVyYXRlKG1vZGVscyk6CiAgICAgICAgICAgIHZhbHVlcyA9IFtdCiAgICAgICAgICAgIGZvciBoIGluIGhvcml6b25zOgogICAgICAgICAgICAgICAgbSA9IGFsbF9tZXRyaWNzLmdldChtb2RlbCwge30pLmdldChoLCB7fSkKICAgICAgICAgICAgICAgIHZhbHVlcy5hcHBlbmQobS5nZXQobWV0cmljX25hbWUsIDApKQoKICAgICAgICAgICAgY29sb3IgPSBDT0xPUlMuZ2V0KG1vZGVsLCAiIzY2NjY2NiIpCiAgICAgICAgICAgIG9mZnNldCA9IChpIC0gbl9tb2RlbHMgLyAyICsgMC41KSAqIGJhcl93aWR0aAogICAgICAgICAgICBiYXJzID0gYXguYmFyKAogICAgICAgICAgICAgICAgeCArIG9mZnNldCwgdmFsdWVzLCBiYXJfd2lkdGgsCiAgICAgICAgICAgICAgICBsYWJlbD1tb2RlbCwgY29sb3I9Y29sb3IsIGFscGhhPTAuODUsCiAgICAgICAgICAgICAgICBlZGdlY29sb3I9IndoaXRlIiwgbGluZXdpZHRoPTAuNSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyBBZGQgdmFsdWUgbGFiZWxzIG9uIHRvcAogICAgICAgICAgICBmb3IgYmFyLCB2YWwgaW4gemlwKGJhcnMsIHZhbHVlcyk6CiAgICAgICAgICAgICAgICBpZiB2YWwgPiAwOgogICAgICAgICAgICAgICAgICAgIGF4LnRleHQoCiAgICAgICAgICAgICAgICAgICAgICAgIGJhci5nZXRfeCgpICsgYmFyLmdldF93aWR0aCgpIC8gMiwgYmFyLmdldF9oZWlnaHQoKSwKICAgICAgICAgICAgICAgICAgICAgICAgZiJ7dmFsOi4zZn0iLCBoYT0iY2VudGVyIiwgdmE9ImJvdHRvbSIsIGZvbnRzaXplPTcsCiAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICBheC5zZXRfeHRpY2tzKHgpCiAgICAgICAgYXguc2V0X3h0aWNrbGFiZWxzKGhvcml6b25fbGFiZWxzKQogICAgICAgIGF4LnNldF94bGFiZWwoIlByZWRpY3Rpb24gSG9yaXpvbiIpCiAgICAgICAgYXguc2V0X3lsYWJlbChtZXRyaWNfbmFtZS51cHBlcigpKQogICAgICAgIGF4LnNldF90aXRsZShmIk1vZGVsIENvbXBhcmlzb24gQWNyb3NzIFByZWRpY3Rpb24gSG9yaXpvbnMiKQogICAgICAgIGF4LmxlZ2VuZChsb2M9InVwcGVyIHJpZ2h0IiwgbmNvbD0yLCBmcmFtZWFscGhhPTAuOSkKICAgICAgICBheC5zZXRfeWxpbShbMCwgMS4xXSkKCiAgICAgICAgZmlnLnNhdmVmaWcoc2VsZi5vdXRwdXRfZGlyIC8gZmlsZW5hbWUpCiAgICAgICAgcGx0LmNsb3NlKGZpZykKICAgICAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIG11bHRpLWhvcml6b24gY29tcGFyaXNvbiB0byB7ZmlsZW5hbWV9IikKCiAgICBkZWYgcGxvdF90cmFpbmluZ19jdXJ2ZXMoCiAgICAgICAgc2VsZiwKICAgICAgICB0cmFpbl9sb3NzZXM6IGxpc3RbZmxvYXRdLAogICAgICAgIHZhbF9sb3NzZXM6IGxpc3RbZmxvYXRdLAogICAgICAgIGZpbGVuYW1lOiBzdHIgPSAidHJhaW5pbmdfY3VydmVzLnBkZiIsCiAgICApOgogICAgICAgICIiIlBsb3QgdHJhaW5pbmcgYW5kIHZhbGlkYXRpb24gbG9zcyBjdXJ2ZXMuIiIiCiAgICAgICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg1LCAzLjUpKQoKICAgICAgICBlcG9jaHMgPSByYW5nZSgxLCBsZW4odHJhaW5fbG9zc2VzKSArIDEpCiAgICAgICAgYXgucGxvdChlcG9jaHMsIHRyYWluX2xvc3NlcywgbGFiZWw9IlRyYWluIiwgY29sb3I9Q09MT1JTWyJUR04iXSwKICAgICAgICAgICAgICAgIGxpbmV3aWR0aD0xLjUpCiAgICAgICAgYXgucGxvdChlcG9jaHMsIHZhbF9sb3NzZXMsIGxhYmVsPSJWYWxpZGF0aW9uIiwgY29sb3I9Q09MT1JTWyJYR0Jvb3N0Il0sCiAgICAgICAgICAgICAgICBsaW5ld2lkdGg9MS41KQoKICAgICAgICBiZXN0X2Vwb2NoID0gbnAuYXJnbWluKHZhbF9sb3NzZXMpICsgMQogICAgICAgIGF4LmF4dmxpbmUoYmVzdF9lcG9jaCwgY29sb3I9ImdyYXkiLCBsaW5lc3R5bGU9Ii0tIiwgYWxwaGE9MC41LAogICAgICAgICAgICAgICAgICAgIGxhYmVsPWYiQmVzdCBlcG9jaCAoe2Jlc3RfZXBvY2h9KSIpCgogICAgICAgIGF4LnNldF94bGFiZWwoIkVwb2NoIikKICAgICAgICBheC5zZXRfeWxhYmVsKCJMb3NzIikKICAgICAgICBheC5zZXRfdGl0bGUoIlRyYWluaW5nIENvbnZlcmdlbmNlIikKICAgICAgICBheC5sZWdlbmQoZnJhbWVhbHBoYT0wLjkpCgogICAgICAgIGZpZy5zYXZlZmlnKHNlbGYub3V0cHV0X2RpciAvIGZpbGVuYW1lKQogICAgICAgIHBsdC5jbG9zZShmaWcpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJTYXZlZCB0cmFpbmluZyBjdXJ2ZXMgdG8ge2ZpbGVuYW1lfSIpCgogICAgZGVmIHBsb3RfYWJsYXRpb25faGVhdG1hcCgKICAgICAgICBzZWxmLAogICAgICAgIGFibGF0aW9uX3Jlc3VsdHM6IGRpY3QsCiAgICAgICAgZmlsZW5hbWU6IHN0ciA9ICJhYmxhdGlvbl9oZWF0bWFwLnBkZiIsCiAgICApOgogICAgICAgICIiIkhlYXRtYXAgc2hvd2luZyBBVVJPQyBkcm9wIGZvciBlYWNoIGFibGF0aW9uLiIiIgogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNiwgNSkpCgogICAgICAgIGhvcml6b25zID0gWyJjYXNjYWRlXzI0aCIsICJjYXNjYWRlXzcyaCIsICJjYXNjYWRlXzE2OGgiLCAiY2FzY2FkZV83MjBoIl0KICAgICAgICBob3Jpem9uX2xhYmVscyA9IFsiMWQiLCAiM2QiLCAiN2QiLCAiMzBkIl0KCiAgICAgICAgY29uZmlncyA9IFtdCiAgICAgICAgZGF0YSA9IFtdCgogICAgICAgIGZvciBjb25maWdfbmFtZSwgY29uZmlnX2RhdGEgaW4gYWJsYXRpb25fcmVzdWx0cy5pdGVtcygpOgogICAgICAgICAgICBpZiBjb25maWdfbmFtZSA9PSAiZnVsbF9tb2RlbCI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGNvbmZpZ19kYXRhLCBkaWN0KSBhbmQgImRlbHRhIiBpbiBjb25maWdfZGF0YToKICAgICAgICAgICAgICAgIHJvdyA9IFtdCiAgICAgICAgICAgICAgICBmb3IgaGsgaW4gaG9yaXpvbnM6CiAgICAgICAgICAgICAgICAgICAgZCA9IGNvbmZpZ19kYXRhLmdldCgiZGVsdGEiLCB7fSkuZ2V0KGhrLCB7fSkKICAgICAgICAgICAgICAgICAgICByb3cuYXBwZW5kKGQuZ2V0KCJhdXJvYyIsIDApKQogICAgICAgICAgICAgICAgZGF0YS5hcHBlbmQocm93KQogICAgICAgICAgICAgICAgY29uZmlncy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgY29uZmlnX25hbWUucmVwbGFjZSgid2l0aG91dF8iLCAidy9vICIpCiAgICAgICAgICAgICAgICAgICAgLnJlcGxhY2UoIl8iLCAiICIpLnRpdGxlKCkKICAgICAgICAgICAgICAgICkKCiAgICAgICAgaWYgbm90IGRhdGE6CiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKCJObyBhYmxhdGlvbiBkYXRhIHRvIHBsb3QiKQogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgZGF0YSA9IG5wLmFycmF5KGRhdGEpCiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUoZGF0YSwgaW5kZXg9Y29uZmlncywgY29sdW1ucz1ob3Jpem9uX2xhYmVscykKCiAgICAgICAgc25zLmhlYXRtYXAoCiAgICAgICAgICAgIGRmLCBhbm5vdD1UcnVlLCBmbXQ9Ii40ZiIsIGNtYXA9IlJkWWxCdV9yIiwKICAgICAgICAgICAgY2VudGVyPTAsIGF4PWF4LCBsaW5ld2lkdGhzPTAuNSwKICAgICAgICAgICAgY2Jhcl9rd3M9eyJsYWJlbCI6ICJBVVJPQyBEcm9wIChwb3NpdGl2ZSA9IGNvbXBvbmVudCBoZWxwcykifSwKICAgICAgICApCiAgICAgICAgYXguc2V0X3RpdGxlKCJGZWF0dXJlL0NvbXBvbmVudCBBYmxhdGlvbiBJbXBhY3QiKQogICAgICAgIGF4LnNldF94bGFiZWwoIlByZWRpY3Rpb24gSG9yaXpvbiIpCgogICAgICAgIGZpZy5zYXZlZmlnKHNlbGYub3V0cHV0X2RpciAvIGZpbGVuYW1lKQogICAgICAgIHBsdC5jbG9zZShmaWcpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJTYXZlZCBhYmxhdGlvbiBoZWF0bWFwIHRvIHtmaWxlbmFtZX0iKQoKICAgIGRlZiBwbG90X2Nhc2NhZGVfY2FzZV9zdHVkeSgKICAgICAgICBzZWxmLAogICAgICAgIGRhdGVzOiBucC5uZGFycmF5LAogICAgICAgIHR2bF92YWx1ZXM6IG5wLm5kYXJyYXksCiAgICAgICAgcHJlZGljdGlvbnM6IG5wLm5kYXJyYXksCiAgICAgICAgZXZlbnRfbmFtZTogc3RyLAogICAgICAgIGV2ZW50X3N0YXJ0OiBzdHIsCiAgICAgICAgZXZlbnRfZW5kOiBzdHIsCiAgICAgICAgZmlsZW5hbWU6IHN0ciA9ICJjYXNlX3N0dWR5LnBkZiIsCiAgICApOgogICAgICAgICIiIlRpbWVsaW5lIHBsb3QgZm9yIGEgc3BlY2lmaWMgY2FzY2FkZSBldmVudCBjYXNlIHN0dWR5LiIiIgogICAgICAgIGZpZywgKGF4MSwgYXgyKSA9IHBsdC5zdWJwbG90cygKICAgICAgICAgICAgMiwgMSwgZmlnc2l6ZT0oNywgNSksIHNoYXJleD1UcnVlLAogICAgICAgICAgICBncmlkc3BlY19rdz17ImhlaWdodF9yYXRpb3MiOiBbMiwgMV19LAogICAgICAgICkKCiAgICAgICAgIyBUVkwgdGltZWxpbmUKICAgICAgICBheDEucGxvdChkYXRlcywgdHZsX3ZhbHVlcyAvIDFlOSwgY29sb3I9Q09MT1JTWyJUR04iXSwgbGluZXdpZHRoPTEuNSkKICAgICAgICBheDEuYXh2c3BhbigKICAgICAgICAgICAgcGQuVGltZXN0YW1wKGV2ZW50X3N0YXJ0KSwgcGQuVGltZXN0YW1wKGV2ZW50X2VuZCksCiAgICAgICAgICAgIGFscGhhPTAuMTUsIGNvbG9yPSJyZWQiLCBsYWJlbD0iQ2FzY2FkZSBFdmVudCIsCiAgICAgICAgKQogICAgICAgIGF4MS5zZXRfeWxhYmVsKCJUb3RhbCBUVkwgKCQgQmlsbGlvbikiKQogICAgICAgIGF4MS5zZXRfdGl0bGUoZiJDYXNlIFN0dWR5OiB7ZXZlbnRfbmFtZX0iKQogICAgICAgIGF4MS5sZWdlbmQobG9jPSJ1cHBlciByaWdodCIpCgogICAgICAgICMgUHJlZGljdGlvbiB0aW1lbGluZQogICAgICAgIGF4Mi5wbG90KGRhdGVzLCBwcmVkaWN0aW9ucywgY29sb3I9Q09MT1JTWyJYR0Jvb3N0Il0sIGxpbmV3aWR0aD0xLjUpCiAgICAgICAgYXgyLmF4aGxpbmUoMC41LCBjb2xvcj0iZ3JheSIsIGxpbmVzdHlsZT0iLS0iLCBhbHBoYT0wLjUpCiAgICAgICAgYXgyLmF4dnNwYW4oCiAgICAgICAgICAgIHBkLlRpbWVzdGFtcChldmVudF9zdGFydCksIHBkLlRpbWVzdGFtcChldmVudF9lbmQpLAogICAgICAgICAgICBhbHBoYT0wLjE1LCBjb2xvcj0icmVkIiwKICAgICAgICApCiAgICAgICAgYXgyLnNldF95bGFiZWwoIkNhc2NhZGUgUHJvYmFiaWxpdHkiKQogICAgICAgIGF4Mi5zZXRfeGxhYmVsKCJEYXRlIikKICAgICAgICBheDIuc2V0X3lsaW0oWzAsIDFdKQoKICAgICAgICBmaWcuc2F2ZWZpZyhzZWxmLm91dHB1dF9kaXIgLyBmaWxlbmFtZSkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgY2FzZSBzdHVkeSB0byB7ZmlsZW5hbWV9IikKCiAgICBkZWYgcGxvdF9jb25maWRlbmNlX2ludGVydmFscygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsX25hbWVzOiBsaXN0W3N0cl0sCiAgICAgICAgbWVhbnM6IGxpc3RbZmxvYXRdLAogICAgICAgIGNpX2xvd2VyczogbGlzdFtmbG9hdF0sCiAgICAgICAgY2lfdXBwZXJzOiBsaXN0W2Zsb2F0XSwKICAgICAgICBtZXRyaWNfbmFtZTogc3RyID0gIkFVUk9DIiwKICAgICAgICBmaWxlbmFtZTogc3RyID0gImNvbmZpZGVuY2VfaW50ZXJ2YWxzLnBkZiIsCiAgICApOgogICAgICAgICIiIkZvcmVzdCBwbG90IG9mIG1vZGVsIHBlcmZvcm1hbmNlIHdpdGggY29uZmlkZW5jZSBpbnRlcnZhbHMuIiIiCiAgICAgICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg1LCAzLjUpKQoKICAgICAgICB5X3BvcyA9IG5wLmFyYW5nZShsZW4obW9kZWxfbmFtZXMpKQogICAgICAgIGVycm9ycyA9IG5wLmFycmF5KAogICAgICAgICAgICBbW20gLSBsIGZvciBtLCBsIGluIHppcChtZWFucywgY2lfbG93ZXJzKV0sCiAgICAgICAgICAgICBbdSAtIG0gZm9yIG0sIHUgaW4gemlwKG1lYW5zLCBjaV91cHBlcnMpXV0KICAgICAgICApCgogICAgICAgIGNvbG9ycyA9IFtDT0xPUlMuZ2V0KG0sICIjNjY2NjY2IikgZm9yIG0gaW4gbW9kZWxfbmFtZXNdCiAgICAgICAgYXguYmFyaCh5X3BvcywgbWVhbnMsIHhlcnI9ZXJyb3JzLCBjb2xvcj1jb2xvcnMsIGFscGhhPTAuOCwKICAgICAgICAgICAgICAgIGNhcHNpemU9MywgZWRnZWNvbG9yPSJ3aGl0ZSIsIGxpbmV3aWR0aD0wLjUpCgogICAgICAgIGF4LnNldF95dGlja3MoeV9wb3MpCiAgICAgICAgYXguc2V0X3l0aWNrbGFiZWxzKG1vZGVsX25hbWVzKQogICAgICAgIGF4LnNldF94bGFiZWwobWV0cmljX25hbWUpCiAgICAgICAgYXguc2V0X3RpdGxlKGYie21ldHJpY19uYW1lfSB3aXRoIDk1JSBCb290c3RyYXAgQ0lzIikKICAgICAgICBheC5pbnZlcnRfeWF4aXMoKQoKICAgICAgICBmaWcuc2F2ZWZpZyhzZWxmLm91dHB1dF9kaXIgLyBmaWxlbmFtZSkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgQ0kgcGxvdCB0byB7ZmlsZW5hbWV9IikKCiAgICBkZWYgcGxvdF9jb21wb3NhYmlsaXR5X2dyYXBoKAogICAgICAgIHNlbGYsCiAgICAgICAgYWRqYWNlbmN5OiBucC5uZGFycmF5LAogICAgICAgIHByb3RvY29sX25hbWVzOiBsaXN0W3N0cl0sCiAgICAgICAgcmlza19zY29yZXM6IE9wdGlvbmFsW25wLm5kYXJyYXldID0gTm9uZSwKICAgICAgICBmaWxlbmFtZTogc3RyID0gImNvbXBvc2FiaWxpdHlfZ3JhcGgucGRmIiwKICAgICk6CiAgICAgICAgIiIiVmlzdWFsaXplIHRoZSBEZUZpIGNvbXBvc2FiaWxpdHkgZ3JhcGguIiIiCiAgICAgICAgaW1wb3J0IG5ldHdvcmt4IGFzIG54CgogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNywgNikpCiAgICAgICAgRyA9IG54LmZyb21fbnVtcHlfYXJyYXkoYWRqYWNlbmN5KQoKICAgICAgICAjIExheW91dAogICAgICAgIHBvcyA9IG54LnNwcmluZ19sYXlvdXQoRywgaz0yLCBzZWVkPTQyKQoKICAgICAgICAjIE5vZGUgc2l6ZXMgYW5kIGNvbG9ycyBiYXNlZCBvbiByaXNrCiAgICAgICAgaWYgcmlza19zY29yZXMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG5vZGVfY29sb3JzID0gcmlza19zY29yZXMKICAgICAgICAgICAgY21hcCA9IHBsdC5jbS5SZFlsR25fcgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG5vZGVfY29sb3JzID0gWzAuNV0gKiBsZW4ocHJvdG9jb2xfbmFtZXMpCiAgICAgICAgICAgIGNtYXAgPSBwbHQuY20uQmx1ZXMKCiAgICAgICAgIyBEcmF3IGVkZ2VzCiAgICAgICAgZWRnZV93aWR0aHMgPSBbYWRqYWNlbmN5W3VdW3ZdICogMiBmb3IgdSwgdiBpbiBHLmVkZ2VzKCldCiAgICAgICAgbnguZHJhd19uZXR3b3JreF9lZGdlcygKICAgICAgICAgICAgRywgcG9zLCBheD1heCwgd2lkdGg9ZWRnZV93aWR0aHMsIGFscGhhPTAuMywKICAgICAgICAgICAgZWRnZV9jb2xvcj0iZ3JheSIsCiAgICAgICAgKQoKICAgICAgICAjIERyYXcgbm9kZXMKICAgICAgICBub2RlcyA9IG54LmRyYXdfbmV0d29ya3hfbm9kZXMoCiAgICAgICAgICAgIEcsIHBvcywgYXg9YXgsIG5vZGVfc2l6ZT04MDAsCiAgICAgICAgICAgIG5vZGVfY29sb3I9bm9kZV9jb2xvcnMsIGNtYXA9Y21hcCwKICAgICAgICAgICAgdm1pbj0wLCB2bWF4PTEsIGVkZ2Vjb2xvcnM9ImJsYWNrIiwgbGluZXdpZHRocz0wLjUsCiAgICAgICAgKQoKICAgICAgICAjIExhYmVscwogICAgICAgIGxhYmVscyA9IHtpOiBuYW1lLnJlcGxhY2UoIi0iLCAiXG4iKSBmb3IgaSwgbmFtZSBpbiBlbnVtZXJhdGUocHJvdG9jb2xfbmFtZXMpfQogICAgICAgIG54LmRyYXdfbmV0d29ya3hfbGFiZWxzKEcsIHBvcywgbGFiZWxzLCBheD1heCwgZm9udF9zaXplPTcpCgogICAgICAgIGlmIHJpc2tfc2NvcmVzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBwbHQuY29sb3JiYXIobm9kZXMsIGF4PWF4LCBsYWJlbD0iUmlzayBTY29yZSIsIHNocmluaz0wLjgpCgogICAgICAgIGF4LnNldF90aXRsZSgiRGVGaSBDb21wb3NhYmlsaXR5IEdyYXBoIikKICAgICAgICBheC5heGlzKCJvZmYiKQoKICAgICAgICBmaWcuc2F2ZWZpZyhzZWxmLm91dHB1dF9kaXIgLyBmaWxlbmFtZSkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgZ3JhcGggdmlzdWFsaXphdGlvbiB0byB7ZmlsZW5hbWV9IikKCiAgICBkZWYgZ2VuZXJhdGVfYWxsX3BhcGVyX2ZpZ3VyZXMoCiAgICAgICAgc2VsZiwKICAgICAgICByZXN1bHRzOiBkaWN0LAogICAgKToKICAgICAgICAiIiJHZW5lcmF0ZSBhbGwgZmlndXJlcyBmb3IgdGhlIHBhcGVyIGZyb20gZXhwZXJpbWVudCByZXN1bHRzLiIiIgogICAgICAgIGxvZ2dlci5pbmZvKCJHZW5lcmF0aW5nIGFsbCBwYXBlciBmaWd1cmVzIikKCiAgICAgICAgaWYgInRyYWluaW5nX2hpc3RvcnkiIGluIHJlc3VsdHM6CiAgICAgICAgICAgIHNlbGYucGxvdF90cmFpbmluZ19jdXJ2ZXMoCiAgICAgICAgICAgICAgICByZXN1bHRzWyJ0cmFpbmluZ19oaXN0b3J5Il1bInRyYWluX2xvc3NlcyJdLAogICAgICAgICAgICAgICAgcmVzdWx0c1sidHJhaW5pbmdfaGlzdG9yeSJdWyJ2YWxfbG9zc2VzIl0sCiAgICAgICAgICAgICkKCiAgICAgICAgaWYgIm1vZGVsX2NvbXBhcmlzb24iIGluIHJlc3VsdHM6CiAgICAgICAgICAgIHNlbGYucGxvdF9tdWx0aV9ob3Jpem9uX2NvbXBhcmlzb24oCiAgICAgICAgICAgICAgICByZXN1bHRzWyJtb2RlbF9jb21wYXJpc29uIl0sCiAgICAgICAgICAgICAgICBtZXRyaWNfbmFtZT0iYXVyb2MiLAogICAgICAgICAgICApCgogICAgICAgIGlmICJhYmxhdGlvbiIgaW4gcmVzdWx0czoKICAgICAgICAgICAgZm9yIGFibGF0aW9uX3R5cGUsIGFibF9yZXN1bHRzIGluIHJlc3VsdHNbImFibGF0aW9uIl0uaXRlbXMoKToKICAgICAgICAgICAgICAgIHNlbGYucGxvdF9hYmxhdGlvbl9oZWF0bWFwKAogICAgICAgICAgICAgICAgICAgIGFibF9yZXN1bHRzLAogICAgICAgICAgICAgICAgICAgIGZpbGVuYW1lPWYiYWJsYXRpb25fe2FibGF0aW9uX3R5cGV9LnBkZiIsCiAgICAgICAgICAgICAgICApCgogICAgICAgIGxvZ2dlci5pbmZvKGYiQWxsIGZpZ3VyZXMgc2F2ZWQgdG8ge3NlbGYub3V0cHV0X2Rpcn0iKQo=", "evaluation/ablation.py": "IiIiCkFibGF0aW9uIHN0dWR5IGZyYW1ld29yay4KClN5c3RlbWF0aWNhbGx5IHJlbW92ZXMgY29tcG9uZW50cyB0byBxdWFudGlmeSB0aGVpciBjb250cmlidXRpb246CiAgMS4gRmVhdHVyZSBncm91cCBhYmxhdGlvbiAocmVtb3ZlIGVhY2ggZmVhdHVyZSBncm91cCkKICAyLiBFZGdlIHR5cGUgYWJsYXRpb24gKHJlbW92ZSBlYWNoIGVkZ2UgdHlwZSBmcm9tIGdyYXBoKQogIDMuIFRlbXBvcmFsIHdpbmRvdyBhYmxhdGlvbiAodmFyeSBsb29rYmFjayBwZXJpb2QpCiAgNC4gTW9kZWwgY29tcG9uZW50IGFibGF0aW9uIChyZW1vdmUgbWVtb3J5LCBhdHRlbnRpb24sIGV0Yy4pCiAgNS4gUHJlZGljdGlvbiBob3Jpem9uIGFuYWx5c2lzCiIiIgoKaW1wb3J0IGNvcHkKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCgpmcm9tIGV2YWx1YXRpb24ubWV0cmljcyBpbXBvcnQgTWV0cmljc0NhbGN1bGF0b3IKCgpjbGFzcyBBYmxhdGlvblN0dWR5OgogICAgIiIiU3lzdGVtYXRpYyBhYmxhdGlvbiBzdHVkeSBmcmFtZXdvcmsgZm9yIHRoZSBUR04gbW9kZWwuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgYmFzZV9tb2RlbDogdG9yY2gubm4uTW9kdWxlLAogICAgICAgIGNvbmZpZzogZGljdCwKICAgICAgICBtZXRyaWNzX2NhbGN1bGF0b3I6IE1ldHJpY3NDYWxjdWxhdG9yLAogICAgICAgIG91dHB1dF9kaXI6IHN0ciA9ICJvdXRwdXRzL3Jlc3VsdHMiLAogICAgKToKICAgICAgICBzZWxmLmJhc2VfbW9kZWwgPSBiYXNlX21vZGVsCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLm1ldHJpY3MgPSBtZXRyaWNzX2NhbGN1bGF0b3IKICAgICAgICBzZWxmLm91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICAgICAgc2VsZi5vdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxmLnJlc3VsdHMgPSB7fQoKICAgIGRlZiBydW5fZmVhdHVyZV9ncm91cF9hYmxhdGlvbigKICAgICAgICBzZWxmLAogICAgICAgIHRyYWluX2ZuOiBjYWxsYWJsZSwKICAgICAgICBldmFsdWF0ZV9mbjogY2FsbGFibGUsCiAgICAgICAgZmVhdHVyZV9ncm91cHM6IGxpc3Rbc3RyXSwKICAgICAgICBiYXNlX21ldHJpY3M6IGRpY3QsCiAgICApIC0+IGRpY3Q6CiAgICAgICAgIiIiQWJsYXRlIGVhY2ggZmVhdHVyZSBncm91cCBieSB6ZXJvaW5nIGl0IG91dC4KCiAgICAgICAgRm9yIGVhY2ggZ3JvdXAsIHJldHJhaW4gdGhlIG1vZGVsIHdpdGggdGhhdCBmZWF0dXJlIGdyb3VwIHplcm9lZAogICAgICAgIGFuZCBtZWFzdXJlIHRoZSBwZXJmb3JtYW5jZSBkcm9wLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0cmFpbl9mbjogRnVuY3Rpb24oZXhjbHVkZV9ncm91cHMpIC0+IHRyYWluZWRfbW9kZWwuCiAgICAgICAgICAgIGV2YWx1YXRlX2ZuOiBGdW5jdGlvbihtb2RlbCkgLT4gbWV0cmljc19kaWN0LgogICAgICAgICAgICBmZWF0dXJlX2dyb3VwczogTGlzdCBvZiBmZWF0dXJlIGdyb3VwIG5hbWVzLgogICAgICAgICAgICBiYXNlX21ldHJpY3M6IEZ1bGwgbW9kZWwgbWV0cmljcyBmb3IgY29tcGFyaXNvbi4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCBvZiBncm91cF9uYW1lIC0+IHttZXRyaWNzLCBkZWx0YV9mcm9tX2Z1bGx9LgogICAgICAgICIiIgogICAgICAgIGxvZ2dlci5pbmZvKCI9IiAqIDYwKQogICAgICAgIGxvZ2dlci5pbmZvKCJBQkxBVElPTjogRmVhdHVyZSBHcm91cHMiKQogICAgICAgIGxvZ2dlci5pbmZvKCI9IiAqIDYwKQoKICAgICAgICByZXN1bHRzID0geyJmdWxsX21vZGVsIjogYmFzZV9tZXRyaWNzfQoKICAgICAgICBmb3IgZ3JvdXAgaW4gZmVhdHVyZV9ncm91cHM6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiQWJsYXRpbmcgZmVhdHVyZSBncm91cDoge2dyb3VwfSIpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG1vZGVsID0gdHJhaW5fZm4oZXhjbHVkZV9ncm91cHM9W2dyb3VwXSkKICAgICAgICAgICAgICAgIGdyb3VwX21ldHJpY3MgPSBldmFsdWF0ZV9mbihtb2RlbCkKCiAgICAgICAgICAgICAgICAjIENvbXB1dGUgZGVsdGEKICAgICAgICAgICAgICAgIGRlbHRhID0ge30KICAgICAgICAgICAgICAgIGZvciBob3Jpem9uX2tleSBpbiBiYXNlX21ldHJpY3M6CiAgICAgICAgICAgICAgICAgICAgaWYgaG9yaXpvbl9rZXkgaW4gZ3JvdXBfbWV0cmljczoKICAgICAgICAgICAgICAgICAgICAgICAgZGVsdGFbaG9yaXpvbl9rZXldID0ge30KICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG1ldHJpY19uYW1lIGluIGJhc2VfbWV0cmljc1tob3Jpem9uX2tleV06CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBiYXNlX3ZhbCA9IGJhc2VfbWV0cmljc1tob3Jpem9uX2tleV1bbWV0cmljX25hbWVdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBhYmxhdGVkX3ZhbCA9IGdyb3VwX21ldHJpY3NbaG9yaXpvbl9rZXldW21ldHJpY19uYW1lXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShiYXNlX3ZhbCwgKGludCwgZmxvYXQpKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZWx0YVtob3Jpem9uX2tleV1bbWV0cmljX25hbWVdID0gKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBiYXNlX3ZhbCAtIGFibGF0ZWRfdmFsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgICAgIHJlc3VsdHNbZiJ3aXRob3V0X3tncm91cH0iXSA9IHsKICAgICAgICAgICAgICAgICAgICAibWV0cmljcyI6IGdyb3VwX21ldHJpY3MsCiAgICAgICAgICAgICAgICAgICAgImRlbHRhIjogZGVsdGEsCiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgIyBMb2cgaW1wYWN0CiAgICAgICAgICAgICAgICBmb3IgaGsgaW4gZGVsdGE6CiAgICAgICAgICAgICAgICAgICAgaWYgImF1cm9jIiBpbiBkZWx0YVtoa106CiAgICAgICAgICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZiIgIHtncm91cH0gLT4ge2hrfSBBVVJPQyBkcm9wOiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmIntkZWx0YVtoa11bJ2F1cm9jJ106LjRmfSIKICAgICAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICAgICAgbG9nZ2VyLmVycm9yKGYiRmVhdHVyZSBhYmxhdGlvbiBmYWlsZWQgZm9yIHtncm91cH06IHtlfSIpCiAgICAgICAgICAgICAgICByZXN1bHRzW2Yid2l0aG91dF97Z3JvdXB9Il0gPSB7ImVycm9yIjogc3RyKGUpfQoKICAgICAgICBzZWxmLnJlc3VsdHNbImZlYXR1cmVfZ3JvdXBfYWJsYXRpb24iXSA9IHJlc3VsdHMKICAgICAgICByZXR1cm4gcmVzdWx0cwoKICAgIGRlZiBydW5fZWRnZV90eXBlX2FibGF0aW9uKAogICAgICAgIHNlbGYsCiAgICAgICAgdHJhaW5fZm46IGNhbGxhYmxlLAogICAgICAgIGV2YWx1YXRlX2ZuOiBjYWxsYWJsZSwKICAgICAgICBlZGdlX3R5cGVzOiBsaXN0W3N0cl0sCiAgICAgICAgYmFzZV9tZXRyaWNzOiBkaWN0LAogICAgKSAtPiBkaWN0OgogICAgICAgICIiIkFibGF0ZSBlYWNoIGVkZ2UgdHlwZSBmcm9tIHRoZSBjb21wb3NhYmlsaXR5IGdyYXBoLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0cmFpbl9mbjogRnVuY3Rpb24oZXhjbHVkZV9lZGdlcykgLT4gdHJhaW5lZF9tb2RlbC4KICAgICAgICAgICAgZXZhbHVhdGVfZm46IEZ1bmN0aW9uKG1vZGVsKSAtPiBtZXRyaWNzX2RpY3QuCiAgICAgICAgICAgIGVkZ2VfdHlwZXM6IExpc3Qgb2YgZWRnZSB0eXBlIG5hbWVzLgogICAgICAgICAgICBiYXNlX21ldHJpY3M6IEZ1bGwgbW9kZWwgbWV0cmljcy4KICAgICAgICAiIiIKICAgICAgICBsb2dnZXIuaW5mbygiPSIgKiA2MCkKICAgICAgICBsb2dnZXIuaW5mbygiQUJMQVRJT046IEVkZ2UgVHlwZXMiKQogICAgICAgIGxvZ2dlci5pbmZvKCI9IiAqIDYwKQoKICAgICAgICByZXN1bHRzID0geyJmdWxsX21vZGVsIjogYmFzZV9tZXRyaWNzfQoKICAgICAgICBmb3IgZXR5cGUgaW4gZWRnZV90eXBlczoKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiJBYmxhdGluZyBlZGdlIHR5cGU6IHtldHlwZX0iKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBtb2RlbCA9IHRyYWluX2ZuKGV4Y2x1ZGVfZWRnZXM9W2V0eXBlXSkKICAgICAgICAgICAgICAgIGV0eXBlX21ldHJpY3MgPSBldmFsdWF0ZV9mbihtb2RlbCkKCiAgICAgICAgICAgICAgICBkZWx0YSA9IHt9CiAgICAgICAgICAgICAgICBmb3IgaGsgaW4gYmFzZV9tZXRyaWNzOgogICAgICAgICAgICAgICAgICAgIGlmIGhrIGluIGV0eXBlX21ldHJpY3M6CiAgICAgICAgICAgICAgICAgICAgICAgIGRlbHRhW2hrXSA9IHt9CiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBtayBpbiBiYXNlX21ldHJpY3NbaGtdOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgYnYgPSBiYXNlX21ldHJpY3NbaGtdW21rXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgYXYgPSBldHlwZV9tZXRyaWNzW2hrXVtta10KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UoYnYsIChpbnQsIGZsb2F0KSk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGVsdGFbaGtdW21rXSA9IGJ2IC0gYXYKCiAgICAgICAgICAgICAgICByZXN1bHRzW2Yid2l0aG91dF97ZXR5cGV9Il0gPSB7CiAgICAgICAgICAgICAgICAgICAgIm1ldHJpY3MiOiBldHlwZV9tZXRyaWNzLAogICAgICAgICAgICAgICAgICAgICJkZWx0YSI6IGRlbHRhLAogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICAgICAgbG9nZ2VyLmVycm9yKGYiRWRnZSBhYmxhdGlvbiBmYWlsZWQgZm9yIHtldHlwZX06IHtlfSIpCiAgICAgICAgICAgICAgICByZXN1bHRzW2Yid2l0aG91dF97ZXR5cGV9Il0gPSB7ImVycm9yIjogc3RyKGUpfQoKICAgICAgICBzZWxmLnJlc3VsdHNbImVkZ2VfdHlwZV9hYmxhdGlvbiJdID0gcmVzdWx0cwogICAgICAgIHJldHVybiByZXN1bHRzCgogICAgZGVmIHJ1bl90ZW1wb3JhbF93aW5kb3dfYWJsYXRpb24oCiAgICAgICAgc2VsZiwKICAgICAgICB0cmFpbl9mbjogY2FsbGFibGUsCiAgICAgICAgZXZhbHVhdGVfZm46IGNhbGxhYmxlLAogICAgICAgIHdpbmRvd3M6IGxpc3RbaW50XSwKICAgICAgICBiYXNlX3dpbmRvdzogaW50ID0gMzAsCiAgICAgICAgYmFzZV9tZXRyaWNzOiBkaWN0ID0gTm9uZSwKICAgICkgLT4gZGljdDoKICAgICAgICAiIiJUZXN0IGRpZmZlcmVudCB0ZW1wb3JhbCBsb29rYmFjayB3aW5kb3dzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0cmFpbl9mbjogRnVuY3Rpb24od2luZG93KSAtPiB0cmFpbmVkX21vZGVsLgogICAgICAgICAgICBldmFsdWF0ZV9mbjogRnVuY3Rpb24obW9kZWwpIC0+IG1ldHJpY3NfZGljdC4KICAgICAgICAgICAgd2luZG93czogTGlzdCBvZiB3aW5kb3cgc2l6ZXMgdG8gdGVzdC4KICAgICAgICAgICAgYmFzZV93aW5kb3c6IERlZmF1bHQgd2luZG93IHNpemUuCiAgICAgICAgICAgIGJhc2VfbWV0cmljczogTWV0cmljcyBmb3IgdGhlIGJhc2Ugd2luZG93LgogICAgICAgICIiIgogICAgICAgIGxvZ2dlci5pbmZvKCI9IiAqIDYwKQogICAgICAgIGxvZ2dlci5pbmZvKCJBQkxBVElPTjogVGVtcG9yYWwgV2luZG93cyIpCiAgICAgICAgbG9nZ2VyLmluZm8oIj0iICogNjApCgogICAgICAgIHJlc3VsdHMgPSB7fQoKICAgICAgICBmb3Igd2luZG93IGluIHdpbmRvd3M6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiVGVzdGluZyB0ZW1wb3JhbCB3aW5kb3c6IHt3aW5kb3d9IGRheXMiKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBtb2RlbCA9IHRyYWluX2ZuKHdpbmRvdz13aW5kb3cpCiAgICAgICAgICAgICAgICB3aW5kb3dfbWV0cmljcyA9IGV2YWx1YXRlX2ZuKG1vZGVsKQogICAgICAgICAgICAgICAgcmVzdWx0c1tmIndpbmRvd197d2luZG93fWQiXSA9IHdpbmRvd19tZXRyaWNzCgogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJXaW5kb3cgYWJsYXRpb24gZmFpbGVkIGZvciB7d2luZG93fToge2V9IikKICAgICAgICAgICAgICAgIHJlc3VsdHNbZiJ3aW5kb3dfe3dpbmRvd31kIl0gPSB7ImVycm9yIjogc3RyKGUpfQoKICAgICAgICBpZiBiYXNlX21ldHJpY3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlc3VsdHNbZiJ3aW5kb3dfe2Jhc2Vfd2luZG93fWRfYmFzZSJdID0gYmFzZV9tZXRyaWNzCgogICAgICAgIHNlbGYucmVzdWx0c1sidGVtcG9yYWxfd2luZG93X2FibGF0aW9uIl0gPSByZXN1bHRzCiAgICAgICAgcmV0dXJuIHJlc3VsdHMKCiAgICBkZWYgcnVuX21vZGVsX2NvbXBvbmVudF9hYmxhdGlvbigKICAgICAgICBzZWxmLAogICAgICAgIGV2YWx1YXRlX2ZuOiBjYWxsYWJsZSwKICAgICAgICBiYXNlX21ldHJpY3M6IGRpY3QsCiAgICAgICAgY29tcG9uZW50czogbGlzdFtzdHJdID0gTm9uZSwKICAgICkgLT4gZGljdDoKICAgICAgICAiIiJBYmxhdGUgbW9kZWwgY29tcG9uZW50cyB0byBtZWFzdXJlIHRoZWlyIGluZGl2aWR1YWwgY29udHJpYnV0aW9uLgoKICAgICAgICBDb21wb25lbnRzOgogICAgICAgICAgLSBtZW1vcnlfbW9kdWxlOiBSZW1vdmUgcGVyLW5vZGUgbWVtb3J5ICh1c2UgemVybyBtZW1vcnkpCiAgICAgICAgICAtIHRlbXBvcmFsX2VuY29kaW5nOiBSZW1vdmUgdGltZSBlbmNvZGluZwogICAgICAgICAgLSBtdWx0aV9oZWFkX2F0dGVudGlvbjogVXNlIHNpbmdsZS1oZWFkIGF0dGVudGlvbgogICAgICAgICAgLSBncmFwaF9zdHJ1Y3R1cmU6IFJlbW92ZSBhbGwgZWRnZXMgKGlzb2xhdGVkIG5vZGVzKQogICAgICAgICIiIgogICAgICAgIGxvZ2dlci5pbmZvKCI9IiAqIDYwKQogICAgICAgIGxvZ2dlci5pbmZvKCJBQkxBVElPTjogTW9kZWwgQ29tcG9uZW50cyIpCiAgICAgICAgbG9nZ2VyLmluZm8oIj0iICogNjApCgogICAgICAgIGlmIGNvbXBvbmVudHMgaXMgTm9uZToKICAgICAgICAgICAgY29tcG9uZW50cyA9IFsKICAgICAgICAgICAgICAgICJtZW1vcnlfbW9kdWxlIiwKICAgICAgICAgICAgICAgICJ0ZW1wb3JhbF9lbmNvZGluZyIsCiAgICAgICAgICAgICAgICAibXVsdGlfaGVhZF9hdHRlbnRpb24iLAogICAgICAgICAgICAgICAgImdyYXBoX3N0cnVjdHVyZSIsCiAgICAgICAgICAgIF0KCiAgICAgICAgcmVzdWx0cyA9IHsiZnVsbF9tb2RlbCI6IGJhc2VfbWV0cmljc30KCiAgICAgICAgZm9yIGNvbXBvbmVudCBpbiBjb21wb25lbnRzOgogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIkFibGF0aW5nIGNvbXBvbmVudDoge2NvbXBvbmVudH0iKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAjIENyZWF0ZSBtb2RpZmllZCBtb2RlbAogICAgICAgICAgICAgICAgbW9kaWZpZWRfbW9kZWwgPSBjb3B5LmRlZXBjb3B5KHNlbGYuYmFzZV9tb2RlbCkKCiAgICAgICAgICAgICAgICBpZiBjb21wb25lbnQgPT0gIm1lbW9yeV9tb2R1bGUiOgogICAgICAgICAgICAgICAgICAgICMgWmVybyBvdXQgbWVtb3J5IGFuZCBkaXNhYmxlIHVwZGF0ZXMKICAgICAgICAgICAgICAgICAgICBpZiBoYXNhdHRyKG1vZGlmaWVkX21vZGVsLCAibWVtb3J5Iik6CiAgICAgICAgICAgICAgICAgICAgICAgIG1vZGlmaWVkX21vZGVsLm1lbW9yeS5yZXNldF9tZW1vcnkoKQogICAgICAgICAgICAgICAgICAgICAgICAjIFJlcGxhY2UgdXBkYXRlIHdpdGggbm8tb3AKICAgICAgICAgICAgICAgICAgICAgICAgb3JpZ2luYWxfdXBkYXRlID0gbW9kaWZpZWRfbW9kZWwubWVtb3J5LnVwZGF0ZV9tZW1vcnkKICAgICAgICAgICAgICAgICAgICAgICAgbW9kaWZpZWRfbW9kZWwubWVtb3J5LnVwZGF0ZV9tZW1vcnkgPSAoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBsYW1iZGEgKmFyZ3MsICoqa3dhcmdzOiBOb25lCiAgICAgICAgICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgICAgICBlbGlmIGNvbXBvbmVudCA9PSAidGVtcG9yYWxfZW5jb2RpbmciOgogICAgICAgICAgICAgICAgICAgICMgUmVwbGFjZSB0aW1lIGVuY29kZXIgd2l0aCB6ZXJvcwogICAgICAgICAgICAgICAgICAgIGlmIGhhc2F0dHIobW9kaWZpZWRfbW9kZWwsICJ0aW1lX2VuY29kZXIiKToKICAgICAgICAgICAgICAgICAgICAgICAgbW9kaWZpZWRfbW9kZWwudGltZV9lbmNvZGVyID0gX1plcm9UaW1lRW5jb2RlcigKICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1vZGlmaWVkX21vZGVsLnRpbWVfZW5jb2Rlci5kaW0KICAgICAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgICAgIGVsaWYgY29tcG9uZW50ID09ICJncmFwaF9zdHJ1Y3R1cmUiOgogICAgICAgICAgICAgICAgICAgICMgV2lsbCBiZSBoYW5kbGVkIGJ5IHBhc3NpbmcgZW1wdHkgZWRnZSBkaWN0cwogICAgICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgICAgICAgICBjb21wX21ldHJpY3MgPSBldmFsdWF0ZV9mbigKICAgICAgICAgICAgICAgICAgICBtb2RpZmllZF9tb2RlbCwgYWJsYXRlX2NvbXBvbmVudD1jb21wb25lbnQKICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgICAgICBkZWx0YSA9IHt9CiAgICAgICAgICAgICAgICBmb3IgaGsgaW4gYmFzZV9tZXRyaWNzOgogICAgICAgICAgICAgICAgICAgIGlmIGhrIGluIGNvbXBfbWV0cmljczoKICAgICAgICAgICAgICAgICAgICAgICAgZGVsdGFbaGtdID0ge30KICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG1rIGluIGJhc2VfbWV0cmljc1toa106CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBidiA9IGJhc2VfbWV0cmljc1toa11bbWtdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBhdiA9IGNvbXBfbWV0cmljc1toa11bbWtdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKGJ2LCAoaW50LCBmbG9hdCkpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlbHRhW2hrXVtta10gPSBidiAtIGF2CgogICAgICAgICAgICAgICAgcmVzdWx0c1tmIndpdGhvdXRfe2NvbXBvbmVudH0iXSA9IHsKICAgICAgICAgICAgICAgICAgICAibWV0cmljcyI6IGNvbXBfbWV0cmljcywKICAgICAgICAgICAgICAgICAgICAiZGVsdGEiOiBkZWx0YSwKICAgICAgICAgICAgICAgIH0KCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIkNvbXBvbmVudCBhYmxhdGlvbiBmYWlsZWQgZm9yIHtjb21wb25lbnR9OiB7ZX0iKQogICAgICAgICAgICAgICAgcmVzdWx0c1tmIndpdGhvdXRfe2NvbXBvbmVudH0iXSA9IHsiZXJyb3IiOiBzdHIoZSl9CgogICAgICAgIHNlbGYucmVzdWx0c1sibW9kZWxfY29tcG9uZW50X2FibGF0aW9uIl0gPSByZXN1bHRzCiAgICAgICAgcmV0dXJuIHJlc3VsdHMKCiAgICBkZWYgZ2VuZXJhdGVfYWJsYXRpb25fdGFibGUoc2VsZikgLT4gc3RyOgogICAgICAgICIiIkdlbmVyYXRlIGEgZm9ybWF0dGVkIExhVGVYIHRhYmxlIG9mIGFibGF0aW9uIHJlc3VsdHMuIiIiCiAgICAgICAgbGluZXMgPSBbXQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxiZWdpbnt0YWJsZX1baHRicF0iKQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxjZW50ZXJpbmciKQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxjYXB0aW9ue0FibGF0aW9uIFN0dWR5IFJlc3VsdHN9IikKICAgICAgICBsaW5lcy5hcHBlbmQoIlxcbGFiZWx7dGFiOmFibGF0aW9ufSIpCgogICAgICAgIGhvcml6b25zID0gW2YiY2FzY2FkZV97aH1oIiBmb3IgaCBpbiBbMjQsIDcyLCAxNjgsIDcyMF1dCiAgICAgICAgaG9yaXpvbl9sYWJlbHMgPSBbIjFkIiwgIjNkIiwgIjdkIiwgIjMwZCJdCgogICAgICAgICMgSGVhZGVyCiAgICAgICAgY29scyA9ICJsIiArICJjIiAqIGxlbihob3Jpem9ucykKICAgICAgICBsaW5lcy5hcHBlbmQoZiJcXGJlZ2lue3t0YWJ1bGFyfX17e3tjb2xzfX19IikKICAgICAgICBsaW5lcy5hcHBlbmQoIlxcdG9wcnVsZSIpCiAgICAgICAgaGVhZGVyID0gIkNvbmZpZ3VyYXRpb24gJiAiICsgIiAmICIuam9pbigKICAgICAgICAgICAgW2YiQVVST0MgKHtofSkiIGZvciBoIGluIGhvcml6b25fbGFiZWxzXQogICAgICAgICkKICAgICAgICBsaW5lcy5hcHBlbmQoaGVhZGVyICsgIiBcXFxcIikKICAgICAgICBsaW5lcy5hcHBlbmQoIlxcbWlkcnVsZSIpCgogICAgICAgICMgRnVsbCBtb2RlbAogICAgICAgIGxpbmVzLmFwcGVuZCgiXFx0ZXh0YmZ7RnVsbCBNb2RlbCAoVEdOKX0gJiAiICsgIiAmICIuam9pbigKICAgICAgICAgICAgWyItLSJdICogbGVuKGhvcml6b25zKQogICAgICAgICkgKyAiIFxcXFwiKQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxtaWRydWxlIikKCiAgICAgICAgZm9yIGFibGF0aW9uX3R5cGUsIHR5cGVfcmVzdWx0cyBpbiBzZWxmLnJlc3VsdHMuaXRlbXMoKToKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiXFxtdWx0aWNvbHVtbnt7e2xlbihob3Jpem9ucykgKyAxfX19e3tsfX0iCiAgICAgICAgICAgICAgICAgICAgICAgICBmInt7XFx0ZXh0aXR7e3thYmxhdGlvbl90eXBlLnJlcGxhY2UoJ18nLCAnICcpLnRpdGxlKCl9fX19fSBcXFxcIikKCiAgICAgICAgICAgIGZvciBjb25maWdfbmFtZSwgY29uZmlnX2RhdGEgaW4gdHlwZV9yZXN1bHRzLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBpZiBjb25maWdfbmFtZSA9PSAiZnVsbF9tb2RlbCI6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UoY29uZmlnX2RhdGEsIGRpY3QpIGFuZCAiZGVsdGEiIGluIGNvbmZpZ19kYXRhOgogICAgICAgICAgICAgICAgICAgIHZhbHVlcyA9IFtdCiAgICAgICAgICAgICAgICAgICAgZm9yIGhrIGluIGhvcml6b25zOgogICAgICAgICAgICAgICAgICAgICAgICBkID0gY29uZmlnX2RhdGFbImRlbHRhIl0uZ2V0KGhrLCB7fSkKICAgICAgICAgICAgICAgICAgICAgICAgYXVyb2NfZHJvcCA9IGQuZ2V0KCJhdXJvYyIsIDApCiAgICAgICAgICAgICAgICAgICAgICAgIHZhbHVlcy5hcHBlbmQoZiJ7YXVyb2NfZHJvcDorLjRmfSIpCgogICAgICAgICAgICAgICAgICAgIGNsZWFuX25hbWUgPSBjb25maWdfbmFtZS5yZXBsYWNlKCJ3aXRob3V0XyIsICJ3L28gIikucmVwbGFjZSgiXyIsICIgIikKICAgICAgICAgICAgICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgICAgIGYiICB7Y2xlYW5fbmFtZX0gJiAiICsgIiAmICIuam9pbih2YWx1ZXMpICsgIiBcXFxcIgogICAgICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgIGxpbmVzLmFwcGVuZCgiXFxtaWRydWxlIikKCiAgICAgICAgbGluZXMuYXBwZW5kKCJcXGJvdHRvbXJ1bGUiKQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxlbmR7dGFidWxhcn0iKQogICAgICAgIGxpbmVzLmFwcGVuZCgiXFxlbmR7dGFibGV9IikKCiAgICAgICAgdGFibGUgPSAiXG4iLmpvaW4obGluZXMpCgogICAgICAgICMgU2F2ZQogICAgICAgIHdpdGggb3BlbihzZWxmLm91dHB1dF9kaXIgLyAiYWJsYXRpb25fdGFibGUudGV4IiwgInciKSBhcyBmOgogICAgICAgICAgICBmLndyaXRlKHRhYmxlKQoKICAgICAgICByZXR1cm4gdGFibGUKCiAgICBkZWYgc2F2ZV9yZXN1bHRzKHNlbGYpOgogICAgICAgICIiIlNhdmUgYWxsIGFibGF0aW9uIHJlc3VsdHMgdG8gSlNPTi4iIiIKCiAgICAgICAgZGVmIGNvbnZlcnQob2JqKToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShvYmosIG5wLmZsb2F0aW5nKToKICAgICAgICAgICAgICAgIHJldHVybiBmbG9hdChvYmopCiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2Uob2JqLCBucC5pbnRlZ2VyKToKICAgICAgICAgICAgICAgIHJldHVybiBpbnQob2JqKQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKG9iaiwgbnAubmRhcnJheSk6CiAgICAgICAgICAgICAgICByZXR1cm4gb2JqLnRvbGlzdCgpCiAgICAgICAgICAgIHJhaXNlIFR5cGVFcnJvcihmIk5vdCBzZXJpYWxpemFibGU6IHt0eXBlKG9iail9IikKCiAgICAgICAgd2l0aCBvcGVuKHNlbGYub3V0cHV0X2RpciAvICJhYmxhdGlvbl9yZXN1bHRzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAgICAgIGpzb24uZHVtcChzZWxmLnJlc3VsdHMsIGYsIGluZGVudD0yLCBkZWZhdWx0PWNvbnZlcnQpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJBYmxhdGlvbiByZXN1bHRzIHNhdmVkIHRvIHtzZWxmLm91dHB1dF9kaXJ9IikKCgpjbGFzcyBfWmVyb1RpbWVFbmNvZGVyKHRvcmNoLm5uLk1vZHVsZSk6CiAgICAiIiJEdW1teSB0aW1lIGVuY29kZXIgdGhhdCBhbHdheXMgcmV0dXJucyB6ZXJvcyAoZm9yIGFibGF0aW9uKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltOiBpbnQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZGltID0gZGltCgogICAgZGVmIGZvcndhcmQoc2VsZiwgdDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgaWYgdC5kaW0oKSA9PSAxOgogICAgICAgICAgICByZXR1cm4gdG9yY2guemVyb3ModC5zaXplKDApLCBzZWxmLmRpbSwgZGV2aWNlPXQuZGV2aWNlKQogICAgICAgIHJldHVybiB0b3JjaC56ZXJvcyh0LnNpemUoMCksIHNlbGYuZGltLCBkZXZpY2U9dC5kZXZpY2UpCg==", "experiments/run_experiments.py": "...", "config/config.yaml": "IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBEZUZpIExpcXVpZGF0aW9uIENhc2NhZGUgUHJlZGljdG9yIOKAlCBDb25maWd1cmF0aW9uCiMgSUVFRSBUQ1NTIFBhcGVyOiBUZW1wb3JhbCBHTk4gZm9yIENyb3NzLVByb3RvY29sIENhc2NhZGVzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09Cgpwcm9qZWN0OgogIG5hbWU6ICJkZWZpX2Nhc2NhZGVfcHJlZGljdG9yIgogIHNlZWQ6IDQyCiAgZGV2aWNlOiAiY3VkYSIgICMgImN1ZGEiIG9yICJjcHUiCiAgb3V0cHV0X2RpcjogIm91dHB1dHMiCgpkYXRhOgogIHJhd19kaXI6ICJkYXRhL3JhdyIKICBwcm9jZXNzZWRfZGlyOiAiZGF0YS9wcm9jZXNzZWQiCiAgIyBEYXRlIHJhbmdlIGZvciBkYXRhIGNvbGxlY3Rpb24KICBzdGFydF9kYXRlOiAiMjAyMS0wNi0wMSIKICBlbmRfZGF0ZTogIjIwMjYtMDItMjgiCiAgZGF0YV9zb3VyY2U6ICJyZWFsIgogIHR2bF9kYXRhX3BhdGg6ICJkYXRhL3JlYWwvdHZsX2NvbWJpbmVkLmNzdiIKICAjIFRlbXBvcmFsIHJlc29sdXRpb24KICBzbmFwc2hvdF9pbnRlcnZhbF9ob3VyczogMjQgICMgZGFpbHkgc25hcHNob3RzCiAgIyBQcm90b2NvbHMgdG8gdHJhY2sKICBwcm90b2NvbHM6CiAgICAtIG5hbWU6ICJhYXZlLXYzIgogICAgICBjaGFpbjogImV0aGVyZXVtIgogICAgICB0eXBlOiAibGVuZGluZyIKICAgIC0gbmFtZTogImFhdmUtdjIiCiAgICAgIGNoYWluOiAiZXRoZXJldW0iCiAgICAgIHR5cGU6ICJsZW5kaW5nIgogICAgLSBuYW1lOiAiY29tcG91bmQtdjMiCiAgICAgIGNoYWluOiAiZXRoZXJldW0iCiAgICAgIHR5cGU6ICJsZW5kaW5nIgogICAgLSBuYW1lOiAiY29tcG91bmQtdjIiCiAgICAgIGNoYWluOiAiZXRoZXJldW0iCiAgICAgIHR5cGU6ICJsZW5kaW5nIgogICAgLSBuYW1lOiAibWFrZXJkYW8iCiAgICAgIGNoYWluOiAiZXRoZXJldW0iCiAgICAgIHR5cGU6ICJjZHAiCiAgICAtIG5hbWU6ICJ1bmlzd2FwLXYzIgogICAgICBjaGFpbjogImV0aGVyZXVtIgogICAgICB0eXBlOiAiZGV4IgogICAgLSBuYW1lOiAidW5pc3dhcC12MiIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogImRleCIKICAgIC0gbmFtZTogImN1cnZlLWRleCIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogImRleCIKICAgIC0gbmFtZTogImxpZG8iCiAgICAgIGNoYWluOiAiZXRoZXJldW0iCiAgICAgIHR5cGU6ICJsaXF1aWRfc3Rha2luZyIKICAgIC0gbmFtZTogInJvY2tldC1wb29sIgogICAgICBjaGFpbjogImV0aGVyZXVtIgogICAgICB0eXBlOiAibGlxdWlkX3N0YWtpbmciCiAgICAtIG5hbWU6ICJjb252ZXgtZmluYW5jZSIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogInlpZWxkIgogICAgLSBuYW1lOiAieWVhcm4tZmluYW5jZSIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogInlpZWxkIgogICAgLSBuYW1lOiAiZnJheCIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogInN0YWJsZWNvaW4iCiAgICAtIG5hbWU6ICJiYWxhbmNlciIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogImRleCIKICAgIC0gbmFtZTogIm1vcnBobyIKICAgICAgY2hhaW46ICJldGhlcmV1bSIKICAgICAgdHlwZTogImxlbmRpbmciCgogICMgS25vd24gY2FzY2FkZSBldmVudHMgZm9yIGxhYmVsaW5nCiAgY2FzY2FkZV9ldmVudHM6CiAgICAtIG5hbWU6ICJ0ZXJyYV9sdW5hX2NvbGxhcHNlIgogICAgICBzdGFydDogIjIwMjItMDUtMDciCiAgICAgIHBlYWs6ICIyMDIyLTA1LTEyIgogICAgICBlbmQ6ICIyMDIyLTA1LTE1IgogICAgICBzZXZlcml0eTogImNhdGFzdHJvcGhpYyIKICAgICAgdHZsX2xvc3NfcGN0OiAwLjQ1CiAgICAtIG5hbWU6ICIzYWNfY2Vsc2l1c19jb250YWdpb24iCiAgICAgIHN0YXJ0OiAiMjAyMi0wNi0xMiIKICAgICAgcGVhazogIjIwMjItMDYtMTgiCiAgICAgIGVuZDogIjIwMjItMDYtMjUiCiAgICAgIHNldmVyaXR5OiAic2V2ZXJlIgogICAgICB0dmxfbG9zc19wY3Q6IDAuMzAKICAgIC0gbmFtZTogImZ0eF9jb2xsYXBzZSIKICAgICAgc3RhcnQ6ICIyMDIyLTExLTA2IgogICAgICBwZWFrOiAiMjAyMi0xMS0xMSIKICAgICAgZW5kOiAiMjAyMi0xMS0xNCIKICAgICAgc2V2ZXJpdHk6ICJzZXZlcmUiCiAgICAgIHR2bF9sb3NzX3BjdDogMC4yNQogICAgLSBuYW1lOiAidXNkY19kZXBlZ19zdmIiCiAgICAgIHN0YXJ0OiAiMjAyMy0wMy0xMCIKICAgICAgcGVhazogIjIwMjMtMDMtMTEiCiAgICAgIGVuZDogIjIwMjMtMDMtMTMiCiAgICAgIHNldmVyaXR5OiAibW9kZXJhdGUiCiAgICAgIHR2bF9sb3NzX3BjdDogMC4xMgogICAgLSBuYW1lOiAiZXVsZXJfaGFjayIKICAgICAgc3RhcnQ6ICIyMDIzLTAzLTEzIgogICAgICBwZWFrOiAiMjAyMy0wMy0xMyIKICAgICAgZW5kOiAiMjAyMy0wMy0xNiIKICAgICAgc2V2ZXJpdHk6ICJtb2RlcmF0ZSIKICAgICAgdHZsX2xvc3NfcGN0OiAwLjA4CiAgICAtIG5hbWU6ICJjdXJ2ZV9leHBsb2l0IgogICAgICBzdGFydDogIjIwMjMtMDctMzAiCiAgICAgIHBlYWs6ICIyMDIzLTA3LTMxIgogICAgICBlbmQ6ICIyMDIzLTA4LTAyIgogICAgICBzZXZlcml0eTogIm1vZGVyYXRlIgogICAgICB0dmxfbG9zc19wY3Q6IDAuMTAKCmdyYXBoOgogICMgTm9kZSB0eXBlcwogIG5vZGVfdHlwZXM6CiAgICAtICJwcm90b2NvbCIKICAgIC0gImxpcXVpZGl0eV9wb29sIgogICAgLSAidG9rZW4iCiAgIyBFZGdlIHR5cGVzIGluIHRoZSBjb21wb3NhYmlsaXR5IGdyYXBoCiAgZWRnZV90eXBlczoKICAgIC0gInNoYXJlZF9jb2xsYXRlcmFsIiAgICAgICMgcHJvdG9jb2xzIHNoYXJpbmcgdGhlIHNhbWUgY29sbGF0ZXJhbCBhc3NldHMKICAgIC0gImxpcXVpZGl0eV9mbG93IiAgICAgICAgICMgY2FwaXRhbCBmbG93aW5nIGJldHdlZW4gcHJvdG9jb2xzCiAgICAtICJvcmFjbGVfZGVwZW5kZW5jeSIgICAgICAjIHByb3RvY29scyBzaGFyaW5nIHByaWNlIG9yYWNsZSBzb3VyY2VzCiAgICAtICJnb3Zlcm5hbmNlX292ZXJsYXAiICAgICAjIHNoYXJlZCBnb3Zlcm5hbmNlIHRva2VuIGhvbGRlcnMKICAgIC0gInByaWNlX2NvcnJlbGF0aW9uIiAgICAgICMgaGlnaGx5IGNvcnJlbGF0ZWQgdG9rZW4gZXhwb3N1cmVzCiAgICAtICJsaXF1aWRhdGlvbl9wYXRod2F5IiAgICAjIHByb3RvY29sIEEgbGlxdWlkYXRpb24gdHJpZ2dlcnMgc2VsbHMgb24gcHJvdG9jb2wgQgogICMgTm9kZSBmZWF0dXJlIGRpbWVuc2lvbnMKICBub2RlX2ZlYXR1cmVfZGltOiAzMgogIGVkZ2VfZmVhdHVyZV9kaW06IDE2CiAgIyBUZW1wb3JhbCBwYXJhbWV0ZXJzCiAgdGVtcG9yYWxfd2luZG93OiAzMCAgIyBkYXlzIG9mIGxvb2tiYWNrIGZvciB0ZW1wb3JhbCBlbmNvZGluZwoKbW9kZWw6CiAgIyBUR04gQXJjaGl0ZWN0dXJlCiAgdGduOgogICAgbWVtb3J5X2RpbTogMTI4CiAgICB0aW1lX2VuY29kaW5nX2RpbTogMzIKICAgIGVtYmVkZGluZ19kaW06IDEyOAogICAgbnVtX2F0dGVudGlvbl9oZWFkczogNAogICAgbnVtX2dubl9sYXllcnM6IDIKICAgIGRyb3BvdXQ6IDAuMQogICAgbWVtb3J5X3VwZGF0ZXI6ICJncnUiICAjICJncnUiIG9yICJybm4iCiAgICBtZXNzYWdlX2FnZ3JlZ2F0b3I6ICJsYXN0IiAgIyAibWVhbiIgb3IgImxhc3QiCiAgICBlbWJlZGRpbmdfbW9kdWxlOiAiZ3JhcGhfYXR0ZW50aW9uIiAgIyAiZ3JhcGhfYXR0ZW50aW9uIiwgImdyYXBoX3N1bSIsICJpZGVudGl0eSIKCiAgIyBTdGF0aWMgR05OIGJhc2VsaW5lCiAgc3RhdGljX2dubjoKICAgIGhpZGRlbl9kaW06IDEyOAogICAgbnVtX2xheWVyczogMgogICAgaGVhZHM6IDQKICAgIGRyb3BvdXQ6IDAuMQoKICAjIExTVE0gYmFzZWxpbmUKICBsc3RtOgogICAgaGlkZGVuX2RpbTogMTI4CiAgICBudW1fbGF5ZXJzOiAyCiAgICBkcm9wb3V0OiAwLjIKICAgIGJpZGlyZWN0aW9uYWw6IGZhbHNlCgogICMgWEdCb29zdCBiYXNlbGluZQogIHhnYm9vc3Q6CiAgICBuX2VzdGltYXRvcnM6IDUwMAogICAgbWF4X2RlcHRoOiA4CiAgICBsZWFybmluZ19yYXRlOiAwLjA1CiAgICBzdWJzYW1wbGU6IDAuOAogICAgY29sc2FtcGxlX2J5dHJlZTogMC44CiAgICBtaW5fY2hpbGRfd2VpZ2h0OiA1CiAgICByZWdfYWxwaGE6IDAuMQogICAgcmVnX2xhbWJkYTogMS4wCgogICMgU0lSIGNvbnRhZ2lvbiBiYXNlbGluZQogIHNpcjoKICAgIGJldGFfcmFuZ2U6IFswLjAxLCAwLjVdICAjIGluZmVjdGlvbiByYXRlIHNlYXJjaCByYW5nZQogICAgZ2FtbWFfcmFuZ2U6IFswLjAxLCAwLjNdICAjIHJlY292ZXJ5IHJhdGUgc2VhcmNoIHJhbmdlCiAgICBuX3NpbXVsYXRpb25zOiAxMDAwCgp0cmFpbmluZzoKICBiYXRjaF9zaXplOiA2NAogIGxlYXJuaW5nX3JhdGU6IDAuMDAwMwogIHdlaWdodF9kZWNheTogMC4wMDAxCiAgZXBvY2hzOiAzMDAKICBwYXRpZW5jZTogMzUgICMgZWFybHkgc3RvcHBpbmcKICBudW1fZm9sZHM6IDUgICMgZm9yIGNyb3NzLXZhbGlkYXRpb24KICB2YWxfcmF0aW86IDAuMTUKICB0ZXN0X3JhdGlvOiAwLjE1CiAgIyBDbGFzcyBpbWJhbGFuY2UgaGFuZGxpbmcKICBwb3Nfd2VpZ2h0OiAxMC4wICAjIHBvc2l0aXZlIGNsYXNzIHdlaWdodCBmb3IgQkNFIGxvc3MKICBmb2NhbF9sb3NzX2dhbW1hOiAyLjAKICAjIFByZWRpY3Rpb24gaG9yaXpvbnMgKGhvdXJzKSDigJQgZGFpbHkgc25hcHNob3RzIHNvIG1pbmltdW0gdXNlZnVsIGlzIDI0aAogIHByZWRpY3Rpb25faG9yaXpvbnM6IFsyNCwgNzIsIDE2OCwgNzIwXSAgIyAxZCwgM2QsIDdkLCAzMGQKICAjIFNjaGVkdWxlcgogIHNjaGVkdWxlcjoKICAgIHR5cGU6ICJjb3NpbmUiCiAgICBUX21heDogMjAwCiAgICBldGFfbWluOiAwLjAwMDAwMQoKZXZhbHVhdGlvbjoKICAjIE1ldHJpY3MKICBtZXRyaWNzOgogICAgLSAiYXVyb2MiCiAgICAtICJhdXByYyIKICAgIC0gImYxIgogICAgLSAicHJlY2lzaW9uIgogICAgLSAicmVjYWxsIgogICAgLSAiYnJpZXJfc2NvcmUiCiAgICAtICJsZWFkX3RpbWVfaG91cnMiCiAgICAtICJtY2MiICAjIE1hdHRoZXdzIGNvcnJlbGF0aW9uIGNvZWZmaWNpZW50CiAgIyBTdGF0aXN0aWNhbCB0ZXN0cwogIHN0YXRpc3RpY2FsX3Rlc3RzOgogICAgY29uZmlkZW5jZV9sZXZlbDogMC45NQogICAgYm9vdHN0cmFwX2l0ZXJhdGlvbnM6IDEwMDAwCiAgICBkaWVib2xkX21hcmlhbm86IHRydWUKICAgIG1jbmVtYXI6IHRydWUKICAgIHdpbGNveG9uOiB0cnVlCiAgICBwYWlyZWRfdHRlc3Q6IHRydWUKICAjIEFibGF0aW9uIHN0dWRpZXMKICBhYmxhdGlvbjoKICAgIGZlYXR1cmVfZ3JvdXBzOgogICAgICAtICJ0dmxfZmVhdHVyZXMiCiAgICAgIC0gInByaWNlX2ZlYXR1cmVzIgogICAgICAtICJsaXF1aWRpdHlfZmVhdHVyZXMiCiAgICAgIC0gIm5ldHdvcmtfZmVhdHVyZXMiCiAgICAgIC0gIm1hY3JvX2ZlYXR1cmVzIgogICAgICAtICJ0ZW1wb3JhbF9mZWF0dXJlcyIKICAgIHRlbXBvcmFsX3dpbmRvd3M6IFs3LCAxNCwgMzAsIDYwLCA5MF0KICAgIGVkZ2VfdHlwZXNfdG9fYWJsYXRlOgogICAgICAtICJzaGFyZWRfY29sbGF0ZXJhbCIKICAgICAgLSAibGlxdWlkaXR5X2Zsb3ciCiAgICAgIC0gIm9yYWNsZV9kZXBlbmRlbmN5IgogICAgICAtICJnb3Zlcm5hbmNlX292ZXJsYXAiCiAgICAgIC0gInByaWNlX2NvcnJlbGF0aW9uIgogICAgICAtICJsaXF1aWRhdGlvbl9wYXRod2F5IgogICAgbW9kZWxfY29tcG9uZW50czoKICAgICAgLSAibWVtb3J5X21vZHVsZSIKICAgICAgLSAidGVtcG9yYWxfZW5jb2RpbmciCiAgICAgIC0gIm11bHRpX2hlYWRfYXR0ZW50aW9uIgogICAgICAtICJncmFwaF9zdHJ1Y3R1cmUiCg=="}')

for rel_path, b64_content in SOURCE_FILES.items():
    fpath = PROJECT_ROOT / rel_path
    fpath.write_text(base64.b64decode(b64_content).decode("utf-8"))

print(f"Wrote {len(SOURCE_FILES)} source files to {PROJECT_ROOT}")

# ---- Embed Real TVL Data ----
TVL_DATA_B64 = '...'

tvl_bytes = gzip.decompress(base64.b64decode(TVL_DATA_B64))
(PROJECT_ROOT / "data" / "real" / "tvl_combined.csv").write_bytes(tvl_bytes)
print(f"Wrote TVL data: {len(tvl_bytes):,} bytes")

# ---- Run Full Pipeline ----
print("\n" + "=" * 70)
print("RUNNING FULL EXPERIMENT PIPELINE...")
print("=" * 70)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import yaml
from loguru import logger

logger.remove()
logger.add(sys.stderr, format='<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | <level>{message}</level>', level='INFO')
logger.add('outputs/experiment.log', rotation='10 MB', level='DEBUG')

with open("config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

config["project"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {config["project"]["device"]}')

if config["project"]["device"] == "cpu":
    t = config["training"]
    t["epochs"] = min(t.get("epochs", 200), 100)
    t["patience"] = min(t.get("patience", 25), 20)
    t["baseline_epochs"] = min(t.get("baseline_epochs", 80), 50)
    t["ablation_epochs"] = min(t.get("ablation_epochs", 30), 20)
    config["evaluation"]["statistical_tests"]["bootstrap_iterations"] = min(
        config["evaluation"]["statistical_tests"].get("bootstrap_iterations", 10000), 3000)
    config["model"]["sir"]["n_simulations"] = min(
        config["model"]["sir"].get("n_simulations", 1000), 100)
    print("CPU MODE: capped epochs for reasonable runtime")

from experiments.run_experiments import ExperimentRunner
runner = ExperimentRunner(config)
results = runner.run_full_pipeline()

# ---- Results Summary ----
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

import numpy as np

if "model_comparison" in results:
    horizons = ["cascade_24h", "cascade_72h", "cascade_168h", "cascade_720h"]
    horizon_labels = ["1-day", "3-day", "7-day", "30-day"]
    header = f'{"Model":<15}'
    for hl in horizon_labels:
        header += f" | {hl:>12} AUROC"
    print(header)
    print("-" * 80)
    for model_name, metrics in results["model_comparison"].items():
        row = f"{model_name:<15}"
        for hk in horizons:
            if hk in metrics and isinstance(metrics[hk], dict):
                auroc = metrics[hk].get("auroc", 0)
                auprc = metrics[hk].get("auprc", 0)
                row += f" | {auroc:>6.3f} / {auprc:.3f}"
            else:
                row += f' | {"N/A":>12}'
        print(row)
    print("\nFormat: AUROC / AUPRC")

if "training_history" in results:
    h = results["training_history"]
    print(f'\nTGN Training: {h["total_epochs"]} epochs, best at epoch {h["best_epoch"]}')
    print(f'Best validation loss: {h["best_val_loss"]:.6f}')

# ---- Statistical Tests ----
if "statistical_tests" in results:
    print("\n" + "=" * 70)
    print("STATISTICAL SIGNIFICANCE TESTS")
    print("=" * 70)
    for horizon_key, tests in results["statistical_tests"].items():
        print(f"\n--- {horizon_key} ---")
        ci = tests.get("tgn_auroc_ci", {})
        if ci:
            print(f'  TGN AUROC: {ci.get("mean", 0):.4f} '
                  f'[{ci.get("ci_lower", 0):.4f}, {ci.get("ci_upper", 0):.4f}] (95% CI)')
        for mn, comp in tests.items():
            if mn.startswith("tgn_"): continue
            if isinstance(comp, dict):
                dm = comp.get("diebold_mariano", {})
                mc = comp.get("mcnemar", {})
                print(f"  vs {mn}:")
                if dm:
                    print(f'    DM: stat={dm.get("test_statistic", 0):.3f}, p={dm.get("p_value", 1):.4f}')
                if mc:
                    print(f'    McNemar: stat={mc.get("test_statistic", 0):.3f}, p={mc.get("p_value", 1):.4f}')

# ---- Ablation Studies ----
if "ablation" in results:
    print("\n" + "=" * 70)
    print("ABLATION STUDIES")
    print("=" * 70)
    for abl_type, abl_data in results["ablation"].items():
        print(f'\n--- {abl_type.replace("_", " ").title()} ---')
        for cfg_name, cfg_data in abl_data.items():
            if cfg_name == "full_model": continue
            if isinstance(cfg_data, dict) and "delta" in cfg_data:
                deltas = []
                for hk, hd in cfg_data["delta"].items():
                    if isinstance(hd, dict) and "auroc" in hd:
                        deltas.append(f'{hk.replace("cascade_","")}={hd["auroc"]:+.4f}')
                if deltas:
                    print(f'  {cfg_name:<30} AUROC drop: {", ".join(deltas)}')

# ---- Display Figures ----
print("\n" + "=" * 70)
print("GENERATED FIGURES")
print("=" * 70)

try:
    from IPython.display import display, Image
    fig_dir = PROJECT_ROOT / "outputs" / "figures"
    for fig_file in sorted(fig_dir.glob("*.pdf")):
        print(f"\n--- {fig_file.name} ---")
        try:
            from pdf2image import convert_from_path
            images = convert_from_path(str(fig_file), dpi=150)
            for img in images:
                display(img)
        except Exception:
            print(f"  (PDF saved — download zip to view)")
    for fig_file in sorted(fig_dir.glob("*.png")):
        print(f"\n--- {fig_file.name} ---")
        try:
            display(Image(filename=str(fig_file)))
        except Exception:
            pass
except Exception as e:
    print(f"Figure display error (non-fatal): {e}")

# ---- Export Results ----
def convert(obj):
    import numpy as _np
    if isinstance(obj, (_np.floating, _np.float64, _np.float32)): return float(obj)
    if isinstance(obj, (_np.integer, _np.int64, _np.int32)): return int(obj)
    if isinstance(obj, _np.bool_): return bool(obj)
    if isinstance(obj, _np.ndarray): return obj.tolist()
    if isinstance(obj, torch.Tensor): return obj.cpu().numpy().tolist()
    if hasattr(obj, "isoformat"): return str(obj)
    raise TypeError(f"Not serializable: {type(obj)}")

save_results = {k: v for k, v in results.items() if k not in ("all_preds", "test_targets")}
try:
    with open(PROJECT_ROOT / "outputs" / "results" / "all_results.json", "w") as f:
        json.dump(save_results, f, indent=2, default=convert)
    print(f"\nSaved results JSON")
except Exception as e:
    print(f"Warning: Could not save results JSON: {e}")

# ---- Create Download Zip ----
import shutil
output_zip = "/content/defi_cascade_results"
shutil.make_archive(output_zip, "zip", str(PROJECT_ROOT / "outputs"))
print(f"Created {output_zip}.zip")

try:
    from google.colab import files
    files.download(f"{output_zip}.zip")
    print("Download started!")
except ImportError:
    print(f"Not in Colab. Results at: {PROJECT_ROOT / 'outputs'}")

print("\n" + "=" * 70)
print("ALL DONE!")
print("=" * 70)